# Spotify: clasificación y perfiles musicales
## Unidad 3 · Optimización y evaluación de modelos

Este cuaderno estudia la clasificación de canciones con popularidad alta mediante XGBoost y la optimización de agrupamientos para explorar playlists. Se utilizan los datos preparados en la Unidad 1 y exclusivamente la partición de entrenamiento definida en la Unidad 2.

**Alcance de esta primera etapa:** objetivos, KPI, protocolo de evaluación y comparación del costo de cinco pliegues agrupados frente a Leave-P-Out. No se entrenan modelos ni se presentan resultados predictivos nuevos.

### Recorrido de esta etapa
1. Contexto y objetivos.
2. Indicadores y reglas de decisión.
3. Datos permitidos y prevención de fuga de información.
4. Cinco pliegues agrupados y predicciones OOF.
5. Leave-P-Out: funcionamiento, costo y comparación justa.
6. Búsqueda de hiperparámetros y evaluación anidada.
7. Protocolo de clustering y próximos pasos.

Ejecutar las celdas en orden. Los cálculos de esta etapa usan solo la biblioteca estándar de Python; no requieren instalar XGBoost ni cargar el CSV. Los conteos históricos se identifican como antecedentes, no como controles nuevos sobre archivos.

## 1. Contexto y objetivos

### 1.1. Dos preguntas diferentes

**Clasificación:** ¿podemos identificar canciones cuya popularidad registrada es igual o mayor que 60, y mejorar el procedimiento mediante una búsqueda justificada de hiperparámetros de XGBoost?

**Clustering:** ¿podemos obtener perfiles musicales diferenciados y estables que ayuden a explorar canciones similares para playlists?

La clasificación tiene una respuesta conocida con la que evaluar predicciones; el clustering organiza semejanzas sin aprender esa respuesta. Un grupo musical no equivale a una clase de popularidad. Esta unidad no incluye regresión.

### 1.2. Decisiones y su motivo

- Mantener popularidad ≥60 como clase positiva permite conservar el problema de la Unidad 2. Es una definición operativa del proyecto, no un umbral oficial de éxito de Spotify.
- Usar `XGBClassifier` incorpora boosting supervisado: una secuencia de árboles que mejora progresivamente la función de pérdida. No es el mismo algoritmo que HistGradientBoosting ni garantiza superarlo.
- Comparar referencia mayoritaria, HistGradientBoosting con configuración anterior, XGBoost inicial y XGBoost optimizado permite distinguir el cambio de algoritmo del efecto de optimizarlo.
- Estudiar variantes con y sin artista permite cuantificar el aporte de esa información. Cada variante debe seguir el mismo protocolo; el alcance definitivo de la búsqueda se fijará antes de ejecutarla, considerando su costo.
- Conservar K-Means con K=7 como referencia permite revisar si otra configuración mejora el equilibrio entre estructura, estabilidad e interpretación musical. DBSCAN es una alternativa de densidad.

No se pretende predecir éxito futuro ni demostrar que una característica causa popularidad. El dataset representa popularidad registrada y no aporta aquí una evaluación temporal independiente.

## 2. Objetivo de clasificación y KPI

La respuesta se construirá únicamente en entrenamiento: `1` si popularidad ≥60; `0` si popularidad <60. Popularidad no se entrega como predictor.

| Indicador | Cálculo simple | Meta y lectura |
|---|---|---|
| Precisión positiva | TP / (TP + FP) | ≥0,30: al menos 30% de las canciones señaladas son positivas reales. |
| Recall positivo | TP / (TP + FN) | ≥0,60: detectar al menos 60% de las positivas reales. |
| F1 positivo | 2 × precisión × recall / (precisión + recall) | Comparar equilibrio; sin meta absoluta adicional. |
| ROC-AUC | Capacidad de ordenar positivas sobre negativas a distintos cortes | Complementario, sin meta inventada; no es accuracy ni calibración. |
| Average Precision | Resume la relación precisión–recall a distintos cortes | Complementario para la clase minoritaria; no confundir con F1 ni con cualquier área trapezoidal PR. |
| Accuracy | (TP + TN) / total | Descriptivo; no criterio principal por el desbalance de clases. |

TP: positivas detectadas; FP: falsas alarmas; FN: positivas omitidas; TN: negativas correctamente descartadas. Se reportará también la matriz de confusión y el número de positivas de cada partición.

**Regla de cumplimiento:** precisión y recall deben alcanzar sus metas simultáneamente en el conjunto de predicciones evaluado. Si alguna falla, se informa «No cumple», con sus errores y posibles motivos. No se cambia la meta después de observar el resultado.

**Umbral:** 60 define la clase real; un umbral de probabilidad define la decisión del modelo. El corte 0,30 de HistGradientBoosting no se hereda automáticamente a XGBoost. Los cortes se elegirán dentro de los datos internos permitidos: entre los que cumplan ambas metas, mayor F1; en empate, mayor precisión y después mayor corte. Si ninguno cumple, se conserva 0,50 como referencia y se documenta el incumplimiento. Se fijará la lista de cortes antes de calcular sus resultados.

Si no se predicen positivos, la precisión no está definida: se mostrará esa condición y, si se usa cero para el cálculo computacional, se explicitará; el KPI de precisión no se considerará cumplido.

In [1]:
# Configuración acordada antes de entrenar.
SEMILLA = 42
CORTE_POPULARIDAD = 60
META_PRECISION = 0.30
META_RECALL = 0.60
PLIEGUES_BUSQUEDA = 5
PLIEGUES_EXTERNOS = 3

print(f"Clase positiva: popularidad >= {CORTE_POPULARIDAD}")
print(f"Metas simultáneas: precisión >= {META_PRECISION:.0%}; recall >= {META_RECALL:.0%}")
print(f"Búsqueda interna: {PLIEGUES_BUSQUEDA} pliegues; evaluación externa: {PLIEGUES_EXTERNOS} rondas")

Clase positiva: popularidad >= 60
Metas simultáneas: precisión >= 30%; recall >= 60%
Búsqueda interna: 5 pliegues; evaluación externa: 3 rondas


### Cómo leer cada línea y su resultado

1. `SEMILLA = 42` fija una referencia para reproducir decisiones aleatorias; no mejora por sí misma el modelo.
2. `CORTE_POPULARIDAD = 60` guarda la definición de la clase real; todavía no crea etiquetas ni carga datos.
3. `META_PRECISION` y `META_RECALL` almacenan proporciones: 0,30 significa 30%, no 0,30%.
4. `PLIEGUES_BUSQUEDA = 5` indica cuántas rondas utiliza cada búsqueda interna.
5. `PLIEGUES_EXTERNOS = 3` define una evaluación adicional del procedimiento de selección completo, no otros tres candidatos.
6. `print` muestra los acuerdos. Las expresiones entre llaves insertan valores; `:.0%` muestra proporciones como porcentajes sin decimales.

La salida confirma la configuración escrita; **no demuestra que se hayan cumplido metas ni ejecutado validaciones**.

## 3. Datos permitidos y controles metodológicos

Antecedente de la Unidad 2: 71.728 IDs de entrenamiento y 18.012 de test. Estos números se comprobarán al cargar archivos; no se consideran verificados por escribirlos aquí.

En la siguiente etapa se necesitarán `Spotify_Base_Unidad_2.csv` y `Spotify_Particion_Unidad_2.csv`, junto al cuaderno. El segundo archivo permitirá recuperar la asignación previa: no se generará otra separación para incorporar al entrenamiento canciones que ya pertenecían a test. No se cargarán predicciones ni métricas de test para elegir configuraciones.

### 3.1. Qué significa agrupar versiones

La clave artista+título normalizados mantiene juntas posibles versiones con distintos IDs. No es un cluster y no significa agrupar todas las canciones de un artista. Una misma clave no puede aparecer simultáneamente en ajuste y validación. La clave es una aproximación: títulos diferentes o colaboradores en otro orden pueden impedir reconocer versiones relacionadas.

### 3.2. Controles que se comprobarán al implementar

| Control | Meta | Evidencia prevista |
|---|---|---|
| IDs de test usados para ajustar o seleccionar | 0 | Cruce con el manifiesto de partición previo. |
| Claves compartidas entre ajuste y validación | 0 por ronda | Intersección de conjuntos de claves. |
| Cobertura de validación en cinco pliegues | 100%, una vez por ID | Conteo de asignaciones y alineación por ID. |
| Valores faltantes tras preparación | 0 en entradas que lo requieran | Conteos después de transformar. |
| Transformaciones aprendidas con validación | 0 | Pipeline y controles del origen de cada ajuste. |
| Codificación de artista con respuesta de la propia clave | 0 | Exclusión de la clave y comprobación dentro de cada partición. |

Cada imputador, codificación y transformación se ajustará solo con los datos permitidos en esa ronda. Si artista se representa mediante proporción de positivas, se aplicará codificación con exclusión de la propia clave para las filas de ajuste; validación recibirá el mapa aprendido sin sus respuestas. La jerarquía completa debe respetarse también dentro de búsquedas anidadas. No se usará una columna de artista calculada con todas las respuestas antes de dividir.

**Limitación conservada:** la Unidad 1 ya exploró el dataset completo y test fue observado en la Unidad 2. La evaluación interna propuesta reduce el uso directo de validación durante la selección, pero no convierte estos datos en una prueba externa nueva ni elimina decisiones históricas tomadas sobre ellos.

## 4. Protocolo principal: cinco pliegues agrupados

Se usará `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`. Mantiene cada clave completa en un pliegue e intenta conservar proporciones similares de las clases. La estratificación y el equilibrio de tamaños son aproximados porque los grupos no se pueden partir.

Para una configuración fija: ajustar con cuatro pliegues y predecir el restante; repetir cinco veces. Cada ID queda fuera del ajuste del modelo que lo predice una vez.

**OOF (Out-Of-Fold)** significa predicción obtenida cuando la canción quedó en el pliegue de validación. No es el promedio de cinco predicciones por canción. Una codificación de artista también debe excluir su respuesta y su clave de cualquier aprendizaje que produzca esa predicción.

Sobre el entrenamiento completo, reservar aproximadamente 20% equivale a unas 14.346 canciones para validar y 57.382 para ajustar. No son tamaños garantizados: se mostrarán los conteos reales y ambas clases por ronda.

Cinco pliegues permiten utilizar todas las canciones para validación con cinco ajustes por configuración, y coinciden con la guía de clase. No garantizan mejores métricas que tres pliegues. Las referencias se reevaluarán en las mismas particiones nuevas: no se atribuirán diferencias frente a cifras antiguas únicamente al algoritmo.

Al combinar predicciones se reportarán métricas OOF globales y métricas por ronda. El promedio de F1 o AUC por ronda no tiene por qué ser igual al valor calculado reuniendo todas las predicciones. La dispersión entre pliegues describe variación observada; no se presentará automáticamente como intervalo de confianza.

## 5. Leave-P-Out: qué compara y por qué cuesta más

Leave-One-Out reserva una fila por ronda. Leave-P-Out reserva `p` filas y recorre **todas las combinaciones** posibles. `p` no es el número de rondas ni la cantidad total de registros usados.

En este proyecto, dejar filas sueltas podría separar versiones. La adaptación apropiada es **Leave-P-Groups-Out**, que reserva `p` claves completas. El número de registros reservados varía según los IDs de esas claves. Su implementación es `LeavePGroupsOut(n_groups=p)`.

Con N elementos y p reservados, el número de rondas es la combinación C(N,p), calculada mediante `comb(N, p)`. Mantener juntos los grupos evita esa fuga, pero no elimina el crecimiento combinatorio.

### 5.1. Cálculo de costo sin entrenar modelos

Los siguientes cálculos usan conteos históricos como escenarios de planificación. No ejecutan millones de ajustes ni calculan rendimiento.

In [2]:
from math import comb

REGISTROS_TRAIN_ANTERIOR = 71_728
CLAVES_TRAIN_ANTERIOR = 64_965
CLAVES_DEMOSTRACION = 20
CLAVES_RESERVADAS_DEMOSTRACION = 4

escenarios_validacion = [
    ("Cinco pliegues agrupados", PLIEGUES_BUSQUEDA),
    ("Leave-One-Out por fila, base completa", REGISTROS_TRAIN_ANTERIOR),
    ("Leave-P-Out por fila, p=2, base completa", comb(REGISTROS_TRAIN_ANTERIOR, 2)),
    ("Leave-P-Groups-Out, p=2, claves completas", comb(CLAVES_TRAIN_ANTERIOR, 2)),
    ("Demostración: reservar 4 de 20 claves", comb(CLAVES_DEMOSTRACION, CLAVES_RESERVADAS_DEMOSTRACION)),
]
for nombre, ajustes in escenarios_validacion:
    print(f"{nombre}: {ajustes:,} ajustes por configuración")

Cinco pliegues agrupados: 5 ajustes por configuración
Leave-One-Out por fila, base completa: 71,728 ajustes por configuración
Leave-P-Out por fila, p=2, base completa: 2,572,417,128 ajustes por configuración
Leave-P-Groups-Out, p=2, claves completas: 2,110,193,130 ajustes por configuración
Demostración: reservar 4 de 20 claves: 4,845 ajustes por configuración


### Explicación de cada línea

1. `from math import comb` importa una función de Python que cuenta combinaciones, sin necesidad de construirlas.
2. Los dos primeros conteos proceden del cierre de la Unidad 2; se comprobarán al cargar la partición.
3. `20` y `4` definen un escenario didáctico: reservar cuatro claves de veinte, aproximadamente 20% de claves, no necesariamente 20% de filas.
4. `escenarios_validacion` es una lista de pares: nombre del escenario y cantidad de ajustes.
5. `comb(N, 2)` cuenta todas las parejas posibles. No divide N entre dos.
6. `for` recorre la lista; `print` muestra cada cantidad. `:,` usa comas como separadores de miles en esta salida; no son decimales.

**Interpretación:** cinco pliegues requieren cinco ajustes por configuración. Leave-P-Out con p=2 sobre 71.728 filas requiere 2.572.417.128; por grupos, reservar dos de 64.965 claves requiere 2.110.193.130. Incluso reservar cuatro de veinte claves supone 4.845 ajustes. Estas cifras son costos de planificación; no miden exactitud ni demuestran que un método prediga mejor.

Leave-P-Out con p=1 coincide con Leave-One-Out por filas. Al aumentar p, el costo no siempre disminuye: depende de todas las combinaciones. No se generaliza que siempre aumente para cualquier p.

### 5.2. Comparación justa y decisión de esta etapa

| Aspecto | Cinco pliegues agrupados | Leave-P-Groups-Out |
|---|---|---|
| Separación | Cinco conjuntos de claves | Todas las combinaciones de p claves reservadas |
| Predicciones por canción | Una OOF por configuración fija | Muchas; la misma clave se reserva en varias rondas |
| Control de clases | Estratificación aproximada | No garantiza ambas clases en cada validación |
| Cantidad de ajustes | Cinco por configuración | C(G,p), con G claves |
| Uso acordado | Optimización y evaluación interna principal | Explicación del costo y posible demostración acotada |

**Decisión:** se eligen cinco pliegues agrupados por viabilidad, cobertura y conservación de versiones. No se ejecutará Leave-P-Groups-Out exhaustivo sobre toda la base. Esto no demuestra inferioridad predictiva de Leave-P-Out.

Una comparación predictiva didáctica, si se realiza, deberá usar **la misma muestra, las mismas variables, preparación y una configuración fija de XGBoost** en ambos métodos. Se propone estudiar una muestra de veinte claves completas y reservar cuatro con Leave-P-Groups-Out. Antes de autorizar en código las 4.845 rondas se medirá el tiempo de unos pocos ajustes y se estimará el costo; el piloto no se llamará ejecución exhaustiva ni resultado definitivo.

La muestra se seleccionará con una regla reproducible, procurando presencia de ambas clases, sin buscar canciones que favorezcan al modelo. Se guardarán las claves y los conteos. Algunas combinaciones podrían dejar una sola clase en ajuste o validación: se registrarán esas condiciones y las métricas no definidas; no se eliminarán silenciosamente rondas para mejorar promedios. No se presentará AUC cuando falte una clase.

Para resumir Leave-P-Groups-Out se distinguirán métricas por ronda y métricas por canción tras promediar sus probabilidades de validación; este último constituye un resumen de varias predicciones y no equivale a la única OOF de cinco pliegues. No se tratarán rondas solapadas como observaciones independientes.

**Estado actual:** solo se calculó el número de ajustes. La comparación de tiempos y rendimiento queda pendiente. No se afirma que se haya realizado ni se atribuye al profesor una obligación no confirmada de ejecutar toda la búsqueda exhaustiva.

## 6. Optimización de XGBoost y evaluación del procedimiento

### 6.1. Espacio inicial tomado de la guía

| Hiperparámetro | Valores | Para qué sirve |
|---|---|---|
| n_estimators | 100, 200 | Cantidad de árboles. |
| max_depth | 3, 6 | Profundidad máxima de los árboles. |
| learning_rate | 0,05; 0,10 | Tamaño de la contribución de cada etapa. |
| subsample | 0,8; 1,0 | Fracción de filas utilizada para cada árbol. |

GridSearchCV revisa todas las combinaciones; RandomizedSearchCV toma diez combinaciones del mismo espacio con semilla fija. No se garantiza encontrar un óptimo fuera de ese espacio. Se registrarán puntuaciones, parámetros, tiempo y errores de ejecución.

**Métrica principal de búsqueda:** Average Precision de la clase positiva, porque interesa el ordenamiento de una clase minoritaria sin fijar todavía un corte. Se reportará también ROC-AUC. La elección de configuraciones usa la media de la métrica interna; en empate se preferirá menor cantidad de árboles y menor profundidad, con una regla determinista para los demás valores. Es una decisión del proyecto, no una imposición de la guía. El cumplimiento de precisión/recall se evaluará después con el umbral elegido dentro del ajuste permitido.

El balanceo no se mezclará inicialmente con la búsqueda de estos cuatro hiperparámetros. Si se incorpora ponderación posteriormente, se tratará como experimento explícito y cualquier proporción de clases se calculará solo en el ajuste correspondiente. XGBoost de árboles no necesita escalamiento para funcionar; esto no elimina la obligación de ajustar imputación y codificaciones dentro de cada ronda.

In [3]:
ESPACIO_XGBOOST = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.10],
    "subsample": [0.8, 1.0],
}
combinaciones_grid = 1
for valores in ESPACIO_XGBOOST.values():
    combinaciones_grid *= len(valores)

configuraciones_random = 10
ajustes_grid_internos = combinaciones_grid * PLIEGUES_BUSQUEDA
ajustes_random_internos = configuraciones_random * PLIEGUES_BUSQUEDA
print(f"Grid: {combinaciones_grid} configuraciones; {ajustes_grid_internos} ajustes internos")
print(f"Random: {configuraciones_random} configuraciones; {ajustes_random_internos} ajustes internos")
print(f"En 3 rondas externas, ambas búsquedas suman {PLIEGUES_EXTERNOS * (ajustes_grid_internos + ajustes_random_internos)} ajustes internos por variante")

Grid: 16 configuraciones; 80 ajustes internos
Random: 10 configuraciones; 50 ajustes internos
En 3 rondas externas, ambas búsquedas suman 390 ajustes internos por variante


### Cómo interpretar el cálculo

1. `ESPACIO_XGBOOST` es un diccionario: cada nombre se asocia con dos valores candidatos. Todavía no se crea ni se ajusta un modelo.
2. Se inicia el producto en uno. `.values()` recorre las listas y `len` cuenta sus elementos; `*=` multiplica el total acumulado. El resultado es 16 combinaciones.
3. `configuraciones_random = 10` fija el presupuesto aleatorio; no son diez pliegues.
4. Multiplicar por cinco produce 80 ajustes de Grid y 50 de Random para una búsqueda interna.
5. Multiplicar su suma por tres produce **390 ajustes internos por variante** si se ejecutan ambas búsquedas dentro de las tres rondas externas. Con y sin artista serían 780 ajustes internos.

**No es el costo total:** faltan reajustes de ganadores, predicciones internas para elegir umbrales, referencias, ajuste final y cualquier repetición o demostración Leave-P-Out. La codificación de artista también incorpora trabajo adicional. El tiempo real se medirá antes de lanzar la búsqueda completa.

### 6.2. Tres rondas externas y cinco pliegues internos

El mejor resultado de búsqueda es optimista como evaluación del mismo proceso que lo seleccionó. Por ello se propone esta secuencia, siempre dentro de entrenamiento:

1. Crear tres particiones externas agrupadas y con estratificación aproximada. Cada ronda reserva aproximadamente un tercio para evaluar y deja dos tercios para desarrollar.
2. Dentro de esos dos tercios, ejecutar las búsquedas con cinco pliegues agrupados. Ajustar toda preparación dentro de cada ajuste interno. Aproximadamente, cada ajuste interno usa 38.255 filas y su validación 9.564; son estimaciones, no tamaños garantizados.
3. Elegir la configuración y el método de búsqueda usando solo sus resultados internos. Si también se elige con/sin artista, esa decisión se realiza aquí, no observando la ronda externa.
4. Generar predicciones internas de la configuración elegida para seleccionar el umbral según la regla de KPI. Como se reutilizan datos de selección, esas métricas internas son de desarrollo, no confirmatorias.
5. Reajustar el procedimiento seleccionado con toda la parte externa de ajuste y evaluar una sola vez su partición externa reservada. No escoger el corte con sus respuestas.
6. Reunir las predicciones externas: cada canción queda fuera una vez. Mostrar precisión, recall, F1, Average Precision, ROC-AUC y errores, además de resultados por ronda. Si no cumple KPI se informa sin modificar retrospectivamente la meta.

Con este protocolo se estima el desempeño del **procedimiento de selección**; pueden elegirse configuraciones o cortes distintos por ronda. No se supone un modelo único ya optimizado ni un corte global elegido usando las respuestas externas.

Para comparar métodos se usarán las mismas particiones y se identificarán las comparaciones predefinidas. Si se modifican reglas después de ver resultados externos, se registrará como exploración adicional; la validación anidada tampoco evita el sesgo de decisiones humanas repetidas.

Al terminar la evaluación, la búsqueda final puede ajustarse sobre todo el entrenamiento, ahora con unas 14.346 filas de validación por pliegue. Ese ajuste prepara el modelo final pero no produce otra prueba independiente. El test histórico seguirá sin utilizarse en esta unidad mientras se mantenga este alcance.

## 7. Clustering: objetivos, criterios y protocolo propio

La optimización de clustering utiliza exclusivamente entrenamiento y estas nueve variables de audio: danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence y tempo. No incluye popularidad, clase, artista, géneros ni IDs como entradas del agrupamiento. Imputación y escalamiento se aprenden en la parte de ajuste.

### 7.1. Criterios de evaluación

| Criterio | Lectura | Meta o estado |
|---|---|---|
| Silhouette | Mayor indica, en general, mejor cohesión y separación en el espacio evaluado. | Comparativo, sin umbral universal de aprobación. |
| Davies–Bouldin | Menor indica, en general, grupos más compactos y separados según esa medida. | Complementario; no demuestra utilidad musical. |
| Tamaños y perfiles | Detectan grupos dominantes y permiten describir diferencias musicales. | Reportar cantidades, proporciones y perfiles. |
| Estabilidad | Compara agrupamientos al cambiar semillas o muestras. | Reportar evidencia; sin meta nueva arbitraria. |
| Ruido de DBSCAN | Fracción no asignada, etiqueta -1. | Reportar junto a concentración y grupos válidos. |
| Utilidad para playlists | Requiere evaluar sugerencias y experiencia de uso. | Pendiente; las métricas internas no la prueban. |

Se conserva K=7 como referencia y se propone comparar K=5, 6, 7, 8 y 9. El rango responde al objetivo de explorar varios perfiles; no se afirma que K=2 sea matemáticamente inválido. No se elegirá únicamente el máximo silhouette ni se inventarán límites después de ver resultados.

### 7.2. Evaluación distinta de clasificación

- K-Means: ajustar imputación, escalamiento y centros en la parte de ajuste; transformar y asignar validación a centros fijos. Calcular métricas solo cuando haya suficientes grupos y observaciones, indicando tamaños de muestras y semillas.
- Estabilidad: repetir ajustes con semillas y remuestreos definidos previamente, conservando versiones juntas. Usar medidas que no confundan la numeración arbitraria de clusters con identidad musical.
- DBSCAN: estudiar parámetros, ruido, concentración y estabilidad. No tiene un `predict` nativo equivalente al de K-Means; reajustarlo en validación describe otra estructura y no una predicción del modelo anterior.
- PCA: visualizar, sin confundir separación en dos dimensiones con evaluación completa de los grupos.

No se copiará sin adaptación el ejemplo de la guía que vuelve a ajustar el clusterizador sobre cada validación. La propia guía advierte que esa demostración no mide la generalización del modelo ajustado en las otras particiones. Leave-P-Out con validaciones diminutas tampoco permite evaluar adecuadamente silhouette; el clustering tendrá su justificación y presupuesto propios.

## 8. Estado de esta etapa y continuación

| Elemento | Estado |
|---|---|
| Objetivos, clases y KPI | Definidos antes de entrenar. |
| Protocolo principal y prevención de fuga | Documentados; implementación pendiente. |
| Costo combinatorio de Leave-P-Out | Calculado con Python, sin entrenamientos. |
| Comparación predictiva Leave-P-Out frente a cinco pliegues | No ejecutada; condicionada a viabilidad del piloto. |
| Carga exclusiva de entrenamiento y controles de archivos | Ejecutados en el bloque 9. |
| Entrenamiento de XGBoost y búsquedas | Pendiente. |
| Optimización y evaluación de clustering | Pendiente. |
| Cumplimiento predictivo de KPI | Pendiente; no hay métricas nuevas. |

**Actualización tras el bloque 9:** la carga y los controles ya están ejecutados. El siguiente paso será implementar las particiones agrupadas y la preparación antes de entrenar XGBoost.

La evidencia de esta etapa distingue acuerdos, cálculos de costo y resultados ejecutados. Esa distinción se conservará en las conclusiones y en el informe final.

### Fuentes y alcance de sus indicaciones

- Material de la Unidad 3 aportado por las estudiantes: optimización, ensambles y evaluación. Es un resumen de clase; no reemplaza una rúbrica aún no proporcionada.
- `guia_hiperparametros_grid_random_cv5_editable.pdf`, páginas 1–5: espacios de búsqueda, cinco pliegues y advertencias del ejemplo de clustering. Sus ejemplos no obligan a entrenar todos los algoritmos enumerados.
- [StratifiedGroupKFold, scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html): estratificación aproximada manteniendo grupos intactos.
- [LeavePOut, scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeavePOut.html): combinaciones exhaustivas por filas.
- [LeavePGroupsOut, scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeavePGroupsOut.html): combinaciones exhaustivas por grupos.
- [Validación anidada, scikit-learn](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html): separar selección y evaluación.
- [Parámetros de XGBoost](https://xgboost.readthedocs.io/en/stable/parameter.html): significado de las configuraciones del algoritmo.

La elección de clasificación y clustering y el uso de XGBoost proceden del alcance indicado por las estudiantes. La recomendación del profesor sobre registros reservados se documenta sin atribuirle un valor de p ni un número de rondas que no se haya confirmado.

## 9. Carga de la base preparada y recuperación de entrenamiento

### 9.1. Qué archivos usamos y por qué

`Spotify_Base_Unidad_2.csv` contiene una fila por ID de canción, sus características, géneros Multi-Hot y la clave artista+título. `Spotify_Particion_Unidad_2.csv` conserva la asignación previa de cada ID a entrenamiento o `prueba_interna`.

Se mantienen ambos archivos junto a este cuaderno. No se usa el CSV original ni se repite la limpieza. Tampoco se vuelve a sortear una partición: se recupera la anterior por ID. La clave agrupa posibles versiones; un ID identifica una fila de canción.

Primero leeremos únicamente el manifiesto de partición. Revisar sus IDs y claves de test es un control de separación, no una evaluación de sus respuestas o características. Al leer la base completa se seleccionará inmediatamente entrenamiento y se descartará la tabla temporal; no se calcularán estadísticas, etiquetas ni métricas sobre test. Los CSV se abren en modo lectura y no se sobrescriben.

In [4]:
from pathlib import Path
import pandas as pd

carpeta_datos = Path(".")
ruta_base = carpeta_datos / "Spotify_Base_Unidad_2.csv"
ruta_particion = carpeta_datos / "Spotify_Particion_Unidad_2.csv"
for ruta in [ruta_base, ruta_particion]:
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta {ruta.name}: colócalo junto al cuaderno y ejecuta desde esa carpeta.")

particion_u2 = pd.read_csv(ruta_particion, dtype="string")
columnas_particion = {"track_id", "clave_cancion", "conjunto"}
if not columnas_particion.issubset(particion_u2.columns):
    raise ValueError("El manifiesto no contiene las columnas esperadas.")
if particion_u2[list(columnas_particion)].isna().any().any():
    raise ValueError("Hay identificadores, claves o asignaciones faltantes.")
if particion_u2[list(columnas_particion)].apply(lambda columna: columna.str.strip().eq("").any()).any():
    raise ValueError("Hay identificadores, claves o asignaciones vacías.")
if particion_u2["track_id"].duplicated().any():
    raise ValueError("Un ID aparece más de una vez en el manifiesto.")
if set(particion_u2["conjunto"]) != {"entrenamiento", "prueba_interna"}:
    raise ValueError("Las etiquetas de partición no coinciden con las de la Unidad 2.")

ids_train_u2 = pd.Index(particion_u2.loc[particion_u2["conjunto"].eq("entrenamiento"), "track_id"])
ids_test_u2 = pd.Index(particion_u2.loc[particion_u2["conjunto"].eq("prueba_interna"), "track_id"])
print(f"IDs de entrenamiento en el manifiesto: {len(ids_train_u2):,}")
print(f"IDs compartidos entre entrenamiento y test: {len(ids_train_u2.intersection(ids_test_u2))}")

IDs de entrenamiento en el manifiesto: 71,728
IDs compartidos entre entrenamiento y test: 0


### Explicación del código anterior

1. `Path` permite construir rutas y `pandas` leer y organizar tablas. `Path(".")` indica la carpeta desde la que se ejecuta el cuaderno; debe ser donde están los archivos.
2. `/` añade cada nombre de archivo a esa carpeta. No descarga ni modifica nada.
3. El bucle `for` revisa ambas rutas. `is_file()` comprueba que existen; `raise FileNotFoundError` detiene la ejecución si falta alguna.
4. `read_csv(..., dtype="string")` lee el manifiesto como texto, preservando IDs y claves. No confunde un identificador con una cantidad.
5. El conjunto `columnas_particion` enumera los campos obligatorios. `issubset` comprueba que estén presentes; no exige un orden específico.
6. `isna().any().any()` detecta si existe algún dato faltante en esos campos. El primer `any` resume por columna y el segundo resume toda la tabla.
7. `apply` revisa cada columna; `str.strip().eq("")` detecta campos vacíos o formados solo por espacios. Es un control, no una limpieza silenciosa.
8. `duplicated().any()` detecta IDs repetidos. Si aparecen, detenemos el proceso en vez de borrarlos automáticamente.
9. `set(...)` comprueba que las etiquetas sean exactamente las esperadas. No renombramos etiquetas desconocidas para hacer pasar el control.
10. `.eq(...)` crea una máscara de verdadero/falso; `.loc` selecciona los IDs de cada partición e `Index` los guarda como identificadores.
11. `intersection` busca IDs compartidos. La salida debe ser cero. Los separadores con coma en los conteos son de miles.

**Cómo interpretar:** esperamos 71.728 IDs de entrenamiento y cero compartidos. Esto recupera la asignación histórica, pero todavía no comprueba que los IDs existan en la base preparada ni que las versiones estén separadas correctamente.

In [5]:
base_temporal_u2 = pd.read_csv(
    ruta_base,
    dtype={"track_id": "string", "clave_cancion": "string", "time_signature": "Int64"},
)
if not {"track_id", "clave_cancion", "popularity"}.issubset(base_temporal_u2.columns):
    raise ValueError("Faltan columnas necesarias en la base preparada.")
if base_temporal_u2["track_id"].isna().any() or base_temporal_u2["track_id"].duplicated().any():
    raise ValueError("La base contiene IDs faltantes o repetidos.")
ids_base_u2 = pd.Index(base_temporal_u2["track_id"])
if set(ids_base_u2) != set(particion_u2["track_id"]):
    raise ValueError("La base y el manifiesto no contienen exactamente los mismos IDs.")

base_temporal_u2 = base_temporal_u2.set_index("track_id", verify_integrity=True)
datos_entrenamiento_u3 = base_temporal_u2.loc[ids_train_u2].copy()
del base_temporal_u2

clave_manifiesto_train = particion_u2.set_index("track_id").loc[ids_train_u2, "clave_cancion"]
if not datos_entrenamiento_u3["clave_cancion"].equals(clave_manifiesto_train):
    raise ValueError("Las claves de entrenamiento no coinciden con el manifiesto por ID.")
if len(datos_entrenamiento_u3) != 71_728:
    raise ValueError("Cambió el tamaño histórico de entrenamiento; revisar archivos antes de continuar.")
print(f"Base de trabajo: {len(datos_entrenamiento_u3):,} canciones de entrenamiento")
print(f"Columnas conservadas: {datos_entrenamiento_u3.shape[1]}")
print(f"IDs duplicados en entrenamiento: {datos_entrenamiento_u3.index.duplicated().sum()}")
print(f"IDs de test dentro de la base de trabajo: {len(datos_entrenamiento_u3.index.intersection(ids_test_u2))}")

Base de trabajo: 71,728 canciones de entrenamiento
Columnas conservadas: 130
IDs duplicados en entrenamiento: 0
IDs de test dentro de la base de trabajo: 0


### Explicación de la carga y selección

1. `read_csv` lee la base preparada en una tabla temporal. Los IDs y claves se fuerzan a texto; `Int64` conserva compás como entero que admite faltantes, evitando convertir categorías como `4` en `4.0`.
2. `issubset` comprueba las columnas esenciales. Los siguientes controles verifican IDs no faltantes y únicos, sin analizar la popularidad de test.
3. `set(ids_base_u2) != set(...)` detecta cualquier diferencia de cobertura entre base y manifiesto. La unicidad de ambos archivos se comprobó antes: no se ocultan duplicados mediante los conjuntos.
4. `set_index("track_id", verify_integrity=True)` usa el ID como índice y vuelve a impedir duplicados. El ID deja de contarse como columna; no desaparece su información.
5. `.loc[ids_train_u2].copy()` recupera únicamente los IDs asignados previamente a entrenamiento y crea la tabla de trabajo independiente.
6. `del` elimina la referencia a la tabla temporal completa. No borra el archivo ni constituye por sí solo evidencia de ausencia de fuga: la evidencia está en los controles y en no utilizar test para cálculos o selección.
7. `clave_manifiesto_train` recupera la clave esperada para cada ID en el mismo orden. `.equals` compara índice y contenido; no compara filas solo por posición accidental.
8. La comprobación de 71.728 detecta cambios inesperados respecto de la entrega anterior; no hace que ese número sea una condición universal de Machine Learning.
9. `shape[1]` cuenta columnas, `index.duplicated().sum()` cuenta IDs repetidos y `intersection` revisa presencia de IDs de test.

**Resultado esperado:** 71.728 canciones, 130 columnas, cero IDs duplicados y cero IDs de test. Las 130 columnas incluyen respuesta, clave y variables que después seleccionaremos o excluiremos: **no son 130 predictores**. No se han imputado, escalado ni codificado artistas.

In [6]:
claves_train_u3 = datos_entrenamiento_u3["clave_cancion"]
claves_test_manifiesto = particion_u2.loc[particion_u2["conjunto"].eq("prueba_interna"), "clave_cancion"]
claves_compartidas = set(claves_train_u3).intersection(set(claves_test_manifiesto))
particiones_por_clave = particion_u2.groupby("clave_cancion")["conjunto"].nunique()
if claves_compartidas or particiones_por_clave.gt(1).any():
    raise ValueError("Una clave de canción tiene versiones repartidas entre entrenamiento y test.")

tamanos_versiones_train = datos_entrenamiento_u3.groupby("clave_cancion").size()
claves_con_versiones_train = tamanos_versiones_train[tamanos_versiones_train.gt(1)]
print(f"Claves distintas en entrenamiento: {len(tamanos_versiones_train):,}")
print(f"Claves con más de un ID en entrenamiento: {len(claves_con_versiones_train):,}")
print(f"IDs pertenecientes a esas claves: {claves_con_versiones_train.sum():,}")
print(f"Claves compartidas con test: {len(claves_compartidas)}")

claves_ejemplo = claves_con_versiones_train.sort_index().head(3).index
versiones_ejemplo = (
    datos_entrenamiento_u3.loc[claves_train_u3.isin(claves_ejemplo)]
    .reset_index()[["track_id", "clave_cancion"]]
    .sort_values(["clave_cancion", "track_id"])
)
display(versiones_ejemplo)

Claves distintas en entrenamiento: 64,965
Claves con más de un ID en entrenamiento: 3,777
IDs pertenecientes a esas claves: 10,540
Claves compartidas con test: 0


,track_id,clave_cancion
4,5h7RBt7uQCF788tr9Zy92j,(hed) p.e. || crazy legs
5,5igKQzi92vVl9Pjj86d42O,(hed) p.e. || crazy legs
0,2jzxWQlR0pZ2IUvk95nsfX,2 brothers on the 4th floor || dreams (will come alive)
2,4Fqm5rqB0WMx0hPOwgRjFr,2 brothers on the 4th floor || dreams (will come alive)
1,318kgdwBNFiVKI4tYzlNG7,2 brothers on the 4th floor || fairytales - charly lownoise & mental theo rave edit
3,5W5ri804RuPOSHdZfAcFjO,2 brothers on the 4th floor || fairytales - charly lownoise & mental theo rave edit


### Cómo comprobar e interpretar las versiones

1. `claves_train_u3` conserva una clave por ID de entrenamiento. El manifiesto proporciona únicamente las claves reservadas a test para controlar su separación.
2. La intersección de conjuntos encuentra claves compartidas. `groupby(...).nunique()` cuenta en cuántas particiones aparece cada clave; `.gt(1)` detecta si cruza la separación.
3. Si cualquiera de esos controles falla, se detiene el proceso. No se reasignan versiones para corregir retrospectivamente la partición sin revisar su origen.
4. `groupby("clave_cancion").size()` cuenta cuántos IDs hay por clave en entrenamiento. Filtrar con `.gt(1)` conserva claves con varias filas.
5. `len(claves_con_versiones_train)` cuenta **claves**, mientras `.sum()` cuenta **IDs que pertenecen a ellas**. No son medidas intercambiables ni un conteo de duplicados exactos.
6. `sort_index().head(3)` elige tres ejemplos por orden alfabético, sin usar popularidad o resultados del modelo.
7. `isin` selecciona sus filas; `reset_index` vuelve visible el ID como columna. Se muestran solo ID y clave, y se ordenan para facilitar la comparación.

**Lectura de la tabla real:** dos filas con igual clave y distintos IDs no son IDs duplicados. Pueden representar publicaciones o versiones relacionadas, por lo que se conservan y se mantendrán juntas en las validaciones posteriores. La clave por sí sola no demuestra que sus audios sean idénticos. Repetir artista con otro título tampoco implica duplicación.

Cero claves compartidas demuestra que la **clave utilizada** no cruza entrenamiento y test. No demuestra ausencia de cualquier relación musical entre ambas partes, separación por artista completo ni reconocimiento perfecto de todas las reediciones.

**Resultado ejecutado:** se recuperaron 64.965 claves de entrenamiento; 3.777 tienen más de un ID y reúnen 10.540 canciones. Las otras 61.188 claves contienen un único ID. Estas cantidades suman los 71.728 IDs recuperados. No hay claves compartidas con test.


In [7]:
popularidad_train_u3 = pd.to_numeric(datos_entrenamiento_u3["popularity"], errors="raise")
if popularidad_train_u3.isna().any() or not popularidad_train_u3.between(0, 100).all():
    raise ValueError("Popularidad faltante o fuera del rango 0–100 en entrenamiento.")
y_train_u3 = popularidad_train_u3.ge(CORTE_POPULARIDAD).astype("int8")
y_train_u3.name = "popularidad_alta"
if not y_train_u3.index.equals(datos_entrenamiento_u3.index):
    raise ValueError("Las respuestas no están alineadas con los IDs de entrenamiento.")
if set(y_train_u3.unique()) != {0, 1}:
    raise ValueError("Entrenamiento debe contener ambas clases.")

resumen_clases_train_u3 = y_train_u3.value_counts().reindex([0, 1], fill_value=0).rename("Canciones").to_frame()
resumen_clases_train_u3.index.name = "Clase"
resumen_clases_train_u3["Porcentaje"] = resumen_clases_train_u3["Canciones"].div(len(y_train_u3)).mul(100)
resumen_clases_train_u3["Significado"] = ["Popularidad menor que 60", "Popularidad de 60 o más"]
display(resumen_clases_train_u3.round({"Porcentaje": 2}))
print(f"Respuestas construidas solo para entrenamiento: {len(y_train_u3):,}")
print(f"Claves con ambas clases entre sus versiones: {y_train_u3.groupby(claves_train_u3).nunique().gt(1).sum():,}")

,Canciones,Porcentaje,Significado
Clase,,,
0,63867,89.04,Popularidad menor que 60
1,7861,10.96,Popularidad de 60 o más


Respuestas construidas solo para entrenamiento: 71,728
Claves con ambas clases entre sus versiones: 895


### Explicación de las clases y sus resultados

1. `pd.to_numeric(..., errors="raise")` exige popularidad numérica; un texto inválido genera error en vez de convertirse silenciosamente en una respuesta faltante.
2. `isna` detecta faltantes y `between(0, 100).all()` exige el rango completo válido. Se verifica únicamente entrenamiento.
3. `.ge(CORTE_POPULARIDAD)` significa «mayor o igual que 60». Produce verdadero/falso; `.astype("int8")` lo convierte en 1/0 usando un tipo entero pequeño. No modifica el puntaje original.
4. Asignar `.name` pone un nombre a la respuesta. Comprobar `.index.equals(...)` asegura que cada etiqueta corresponde a su canción; `.unique()` comprueba presencia de ambas clases.
5. `value_counts` cuenta canciones por clase; `reindex([0, 1])` fija el orden y `rename(...).to_frame()` construye una tabla con nombres claros.
6. `.div(len(y_train_u3)).mul(100)` divide cada conteo por el total y lo transforma en porcentaje. `round` redondea solo la tabla mostrada.
7. El último `groupby` cuenta clases diferentes dentro de cada clave. Una clave puede contener versiones con popularidades a ambos lados de 60. Se mantienen sus etiquetas por ID y se conserva la clave completa en el mismo pliegue; no se fuerza una respuesta común.

**Resultado histórico a comprobar:** 63.867 canciones de clase 0 y 7.861 de clase 1, aproximadamente 89,04% y 10,96%. La clase positiva es minoritaria; esto explica por qué accuracy sola puede engañar. Predecir siempre clase 0 acertaría muchas negativas, pero no detectaría ninguna positiva.

Esta tabla describe la distribución, **no el desempeño de XGBoost**. No hay todavía precisión, recall o F1 de un modelo, ni evidencia de cumplimiento predictivo de KPI. No se ha aplicado balanceo ni se han eliminado canciones por su clase.

**Resultado ejecutado:** se confirmaron los 63.867 negativos y 7.861 positivos. Hay 895 claves cuyas versiones incluyen ambas clases. No es un error lógico: el objetivo pertenece a cada ID y las versiones pueden tener distinta popularidad. Mantenerlas juntas evita que esa relación facilite artificialmente la validación; no exige igualar sus etiquetas.


In [8]:
controles_carga_u3 = pd.DataFrame([
    {"Control": "IDs recuperados frente al manifiesto", "Meta": "Coincidencia exacta", "Resultado": datos_entrenamiento_u3.index.equals(ids_train_u2)},
    {"Control": "IDs repetidos en entrenamiento", "Meta": "0", "Resultado": not datos_entrenamiento_u3.index.duplicated().any()},
    {"Control": "IDs de test en la base de trabajo", "Meta": "0", "Resultado": len(datos_entrenamiento_u3.index.intersection(ids_test_u2)) == 0},
    {"Control": "Claves compartidas entre entrenamiento y test", "Meta": "0", "Resultado": len(claves_compartidas) == 0},
    {"Control": "Etiquetas alineadas y ambas clases presentes", "Meta": "Sí", "Resultado": y_train_u3.index.equals(datos_entrenamiento_u3.index) and set(y_train_u3.unique()) == {0, 1}},
])
if not controles_carga_u3["Resultado"].all():
    raise ValueError("Hay controles de carga incumplidos.")
controles_carga_u3["Estado"] = controles_carga_u3["Resultado"].map({True: "Cumple", False: "Revisar"})
display(controles_carga_u3[["Control", "Meta", "Estado"]])

,Control,Meta,Estado
0,IDs recuperados frente al manifiesto,Coincidencia exacta,Cumple
1,IDs repetidos en entrenamiento,0,Cumple
2,IDs de test en la base de trabajo,0,Cumple
3,Claves compartidas entre entrenamiento y test,0,Cumple
4,Etiquetas alineadas y ambas clases presentes,Sí,Cumple


### Cierre del bloque y siguiente paso

`pd.DataFrame` reúne condiciones que ya se comprobaron. Cada diccionario representa una fila con control, meta y condición booleana. `.all()` exige que todos sean verdaderos; `map` convierte verdadero/falso a una etiqueta legible. Se muestran las columnas de interés con `display`.

«Cumple» aquí significa **control metodológico de carga superado**, no KPI predictivo cumplido. No prueba por sí solo que futuros ajustes estén libres de fuga: habrá que repetir controles específicos en cada validación.

Quedan disponibles `datos_entrenamiento_u3` (base por ID), `claves_train_u3` (grupos de versiones), `y_train_u3` (respuesta binaria), `resumen_clases_train_u3` y `controles_carga_u3` (evidencias). No se ha creado aún la matriz definitiva de predictores: popularidad y clave deben excluirse, y artista debe prepararse dentro de cada ajuste.

**Actualización tras el bloque 10:** los cinco pliegues y sus controles están ejecutados. El siguiente paso será implementar la preparación y las referencias antes de entrenar XGBoost. Los CSV originales y los cuadernos de Unidades 1 y 2 permanecen sin cambios.

## 10. Creación y comprobación de cinco pliegues agrupados

### 10.1. Qué vamos a separar

Usaremos los 71.728 IDs de `datos_entrenamiento_u3`, sus respuestas `y_train_u3` y sus claves `claves_train_u3`, ya comprobados. No se incorporan filas de test.

`StratifiedGroupKFold` combina dos condiciones: cada clave permanece completa en un pliegue y se intenta conservar una proporción similar de clases. La primera condición es obligatoria; el equilibrio de clases y tamaños es aproximado. Una clave puede tener versiones de clases distintas y sigue siendo un grupo indivisible.

Se crearán cinco rondas. En cada una se ajustará con cuatro pliegues y se validará con el restante. Aquí **solo guardamos los índices de las separaciones**: no ajustamos imputadores, codificaciones ni modelos.

Estos cinco pliegues cubren todo el entrenamiento y servirán para referencias y búsquedas sobre ese conjunto. **No sustituyen las tres rondas externas del protocolo anidado:** al implementarlas se crearán cinco pliegues internos nuevos exclusivamente dentro del ajuste de cada ronda externa. No se reutilizarán estas separaciones globales como si excluyeran automáticamente la validación externa.

Para ejecutar este bloque se necesita scikit-learn. La ejecución verificada usa la versión 1.6.1, igual que la Unidad 2; se registrará la versión usada. No se modifica la instalación desde estas celdas.

In [9]:
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

if not datos_entrenamiento_u3.index.equals(y_train_u3.index):
    raise ValueError("Los IDs y las respuestas no están alineados.")
if not datos_entrenamiento_u3.index.equals(claves_train_u3.index):
    raise ValueError("Los IDs y las claves no están alineados.")
if claves_train_u3.isna().any():
    raise ValueError("Hay claves faltantes.")

cv_cinco_u3 = StratifiedGroupKFold(
    n_splits=PLIEGUES_BUSQUEDA,
    shuffle=True,
    random_state=SEMILLA,
)
particiones_cinco_u3 = list(cv_cinco_u3.split(
    X=datos_entrenamiento_u3.index.to_numpy().reshape(-1, 1),
    y=y_train_u3.to_numpy(),
    groups=claves_train_u3.to_numpy(),
))
print(f"Versión de scikit-learn: {sklearn.__version__}")
print(f"Rondas creadas: {len(particiones_cinco_u3)}")
print("Solo se generaron posiciones de ajuste y validación; no se entrenaron modelos.")

Versión de scikit-learn: 1.6.1
Rondas creadas: 5
Solo se generaron posiciones de ajuste y validación; no se entrenaron modelos.


### Explicación de cada línea

1. `import sklearn` permite consultar la versión y `from ... import StratifiedGroupKFold` importa el separador de datos; no es un modelo predictivo.
2. Los dos controles `.index.equals` exigen el mismo ID y orden entre la tabla, las respuestas y las claves. De otro modo, una etiqueta podría asociarse a otra canción.
3. `isna().any()` impide construir separaciones con claves ausentes.
4. `n_splits=PLIEGUES_BUSQUEDA` utiliza los cinco pliegues acordados. `shuffle=True` activa el barajado compatible con este método; no es una mezcla irrestricta de filas porque debe respetar grupos y su procedimiento de asignación. `random_state=SEMILLA` permite reproducirlo con los mismos datos, orden y versión.
5. `.split(...)` produce pares de posiciones: unas para ajuste y otras para validación. `list` conserva los cinco pares para reutilizar exactamente las mismas rondas entre modelos.
6. `index.to_numpy().reshape(-1, 1)` proporciona una entrada con una fila por ID. El separador usa su cantidad de filas, las clases y los grupos; **no aprende relaciones a partir de los IDs ni usa popularidad como predictor**. `-1` permite calcular automáticamente el número de filas y `1` indica una columna auxiliar.
7. `y_train_u3.to_numpy()` aporta las clases para intentar equilibrarlas; `groups=...` aporta las claves que no se pueden dividir.
8. Los `print` muestran versión y número de rondas. Crear cinco rondas no demuestra todavía que sus controles se cumplan: eso se revisa a continuación.

Las posiciones se usarán mediante `.iloc`, que selecciona por posición. No deben aplicarse después de reordenar las filas sin regenerar o realinear las separaciones.

In [10]:
filas_resumen_pliegues = []
filas_controles_pliegues = []
asignacion_pliegue_u3 = pd.Series(pd.NA, index=datos_entrenamiento_u3.index, dtype="Int64", name="pliegue_validacion")
veces_validado_u3 = pd.Series(0, index=datos_entrenamiento_u3.index, dtype="int64")

for ronda, (pos_ajuste, pos_validacion) in enumerate(particiones_cinco_u3, start=1):
    ids_ajuste = datos_entrenamiento_u3.index[pos_ajuste]
    ids_validacion = datos_entrenamiento_u3.index[pos_validacion]
    y_ajuste = y_train_u3.iloc[pos_ajuste]
    y_validacion = y_train_u3.iloc[pos_validacion]
    claves_ajuste = set(claves_train_u3.iloc[pos_ajuste])
    claves_validacion = set(claves_train_u3.iloc[pos_validacion])
    claves_cruzadas = len(claves_ajuste.intersection(claves_validacion))
    ids_cruzados = len(ids_ajuste.intersection(ids_validacion))
    cobertura_ronda = len(ids_ajuste.union(ids_validacion)) == len(datos_entrenamiento_u3)
    sin_test = len(ids_ajuste.union(ids_validacion).intersection(ids_test_u2)) == 0
    ambas_clases = set(y_ajuste.unique()) == {0, 1} and set(y_validacion.unique()) == {0, 1}
    if claves_cruzadas != 0 or ids_cruzados != 0 or not cobertura_ronda or not sin_test or not ambas_clases:
        raise ValueError(f"La ronda {ronda} no cumple los controles de separación.")
    veces_validado_u3.loc[ids_validacion] += 1
    asignacion_pliegue_u3.loc[ids_validacion] = ronda
    filas_resumen_pliegues.append({
        "Ronda": ronda,
        "IDs ajuste": len(ids_ajuste),
        "IDs validación": len(ids_validacion),
        "Claves ajuste": len(claves_ajuste),
        "Claves validación": len(claves_validacion),
        "Negativas ajuste": int(y_ajuste.eq(0).sum()),
        "Positivas ajuste": int(y_ajuste.eq(1).sum()),
        "Negativas validación": int(y_validacion.eq(0).sum()),
        "Positivas validación": int(y_validacion.eq(1).sum()),
        "% positivas ajuste": y_ajuste.mean() * 100,
        "% positivas validación": y_validacion.mean() * 100,
    })
    filas_controles_pliegues.append({
        "Ronda": ronda, "IDs compartidos": ids_cruzados,
        "Claves compartidas": claves_cruzadas, "Cobertura completa": cobertura_ronda,
        "Sin IDs test": sin_test, "Ambas clases": ambas_clases,
    })

resumen_pliegues_u3 = pd.DataFrame(filas_resumen_pliegues).set_index("Ronda")
controles_pliegues_u3 = pd.DataFrame(filas_controles_pliegues).set_index("Ronda")
resumen_pliegues_u3["Diferencia positivas vs. train (pp)"] = resumen_pliegues_u3["% positivas validación"] - y_train_u3.mean() * 100
display(resumen_pliegues_u3.round(3))
display(controles_pliegues_u3)

,IDs ajuste,IDs validación,Claves ajuste,Claves validación,Negativas ajuste,Positivas ajuste,Negativas validación,Positivas validación,% positivas ajuste,% positivas validación,Diferencia positivas vs. train (pp)
Ronda,,,,,,,,,,,
1,57340,14388,51988,12977,51014,6326,12853,1535,11.032,10.669,-0.291
2,57348,14380,51970,12995,51048,6300,12819,1561,10.986,10.855,-0.104
3,57226,14502,51973,12992,50964,6262,12903,1599,10.943,11.026,0.067
4,57464,14264,51971,12994,51221,6243,12646,1618,10.864,11.343,0.384
5,57534,14194,51958,13007,51221,6313,12646,1548,10.973,10.906,-0.053


,IDs compartidos,Claves compartidas,Cobertura completa,Sin IDs test,Ambas clases
Ronda,,,,,
1,0,0,True,True,True
2,0,0,True,True,True
3,0,0,True,True,True
4,0,0,True,True,True
5,0,0,True,True,True


### Explicación de la revisión por ronda

1. Las listas vacías recibirán una fila por ronda. `pd.Series` crea dos registros por ID: pliegue asignado inicialmente desconocido (`pd.NA`) y número de validaciones inicialmente cero. `Int64` admite enteros faltantes; `int64` se usa para los conteos que empiezan en cero.
2. `enumerate(..., start=1)` numera del 1 al 5 los pares de posiciones. Cada par se separa en `pos_ajuste` y `pos_validacion`.
3. `index[posiciones]` recupera IDs y `.iloc[posiciones]` recupera etiquetas y claves en esas mismas posiciones. No se crean nuevos datos ni se cambia la respuesta.
4. `set` obtiene claves únicas. `intersection` cuenta las compartidas: debe ser cero tanto para IDs como para claves.
5. `union` reúne ajuste y validación. Su tamaño debe coincidir con todo entrenamiento; la intersección con los IDs del manifiesto de test debe ser vacía.
6. `set(y.unique()) == {0, 1}` comprueba ambas clases en ajuste y validación. Si falla alguno de los controles, `raise ValueError` detiene el bloque.
7. `veces_validado_u3.loc[...] += 1` suma una validación por ID reservado y `asignacion_pliegue_u3` guarda su ronda. El conteo separado permite detectar asignaciones repetidas aunque el número de ronda se sobrescribiera.
8. Cada `.append({...})` añade los resultados a una lista. `len` cuenta canciones o claves; `.eq(0).sum()` y `.eq(1).sum()` cuentan negativas y positivas.
9. Como las respuestas son 0 y 1, `.mean() * 100` calcula el porcentaje de positivas: sumar unos cuenta positivas y dividir por el número de canciones calcula su proporción.
10. `DataFrame(...).set_index("Ronda")` reúne las cinco filas. Restar el porcentaje global da una diferencia en **puntos porcentuales (pp)**, no una variación relativa en porcentaje.
11. `round(3)` redondea únicamente la tabla mostrada. `display` presenta tamaños, clases y controles.

**Lectura:** cada fila representa un futuro ajuste independiente. Sus negativas y positivas de validación deben sumar sus IDs de validación, y ajuste más validación debe sumar 71.728. Diferencias pequeñas de tamaños o porcentajes son esperables: los grupos tienen tamaños y mezclas de clases distintos. No se modifica la semilla para buscar una tabla más favorable. Esta tabla no contiene métricas predictivas y no implica haber ajustado cinco modelos.

In [11]:
pliegues_por_clave_u3 = asignacion_pliegue_u3.groupby(claves_train_u3).nunique()
if not veces_validado_u3.eq(1).all() or asignacion_pliegue_u3.isna().any():
    raise ValueError("Cada ID debe aparecer exactamente una vez en validación.")
if not pliegues_por_clave_u3.eq(1).all():
    raise ValueError("Las versiones de alguna clave quedaron en distintos pliegues.")
if set(asignacion_pliegue_u3.unique()) != set(range(1, PLIEGUES_BUSQUEDA + 1)):
    raise ValueError("Faltan pliegues esperados.")
if int(resumen_pliegues_u3["IDs validación"].sum()) != len(datos_entrenamiento_u3):
    raise ValueError("La suma de validaciones no coincide con entrenamiento.")

asignacion_validacion_u3 = pd.DataFrame({
    "clave_cancion": claves_train_u3,
    "clase": y_train_u3,
    "pliegue_validacion": asignacion_pliegue_u3,
})
claves_mixtas_u3 = y_train_u3.groupby(claves_train_u3).nunique()
claves_mixtas_ejemplo = claves_mixtas_u3[claves_mixtas_u3.gt(1)].sort_index().head(2).index
versiones_pliegue_ejemplo = (
    asignacion_validacion_u3.loc[asignacion_validacion_u3["clave_cancion"].isin(claves_mixtas_ejemplo)]
    .reset_index().sort_values(["clave_cancion", "track_id"])
)
display(versiones_pliegue_ejemplo)
print(f"IDs con exactamente una validación: {veces_validado_u3.eq(1).sum():,} de {len(veces_validado_u3):,}")
print(f"Claves cuyas versiones permanecen en un único pliegue: {pliegues_por_clave_u3.eq(1).sum():,} de {len(pliegues_por_clave_u3):,}")
print("Estado: controles metodológicos cumplidos; KPI predictivos pendientes.")

,track_id,clave_cancion,clase,pliegue_validacion
1,2jzxWQlR0pZ2IUvk95nsfX,2 brothers on the 4th floor || dreams (will come alive),0,4
3,4Fqm5rqB0WMx0hPOwgRjFr,2 brothers on the 4th floor || dreams (will come alive),1,4
0,2Lkn3Of9JHY6RSA8c3omKT,3 doors down || here without you,0,1
2,3NLrRZoMF0Lx6zTlYqeIo4,3 doors down || here without you,1,1


IDs con exactamente una validación: 71,728 de 71,728
Claves cuyas versiones permanecen en un único pliegue: 64,965 de 64,965
Estado: controles metodológicos cumplidos; KPI predictivos pendientes.


### Explicación de cobertura y ejemplos reales

1. `asignacion_pliegue_u3.groupby(claves_train_u3).nunique()` cuenta cuántos pliegues distintos tiene cada clave. El valor correcto es uno, incluso si hay varias versiones y sus clases difieren.
2. `veces_validado_u3.eq(1).all()` exige una validación por ID. `isna().any()` detecta canciones sin pliegue asignado. Se comprueba el conteo por ID, no solo una suma global que podría ocultar repeticiones y ausencias.
3. `range(1, PLIEGUES_BUSQUEDA + 1)` genera 1, 2, 3, 4 y 5; comparar conjuntos exige que existan todos los pliegues. La suma de IDs de validación añade un control de consistencia.
4. `asignacion_validacion_u3` une clave, clase y pliegue usando el índice de ID compartido. Es una tabla de seguimiento; **no debe entregarse como matriz de predictores**. Ni clase ni número de pliegue ni clave son características musicales para el modelo.
5. `y_train_u3.groupby(...).nunique()` localiza claves con ambas clases. `.gt(1)`, `sort_index()` y `head(2)` eligen dos ejemplos alfabéticos reproducibles, sin buscar resultados favorables.
6. `isin` recupera todas sus versiones; `reset_index` muestra el ID y `sort_values` ordena la presentación.
7. Los `print` reúnen evidencia de cobertura y conservación de versiones. No constituyen precisión, recall ni cumplimiento de los KPI predictivos.

**Cómo leer los ejemplos:** cada clave puede tener distintos IDs y clases 0 y 1, pero todos sus IDs deben mostrar el mismo número de pliegue. En la ronda correspondiente se reservan juntos; en las otras cuatro se usan juntos para ajuste. Mantener las versiones juntas no exige borrar filas ni igualar su popularidad.

Estas comprobaciones verifican la clave utilizada, no garantizan que reconozca todas las posibles versiones ni que los artistas completos estén separados.

### 10.2. Estado y siguiente paso

Quedan guardados en memoria `particiones_cinco_u3` (cinco pares de posiciones), `asignacion_validacion_u3` (seguimiento por ID), `resumen_pliegues_u3` y `controles_pliegues_u3`. Los modelos comparados sobre todo entrenamiento deberán reutilizar estos mismos pares sin reordenar la tabla. La semilla y versión se registran para reproducibilidad; no son parámetros que se optimicen para mejorar métricas.

No se exporta una nueva base ni se modifica el manifiesto previo. Tampoco se han calculado probabilidades OOF: solo existe su estructura de validación. Las probabilidades se producirán al entrenar un modelo que excluya el pliegue correspondiente y aprenda toda preparación únicamente con su ajuste.

**Actualización tras el bloque 11:** la preparación sin artista y la referencia mayoritaria ya están ejecutadas. El siguiente paso será XGBoost inicial sin artista. La selección de parámetros y cortes seguirá reservada al nivel interno del protocolo anidado; la validación externa se implementará por separado, sin consultar test.

Referencia: [StratifiedGroupKFold, documentación de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).

**Resultados ejecutados:** los pliegues contienen entre 14,194 y 14,502 canciones de validación, y entre 10.669% y 11.343% de positivas. Se confirmó una validación para cada uno de los 71.728 IDs y un único pliegue para cada una de las 64.965 claves. En las cinco rondas hay cero IDs y cero claves compartidos entre ajuste y validación, cero IDs de test y ambas clases presentes. Estas comprobaciones no evalúan aún un modelo.


## 11. Preparación de características y referencia mayoritaria

### 11.1. Alcance y decisiones

Empezamos por la **variante sin artista**. Se prepara una entrada común para XGBoost inicial y se calcula la referencia mayoritaria en los cinco pliegues guardados. La variante con artista se implementará aparte con exclusión de la propia clave dentro de cada ajuste; no se utiliza aquí una codificación aprendida globalmente ni se descarta ese experimento futuro.

| Entrada | Decisión | Motivo |
|---|---|---|
| Nueve variables de audio y duración en minutos | Imputar faltantes con mediana de ajuste | Permite una base numérica completa sin consultar validación. |
| explicit, mode y time_signature | Texto y One-Hot Encoding aprendido en ajuste | Son categorías, no distancias numéricas que debamos interpretar como cantidades. |
| Géneros Multi-Hot | Conservar indicadores 0/1 ya preparados por canción | Una canción puede pertenecer a varios géneros. |
| Popularidad y clase | Excluir de entradas | Son la respuesta o su origen. |
| ID y clave artista+título | Solo seguimiento y agrupación | No se entregan como predictores. |
| duration_ms | Excluir | Ya se representa duración mediante duration_min. |

**Sin escalamiento en este bloque:** XGBoost de árboles divide variables por umbrales y no necesita estandarización para funcionar. El escalamiento seguirá siendo necesario para las distancias del clustering. Se mantiene imputación explícita para hacer visible su aprendizaje y sus controles, aunque XGBoost también admite faltantes.

La referencia mayoritaria no necesita estas características para decidir: aprende solo cuál es la clase más frecuente en el ajuste. Aquí se ejecuta sobre las matrices preparadas para comprobar simultáneamente el flujo que luego utilizará XGBoost. No se atribuye a la imputación o a los géneros una mejora de la referencia.

In [12]:
import numpy as np
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

variables_audio_u3 = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
variables_numericas_u3 = ["duration_min"] + variables_audio_u3
variables_categoricas_u3 = ["explicit", "mode", "time_signature"]
variables_genero_u3 = sorted(col for col in datos_entrenamiento_u3.columns if col.startswith("genero__"))
columnas_entrada_u3 = variables_numericas_u3 + variables_categoricas_u3 + variables_genero_u3
if len(variables_genero_u3) != 114 or not set(columnas_entrada_u3).issubset(datos_entrenamiento_u3.columns):
    raise ValueError("Las características no coinciden con la base esperada.")
if {"popularity", "popularidad_alta", "clave_cancion", "track_id", "duration_ms"}.intersection(columnas_entrada_u3):
    raise ValueError("Se incluyó una columna excluida.")

X_train_sin_artista_u3 = datos_entrenamiento_u3.loc[:, columnas_entrada_u3].copy()
for columna in variables_categoricas_u3:
    X_train_sin_artista_u3[columna] = X_train_sin_artista_u3[columna].astype("string").fillna("NO_DETERMINADO")
X_train_sin_artista_u3["time_signature"] = X_train_sin_artista_u3["time_signature"].replace({"1": "NO_DETERMINADO"})
if not X_train_sin_artista_u3[variables_genero_u3].isin([0, 1]).all().all():
    raise ValueError("Hay indicadores de género distintos de 0 y 1.")
if np.isinf(X_train_sin_artista_u3[variables_numericas_u3].to_numpy(dtype=float)).any():
    raise ValueError("Hay valores numéricos infinitos.")

preparador_sin_artista_u3 = ColumnTransformer(
    transformers=[
        ("numericas", SimpleImputer(strategy="median"), variables_numericas_u3),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), variables_categoricas_u3),
        ("generos", "passthrough", variables_genero_u3),
    ],
    remainder="drop",
)
print(f"Entradas antes de One-Hot: {X_train_sin_artista_u3.shape[1]} = {len(variables_numericas_u3)} numéricas + {len(variables_categoricas_u3)} categóricas + {len(variables_genero_u3)} géneros")
print("Faltantes numéricos en entrenamiento, antes de imputar:")
print(X_train_sin_artista_u3[variables_numericas_u3].isna().sum().to_string())
print("Preparador definido; todavía no ajustado con todo entrenamiento.")

Entradas antes de One-Hot: 127 = 10 numéricas + 3 categóricas + 114 géneros
Faltantes numéricos en entrenamiento, antes de imputar:
duration_min          0
danceability          0
energy                0
loudness              0
speechiness           0
acousticness          0
instrumentalness      0
liveness              0
valence               0
tempo               136
Preparador definido; todavía no ajustado con todo entrenamiento.


### Explicación de las instrucciones y su salida

1. `numpy` permite comprobar infinitos y matrices. Las importaciones de scikit-learn proporcionan el preparador, la referencia y métricas; no ejecutan ajustes por sí mismas.
2. Las listas declaran explícitamente nueve variables de audio, duración y tres categorías. `sorted(... startswith("genero__"))` selecciona y ordena los indicadores de género para conservar un orden reproducible.
3. Sumar listas concatena nombres de columnas; no suma datos. El control de 114 géneros detecta un cambio inesperado en la base. La intersección con columnas excluidas protege la definición de entradas.
4. `.loc[:, columnas_entrada_u3].copy()` selecciona todas las filas de entrenamiento y solo esas columnas, sin cambiar la base original.
5. El bucle convierte las categorías en texto y reemplaza faltantes por una etiqueta fija. Esta operación no calcula estadísticas ni aprende categorías con validación. El compás `1` se representa como no determinado, conservando la convención de preparación de la Unidad 2; es una decisión de representación, no una afirmación de que todo compás 1 sea universalmente inválido. Si apareciera otra representación incompatible se revisaría, no se adivinaría su equivalencia.
6. `.isin([0, 1]).all().all()` exige que todos los indicadores de género sean binarios y no faltantes. `np.isinf(...).any()` detiene la ejecución si hay infinitos; los faltantes se permiten temporalmente porque después se imputarán.
7. `ColumnTransformer` aplica operaciones a grupos de columnas. `SimpleImputer(strategy="median")` aprenderá una mediana por variable **cuando hagamos fit dentro de la ronda**, no ahora.
8. `OneHotEncoder` aprenderá las categorías presentes solo en ajuste. No elimina una categoría base: todos sus indicadores se conservan. `handle_unknown="ignore"` convierte una categoría nueva de validación en ceros dentro de su bloque, sin aprenderla. No significa que la categoría nueva sea igual a una conocida.
9. `sparse_output=False` produce una matriz densa, donde los ceros de los indicadores son ceros explícitos. Esto evita confundir su representación con entradas ausentes al preparar el posterior XGBoost. La memoria utilizada se mostrará; no se conservan simultáneamente las matrices de las cinco rondas.
10. `passthrough` deja los géneros sin volver a codificarlos; `remainder="drop"` descarta cualquier columna no declarada.
11. Los `print` muestran **127 entradas originales: 10 numéricas, 3 categóricas y 114 géneros**. Después de One-Hot habrá un número distinto de columnas, según las categorías aprendidas. `isna().sum()` cuenta faltantes por variable, no errores del modelo.

Observar cuántos faltantes existen en entrenamiento es diagnóstico. Las medianas no se calcularán con este resumen global; cada ronda aprenderá las suyas exclusivamente en ajuste.

In [13]:
predicciones_referencia_u3 = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64", name="prediccion_mayoritaria")
probabilidades_referencia_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64", name="probabilidad_positiva")
conteo_oof_referencia_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
filas_preparacion_u3 = []
filas_referencia_u3 = []

for ronda, (pos_ajuste, pos_validacion) in enumerate(particiones_cinco_u3, start=1):
    X_ajuste = X_train_sin_artista_u3.iloc[pos_ajuste]
    X_validacion = X_train_sin_artista_u3.iloc[pos_validacion]
    y_ajuste = y_train_u3.iloc[pos_ajuste]
    y_validacion = y_train_u3.iloc[pos_validacion]
    if X_ajuste[variables_numericas_u3].isna().all().any():
        raise ValueError(f"Una variable numérica no tiene datos de ajuste en ronda {ronda}.")
    preparador_ronda = clone(preparador_sin_artista_u3)
    matriz_ajuste = preparador_ronda.fit_transform(X_ajuste)
    matriz_validacion = preparador_ronda.transform(X_validacion)
    if matriz_ajuste.shape[1] != matriz_validacion.shape[1] or not np.isfinite(matriz_ajuste).all() or not np.isfinite(matriz_validacion).all():
        raise ValueError(f"Matrices incompatibles o no finitas en ronda {ronda}.")
    medianas_aprendidas = preparador_ronda.named_transformers_["numericas"].statistics_
    medianas_solo_ajuste = X_ajuste[variables_numericas_u3].median().to_numpy(dtype=float)
    if not np.allclose(medianas_aprendidas, medianas_solo_ajuste):
        raise ValueError("Las medianas no coinciden con las del ajuste permitido.")
    categorias_nuevas = sum(
        int((~X_validacion[columna].isin(categorias)).sum())
        for columna, categorias in zip(variables_categoricas_u3, preparador_ronda.named_transformers_["categoricas"].categories_)
    )
    referencia_ronda = DummyClassifier(strategy="most_frequent")
    referencia_ronda.fit(matriz_ajuste, y_ajuste)
    pred_ronda = referencia_ronda.predict(matriz_validacion)
    columna_positiva = int(np.flatnonzero(referencia_ronda.classes_ == 1)[0])
    proba_ronda = referencia_ronda.predict_proba(matriz_validacion)[:, columna_positiva]
    predicciones_referencia_u3.loc[y_validacion.index] = pred_ronda
    probabilidades_referencia_u3.loc[y_validacion.index] = proba_ronda
    conteo_oof_referencia_u3.loc[y_validacion.index] += 1
    filas_preparacion_u3.append({
        "Ronda": ronda, "Filas fit preparación": len(X_ajuste),
        "Columnas transformadas": matriz_ajuste.shape[1],
        "Faltantes numéricos ajuste": int(X_ajuste[variables_numericas_u3].isna().sum().sum()),
        "Faltantes numéricos validación": int(X_validacion[variables_numericas_u3].isna().sum().sum()),
        "Mediana tempo ajuste": float(medianas_aprendidas[variables_numericas_u3.index("tempo")]),
        "Celdas categóricas nuevas": categorias_nuevas,
        "Medianas verificadas": True, "Matrices finitas": True,
        "MB matrices": (matriz_ajuste.nbytes + matriz_validacion.nbytes) / 1_000_000,
    })
    tn, fp, fn, tp = confusion_matrix(y_validacion, pred_ronda, labels=[0, 1]).ravel()
    filas_referencia_u3.append({
        "Ronda": ronda, "Clase predicha": int(referencia_ronda.classes_[np.argmax(referencia_ronda.class_prior_)]),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Accuracy": accuracy_score(y_validacion, pred_ronda),
        "Recall": recall_score(y_validacion, pred_ronda, zero_division=0),
        "F1": f1_score(y_validacion, pred_ronda, zero_division=0),
    })
    del matriz_ajuste, matriz_validacion

if predicciones_referencia_u3.isna().any() or probabilidades_referencia_u3.isna().any() or not conteo_oof_referencia_u3.eq(1).all():
    raise ValueError("Predicciones OOF incompletas o repetidas.")
revision_preparacion_u3 = pd.DataFrame(filas_preparacion_u3).set_index("Ronda")
resultados_rondas_referencia_u3 = pd.DataFrame(filas_referencia_u3).set_index("Ronda")
display(revision_preparacion_u3.round(3))
display(resultados_rondas_referencia_u3.round(4))

,Filas fit preparación,Columnas transformadas,Faltantes numéricos ajuste,Faltantes numéricos validación,Mediana tempo ajuste,Celdas categóricas nuevas,Medianas verificadas,Matrices finitas,MB matrices
Ronda,,,,,,,,,
1,57340,132,111,25,122.038,0,True,True,75.745
2,57348,132,107,29,122.020,0,True,True,75.745
3,57226,132,103,33,122.012,0,True,True,75.745
4,57464,132,111,25,122.015,0,True,True,75.745
5,57534,132,112,24,122.024,0,True,True,75.745


,Clase predicha,TN,FP,FN,TP,Accuracy,Recall,F1
Ronda,,,,,,,,
1,0,12853,0,1535,0,0.8933,0.0,0.0
2,0,12819,0,1561,0,0.8914,0.0,0.0
3,0,12903,0,1599,0,0.8897,0.0,0.0
4,0,12646,0,1618,0,0.8866,0.0,0.0
5,0,12646,0,1548,0,0.8909,0.0,0.0


### Explicación paso a paso del flujo por ronda

1. Se reservan tres series por ID: decisiones aún faltantes, probabilidades aún faltantes y contador OOF en cero. Almacenar por ID evita confundir el orden de las predicciones.
2. El bucle reutiliza **exactamente** `particiones_cinco_u3`. `.iloc` selecciona entradas y respuestas en las posiciones ya comprobadas. No se vuelve a sortear la separación.
3. `isna().all().any()` detecta una variable numérica completamente ausente en ajuste. Sin datos no existe mediana; se detiene el flujo en lugar de cambiar silenciosamente el número de variables.
4. `clone` crea un preparador nuevo sin aprendizaje previo. `fit_transform(X_ajuste)` aprende medianas y categorías con ajuste y transforma esas mismas filas. `transform(X_validacion)` reutiliza lo aprendido sin reajustarlo.
5. Se exige igual número de columnas en ajuste y validación de esa ronda y valores finitos. Un valor finito no es infinito ni NaN. Las distintas rondas podrían aprender cantidades diferentes de categorías; dentro de cada par las columnas deben corresponder.
6. `named_transformers_["numericas"].statistics_` recupera las medianas utilizadas. Se comparan con las medianas calculadas solo sobre `X_ajuste` mediante `np.allclose`. Esta comprobación y el flujo fit/transform aportan evidencia de origen; una coincidencia numérica aislada no probaría ausencia de fuga.
7. `zip` une cada columna categórica con las categorías aprendidas. `~isin(...)` identifica celdas de validación con categorías nuevas y se suman. El resultado cuenta celdas, no necesariamente canciones distintas.
8. `DummyClassifier(strategy="most_frequent")` aprende la clase más frecuente de `y_ajuste` y la repite para todas las filas. `fit` no aprende relaciones entre audio y popularidad.
9. `predict` devuelve decisiones y `predict_proba` las puntuaciones de la referencia. Se localiza explícitamente la columna de la clase 1 con `classes_`, sin suponer su posición.
10. Se guardan las predicciones en los IDs reservados y se incrementa el contador. En esta referencia la probabilidad de clase 1 será cero porque `most_frequent` asigna toda su masa a la clase mayoritaria; **no es la proporción real de positivas ni una probabilidad calibrada por canción**.
11. Las listas registran filas usadas por fit, columnas transformadas, faltantes previos, mediana de tempo, categorías nuevas y memoria. Un MB corresponde aquí a un millón de bytes y solo cuenta ambas matrices, no todo el consumo del proceso.
12. `confusion_matrix(..., labels=[0, 1]).ravel()` devuelve TN, FP, FN, TP en ese orden. Las métricas por ronda se calculan sobre las filas que quedaron fuera.
13. `del` libera las referencias a ambas matrices antes de la ronda siguiente; no elimina archivos. Los objetos `preparador_ronda` y `referencia_ronda` que quedan después del bucle pertenecen **solo a la última ronda**, no son modelos finales ajustados con todo entrenamiento.
14. Al salir se exige ausencia de predicciones faltantes y un conteo OOF igual a uno para cada ID. Las dos tablas reúnen controles y resultados.

**Qué esperamos observar:** medianas de tempo posiblemente distintas por ronda, matrices completas tras imputación y clase predicha 0 en todas las rondas. Los faltantes de validación se rellenan con la mediana de ajuste de su ronda. Una accuracy cercana al 89% acompañada de recall y F1 cero indica que la referencia acierta negativas pero omite todas las positivas.

**Resultados de preparación ejecutados:** los 136 faltantes corresponden a tempo. En cada ronda se reparten entre ajuste y validación y se imputan con su mediana permitida, de 122,012 a 122,038 BPM según la ronda. Las 127 entradas se convierten en 132 columnas: 10 numéricas, 8 indicadores One-Hot y 114 géneros. No aparecieron celdas categóricas nuevas; las dos matrices ocuparon aproximadamente 75,745 MB por ronda y todos sus valores resultaron finitos. Esto confirma funcionamiento de la preparación, no calidad predictiva.


In [14]:
pred_oof_referencia_u3 = predicciones_referencia_u3.astype("int64")
tn_ref, fp_ref, fn_ref, tp_ref = confusion_matrix(y_train_u3, pred_oof_referencia_u3, labels=[0, 1]).ravel()
precision_referencia_u3 = tp_ref / (tp_ref + fp_ref) if tp_ref + fp_ref > 0 else np.nan
recall_referencia_u3 = recall_score(y_train_u3, pred_oof_referencia_u3, zero_division=0)
f1_referencia_u3 = f1_score(y_train_u3, pred_oof_referencia_u3, zero_division=0)
accuracy_referencia_u3 = accuracy_score(y_train_u3, pred_oof_referencia_u3)
auc_referencia_u3 = roc_auc_score(y_train_u3, probabilidades_referencia_u3)
ap_referencia_u3 = average_precision_score(y_train_u3, probabilidades_referencia_u3)

matriz_referencia_u3 = pd.DataFrame(
    [[tn_ref, fp_ref], [fn_ref, tp_ref]],
    index=["Real 0: popularidad <60", "Real 1: popularidad >=60"],
    columns=["Predice 0", "Predice 1"],
)
metricas_referencia_u3 = pd.DataFrame({
    "Métrica": ["Accuracy", "Precisión positiva", "Recall positivo", "F1 positivo", "ROC-AUC", "Average Precision"],
    "Valor OOF": [accuracy_referencia_u3, precision_referencia_u3, recall_referencia_u3, f1_referencia_u3, auc_referencia_u3, ap_referencia_u3],
    "Interpretación": ["Fracción total de aciertos", "No definida: no predice positivos", "Ninguna positiva detectada", "No consigue equilibrio positivo", "Sin capacidad de ordenar clases", "Igual a la proporción positiva con puntuación constante"],
})
cumple_precision_ref = bool(pd.notna(precision_referencia_u3) and precision_referencia_u3 >= META_PRECISION)
cumple_recall_ref = bool(recall_referencia_u3 >= META_RECALL)
kpi_referencia_u3 = pd.DataFrame([
    {"KPI": "Precisión >=30%", "Resultado": "No definida", "Estado": "Cumple" if cumple_precision_ref else "No cumple"},
    {"KPI": "Recall >=60%", "Resultado": f"{recall_referencia_u3:.2%}", "Estado": "Cumple" if cumple_recall_ref else "No cumple"},
    {"KPI": "Ambas metas simultáneas", "Resultado": "Referencia mayoritaria", "Estado": "Cumple" if cumple_precision_ref and cumple_recall_ref else "No cumple"},
])
display(matriz_referencia_u3)
display(metricas_referencia_u3.round({"Valor OOF": 6}))
display(kpi_referencia_u3)
print(f"Positivas detectadas: {int(tp_ref):,} de {int(tp_ref + fn_ref):,}")
print(f"Accuracy: {accuracy_referencia_u3:.2%}; proporción real de positivas: {y_train_u3.mean():.2%}")

,Predice 0,Predice 1
Real 0: popularidad <60,63867,0
Real 1: popularidad >=60,7861,0


,Métrica,Valor OOF,Interpretación
0,Accuracy,0.890405,Fracción total de aciertos
1,Precisión positiva,NaN,No definida: no predice positivos
2,Recall positivo,0.000000,Ninguna positiva detectada
3,F1 positivo,0.000000,No consigue equilibrio positivo
4,ROC-AUC,0.500000,Sin capacidad de ordenar clases
5,Average Precision,0.109595,Igual a la proporción positiva con puntuación constante


,KPI,Resultado,Estado
0,Precisión >=30%,No definida,No cumple
1,Recall >=60%,0.00%,No cumple
2,Ambas metas simultáneas,Referencia mayoritaria,No cumple


Positivas detectadas: 0 de 7,861
Accuracy: 89.04%; proporción real de positivas: 10.96%


### Explicación de las métricas OOF y los KPI

1. Convertir las decisiones a `int64` permite evaluarlas como enteros 0/1 después de comprobar que no faltan. La matriz global reúne una predicción por cada ID de entrenamiento: ninguna procede de un modelo que haya ajustado con ese ID en su ronda.
2. La precisión se calcula como TP/(TP+FP). Cuando TP+FP es cero, usamos `np.nan` para señalar **no definida**, en vez de ocultarlo como una precisión real de cero. El KPI no se considera cumplido.
3. `recall_score` calcula TP/(TP+FN); aquí su denominador sí contiene positivas reales. `f1_score` calcula 2TP/(2TP+FP+FN); aquí resulta cero. `zero_division=0` especifica la convención para denominadores nulos, aunque no cambia esos dos resultados en este caso.
4. `accuracy_score` cuenta aciertos sobre todas las canciones. No significa que detectemos positivas.
5. ROC-AUC y Average Precision utilizan puntuaciones, no solo decisiones. Como todas las puntuaciones son iguales, ROC-AUC es 0,5: la referencia no distingue qué canción debería ordenarse más arriba. Average Precision coincide con la prevalencia positiva: no se interpreta como porcentaje de canciones correctamente clasificadas.
6. Las tablas incluyen nombres, valores y explicaciones. `round` cambia solo lo mostrado. En la matriz, filas son clases reales y columnas son predicciones.
7. `pd.notna` exige que la precisión esté definida antes de compararla con 0,30. `and` exige ambas metas simultáneamente. Estas condiciones producen el estado «No cumple» sin modificar metas.
8. Los `print` muestran positivas detectadas y accuracy en porcentajes. En la tabla de métricas las proporciones se muestran entre cero y uno: 0,890405 equivale aproximadamente a 89,04%.

**Por qué no cumple:** la estrategia elige siempre la clase más frecuente y no intenta detectar la minoritaria. La accuracy alta proviene del predominio de negativas, no de una capacidad útil para el objetivo. Esta referencia establece un punto de comparación; no es el modelo que queremos recomendar.

### 11.2. Cierre y continuidad

Se prepararon las características **sin artista** dentro de las mismas cinco rondas y se obtuvo una predicción OOF de la referencia por cada canción. No se reajustó la preparación con todo entrenamiento para luego validar, ni se consultó test.

No se deben reutilizar los cinco preparadores como un único preparador global. Para XGBoost se construirá un flujo con la misma definición de columnas y preparación, aprendiendo de nuevo dentro de cada ajuste autorizado. La variante con artista y los niveles internos de evaluación anidada tendrán sus propios ajustes completos.

**Actualización tras el bloque 12:** XGBoost inicial sin artista ya se ejecutó en estos mismos pliegues y se comparó con la referencia. Un primer análisis al corte 0,50 será descriptivo; cualquier selección de umbral e hiperparámetros se realizará después dentro del protocolo previsto. No se ha ejecutado GridSearch ni RandomizedSearch.

**Resultados ejecutados:** 63,867 negativas correctamente descartadas; 7,861 positivas omitidas; cero verdaderos positivos y cero falsos positivos. Accuracy 89.0405%, recall 0%, F1 0, ROC-AUC 0,5 y Average Precision 0.109595. Precisión positiva no definida. No cumple ninguno de los dos KPI ni su condición simultánea.

Se transformaron 132 columnas en cada ronda y se verificaron medianas de ajuste y matrices finitas. La tabla muestra memoria solo de las matrices. Estos resultados corresponden a la referencia y al flujo sin artista; no son resultados de XGBoost.


## 12. XGBoost inicial sin artista

### 12.1. Qué comprobamos antes de optimizar

Se entrena una configuración fija de `XGBClassifier` con las características sin artista del bloque 11 y las mismas cinco particiones agrupadas del bloque 10. Esta es una comparación inicial, no una búsqueda de la mejor configuración.

Se fija **umbral 0,50** antes de observar las predicciones. Popularidad ≥60 sigue definiendo la clase real; 0,50 es el corte aplicado a la puntuación del modelo. No son la misma escala. No se interpreta la puntuación como una probabilidad calibrada: todavía no se evaluó calibración.

| Configuración inicial | Valor | Motivo |
|---|---:|---|
| n_estimators | 100 | Cantidad acotada de árboles, dentro de la guía. |
| max_depth | 3 | Limitar inicialmente la complejidad. |
| learning_rate | 0,10 | Aporte moderado de cada árbol; valor de la guía. |
| subsample | 1,0 | Usar todas las filas permitidas por árbol en este punto de partida. |
| colsample_bytree | 1,0 | Considerar todas las columnas por árbol. |
| min_child_weight | 1 | Mínima suma de hessianos en una hoja; no equivale universalmente a una canción. |
| reg_lambda / reg_alpha | 1 / 0 | Regularización L2 inicial y sin penalización L1 adicional. |
| gamma | 0 | Sin ganancia mínima adicional para dividir, aparte de las demás restricciones. |
| scale_pos_weight | 1 | Sin ponderación adicional de la clase positiva. |
| tree_method | hist | Construcción por histogramas para eficiencia. |

No se usará la validación de estas rondas como `eval_set` para detener o elegir árboles. No hay early stopping: se ajustan los 100 árboles fijados. La optimización posterior podrá cambiar la configuración, pero sus métricas se evaluarán según el protocolo anidado previsto.

**Dependencias:** esta ejecución utiliza XGBoost 2.1.4 y scikit-learn 1.6.1. Si faltan, pueden instalarse en una celda auxiliar con `%pip install xgboost==2.1.4 scikit-learn==1.6.1`; después puede ser necesario reiniciar el kernel y ejecutar en orden. En macOS XGBoost puede requerir además un runtime OpenMP. No se cambian paquetes desde el código de entrenamiento.

In [15]:
import xgboost
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from time import perf_counter

UMBRAL_XGB_INICIAL_U3 = 0.50
modelo_xgb_inicial_u3 = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    max_depth=3,
    learning_rate=0.10,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    scale_pos_weight=1.0,
    tree_method="hist",
    random_state=SEMILLA,
    n_jobs=1,
)
flujo_xgb_inicial_u3 = Pipeline([
    ("preparacion", clone(preparador_sin_artista_u3)),
    ("modelo", modelo_xgb_inicial_u3),
])
print(f"Versiones: XGBoost {xgboost.__version__}; scikit-learn {sklearn.__version__}")
print(f"Configuración inicial fija: 100 árboles, profundidad 3, learning_rate 0.10; umbral {UMBRAL_XGB_INICIAL_U3:.2f}")
print("Variante sin artista y sin ponderación adicional de clases; aún no ajustada.")

Versiones: XGBoost 2.1.4; scikit-learn 1.6.1
Configuración inicial fija: 100 árboles, profundidad 3, learning_rate 0.10; umbral 0.50
Variante sin artista y sin ponderación adicional de clases; aún no ajustada.


### Explicación de la configuración

1. `import xgboost` permite registrar la versión y `XGBClassifier` importa el clasificador. `Pipeline` encadena preparación y modelo; `perf_counter` permite medir tiempo transcurrido.
2. `UMBRAL_XGB_INICIAL_U3 = 0.50` fija el corte antes de evaluar; no selecciona automáticamente el más conveniente.
3. `objective="binary:logistic"` establece el aprendizaje de clasificación binaria con salida de puntuaciones entre cero y uno. No convierte este modelo en la regresión logística lineal de la Unidad 2: aquí se usa boosting de árboles.
4. `eval_metric="logloss"` declara la métrica de evaluación interna de XGBoost. La función objetivo dirige el aprendizaje; precisión, recall, F1, AUC y Average Precision se calculan aparte. Sin `eval_set` esta opción no selecciona árboles ni configura nuestra búsqueda futura.
5. Los siguientes argumentos fijan los valores explicados en la tabla: árboles, profundidad, tasa, fracciones, restricciones y regularización. Se hacen explícitos para que la configuración sea revisable y no dependa solo de valores predeterminados.
6. `scale_pos_weight=1.0` mantiene el peso relativo positivo sin incremento. No se aplican SMOTE, sobremuestreo ni submuestreo.
7. `tree_method="hist"` utiliza histogramas. `random_state=SEMILLA` controla aleatoriedad reproducible; no se prueban semillas para buscar mejores resultados. `n_jobs=1` limita el entrenamiento a un hilo configurado para controlar recursos; no significa un solo árbol.
8. `Pipeline` crea dos pasos nombrados. El preparador se clona sin aprendizaje previo. Después clonaremos **el flujo completo en cada ronda**, evitando reutilizar medianas, categorías o árboles ajustados en otra.
9. Los `print` muestran decisiones y versiones. Aún no son resultados predictivos.

In [16]:
probabilidades_xgb_inicial_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64", name="probabilidad_xgb_inicial")
predicciones_xgb_inicial_u3 = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64", name="prediccion_xgb_inicial")
veces_oof_xgb_inicial_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
filas_rondas_xgb_inicial_u3 = []

for ronda, (pos_ajuste, pos_validacion) in enumerate(particiones_cinco_u3, start=1):
    X_ajuste = X_train_sin_artista_u3.iloc[pos_ajuste]
    X_validacion = X_train_sin_artista_u3.iloc[pos_validacion]
    y_ajuste = y_train_u3.iloc[pos_ajuste]
    y_validacion = y_train_u3.iloc[pos_validacion]
    if set(claves_train_u3.iloc[pos_ajuste]).intersection(set(claves_train_u3.iloc[pos_validacion])):
        raise ValueError("Hay versiones compartidas entre ajuste y validación.")
    if len(X_ajuste.index.union(X_validacion.index).intersection(ids_test_u2)) != 0:
        raise ValueError("Se detectaron IDs de test.")
    if X_ajuste[variables_numericas_u3].isna().all().any():
        raise ValueError("Hay una variable numérica sin observaciones en ajuste.")
    flujo_ronda_xgb = clone(flujo_xgb_inicial_u3)
    inicio_ronda = perf_counter()
    flujo_ronda_xgb.fit(X_ajuste, y_ajuste)
    segundos_fit = perf_counter() - inicio_ronda
    preparacion_ajustada = flujo_ronda_xgb.named_steps["preparacion"]
    if not np.allclose(preparacion_ajustada.named_transformers_["numericas"].statistics_, X_ajuste[variables_numericas_u3].median().to_numpy(dtype=float)):
        raise ValueError("Las medianas no corresponden al ajuste permitido.")
    columna_positiva = int(np.flatnonzero(flujo_ronda_xgb.named_steps["modelo"].classes_ == 1)[0])
    proba_ronda = flujo_ronda_xgb.predict_proba(X_validacion)[:, columna_positiva]
    if not np.isfinite(proba_ronda).all() or not ((proba_ronda >= 0) & (proba_ronda <= 1)).all():
        raise ValueError("Puntuaciones fuera de rango o no finitas.")
    pred_ronda = (proba_ronda >= UMBRAL_XGB_INICIAL_U3).astype("int8")
    probabilidades_xgb_inicial_u3.loc[y_validacion.index] = proba_ronda
    predicciones_xgb_inicial_u3.loc[y_validacion.index] = pred_ronda
    veces_oof_xgb_inicial_u3.loc[y_validacion.index] += 1
    tn, fp, fn, tp = confusion_matrix(y_validacion, pred_ronda, labels=[0, 1]).ravel()
    precision_ronda = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall_ronda = recall_score(y_validacion, pred_ronda, zero_division=0)
    filas_rondas_xgb_inicial_u3.append({
        "Ronda": ronda, "IDs ajuste": len(X_ajuste), "IDs validación": len(X_validacion),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Precisión": precision_ronda, "Recall": recall_ronda,
        "F1": f1_score(y_validacion, pred_ronda, zero_division=0),
        "Accuracy": accuracy_score(y_validacion, pred_ronda),
        "ROC-AUC": roc_auc_score(y_validacion, proba_ronda),
        "Average Precision": average_precision_score(y_validacion, proba_ronda),
        "Cumple ambos KPI": bool(pd.notna(precision_ronda) and precision_ronda >= META_PRECISION and recall_ronda >= META_RECALL),
        "Segundos fit": segundos_fit,
    })
    del flujo_ronda_xgb, preparacion_ajustada

if not veces_oof_xgb_inicial_u3.eq(1).all() or probabilidades_xgb_inicial_u3.isna().any() or predicciones_xgb_inicial_u3.isna().any():
    raise ValueError("Las predicciones OOF no tienen cobertura única completa.")
resultados_rondas_xgb_inicial_u3 = pd.DataFrame(filas_rondas_xgb_inicial_u3).set_index("Ronda")
display(resultados_rondas_xgb_inicial_u3.round(4))
print(f"IDs con una predicción OOF: {veces_oof_xgb_inicial_u3.eq(1).sum():,}")
print("Cinco ajustes completos de preparación y XGBoost; ninguna evaluación de test.")

,IDs ajuste,IDs validación,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI,Segundos fit
Ronda,,,,,,,,,,,,,,
1,57340,14388,12811,42,1383,152,0.7835,0.0990,0.1758,0.9010,0.8308,0.4281,False,0.8569
2,57348,14380,12763,56,1431,130,0.6989,0.0833,0.1488,0.8966,0.8443,0.4254,False,0.7784
3,57226,14502,12864,39,1460,139,0.7809,0.0869,0.1564,0.8966,0.8459,0.4510,False,0.7604
4,57464,14264,12622,24,1490,128,0.8421,0.0791,0.1446,0.8939,0.8408,0.4632,False,0.7685
5,57534,14194,12550,96,1381,167,0.6350,0.1079,0.1844,0.8959,0.8344,0.4183,False,0.7961


IDs con una predicción OOF: 71,728
Cinco ajustes completos de preparación y XGBoost; ninguna evaluación de test.


### Explicación del entrenamiento y las predicciones OOF

1. Se crean series de puntuaciones, decisiones y conteos con el mismo índice de ID que la respuesta. Las dos primeras comienzan faltantes para detectar omisiones.
2. El bucle recorre los mismos cinco pares de posiciones. `.iloc` selecciona entradas y respuestas sin reordenarlas. Se vuelven a comprobar claves compartidas, presencia de test y variables totalmente faltantes en ajuste.
3. `clone(flujo_xgb_inicial_u3)` crea preparación y modelo nuevos. `fit(X_ajuste, y_ajuste)` aprende medianas y categorías solo de esas entradas y después los árboles con sus respuestas. Validación no participa en fit, no se pasa como `eval_set` y no se utiliza para early stopping.
4. `perf_counter` antes y después de fit mide segundos de preparación y entrenamiento juntos; no incluye predicción, métricas ni todo el tiempo del bloque. Depende del equipo, por lo que no es una medida universal del algoritmo.
5. `named_steps` recupera la preparación aprendida. Comparar sus medianas con `X_ajuste` aporta un control adicional; el flujo de datos y la separación siguen siendo la evidencia principal.
6. `classes_` identifica la columna positiva. `predict_proba(X_validacion)` transforma con la preparación ya ajustada y obtiene puntuaciones sin recibir las respuestas. Se exigen valores finitos entre cero y uno.
7. `>=0.50` convierte puntuaciones en decisiones 0/1. No se mueve este corte según el resultado de la ronda.
8. `.loc[y_validacion.index]` guarda cada resultado en su ID; el contador aumenta una vez. Cada canción recibe una predicción de un flujo que no ajustó con ella ni con su clave.
9. La matriz de confusión devuelve TN, FP, FN y TP. La precisión se deja no definida si no hay positivos predichos; recall y F1 usan sus fórmulas positivas. AUC y Average Precision utilizan las puntuaciones continuas.
10. La condición de KPI exige precisión definida y ambas metas simultáneas. El resultado por ronda es descriptivo; el resumen global se calcula después reuniendo las predicciones.
11. `del` elimina los flujos ajustados de cada ronda tras guardar sus resultados, evitando presentar el último como un modelo final de todo entrenamiento. La plantilla original sigue sin ajustar.
12. El control final exige una sola OOF por ID y ningún faltante. `DataFrame` reúne la tabla y `round(4)` limita decimales mostrados; no modifica métricas almacenadas.

**Lectura de una fila:** TP son populares detectadas y FN son populares omitidas en esa validación; FP son falsas alarmas. Alta precisión junto con recall bajo significa que pocas canciones se señalan y muchas positivas quedan fuera. Los porcentajes se obtienen multiplicando las proporciones por cien. Cambios entre rondas describen variación observada, no un intervalo de confianza ni una prueba estadística.

In [17]:
pred_xgb_oof_u3 = predicciones_xgb_inicial_u3.astype("int64")
tn_xgb, fp_xgb, fn_xgb, tp_xgb = confusion_matrix(y_train_u3, pred_xgb_oof_u3, labels=[0, 1]).ravel()
precision_xgb_inicial_u3 = tp_xgb / (tp_xgb + fp_xgb) if tp_xgb + fp_xgb > 0 else np.nan
recall_xgb_inicial_u3 = recall_score(y_train_u3, pred_xgb_oof_u3, zero_division=0)
f1_xgb_inicial_u3 = f1_score(y_train_u3, pred_xgb_oof_u3, zero_division=0)
accuracy_xgb_inicial_u3 = accuracy_score(y_train_u3, pred_xgb_oof_u3)
auc_xgb_inicial_u3 = roc_auc_score(y_train_u3, probabilidades_xgb_inicial_u3)
ap_xgb_inicial_u3 = average_precision_score(y_train_u3, probabilidades_xgb_inicial_u3)

matriz_xgb_inicial_u3 = pd.DataFrame(
    [[tn_xgb, fp_xgb], [fn_xgb, tp_xgb]],
    index=["Real 0: popularidad <60", "Real 1: popularidad >=60"],
    columns=["Predice 0", "Predice 1"],
)
comparacion_xgb_referencia_u3 = pd.DataFrame([
    {"Modelo": "Mayoritaria", "Umbral": "No aplica: clase constante", "Precisión": precision_referencia_u3,
     "Recall": recall_referencia_u3, "F1": f1_referencia_u3, "Accuracy": accuracy_referencia_u3,
     "ROC-AUC": auc_referencia_u3, "Average Precision": ap_referencia_u3},
    {"Modelo": "XGBoost inicial sin artista", "Umbral": "0.50", "Precisión": precision_xgb_inicial_u3,
     "Recall": recall_xgb_inicial_u3, "F1": f1_xgb_inicial_u3, "Accuracy": accuracy_xgb_inicial_u3,
     "ROC-AUC": auc_xgb_inicial_u3, "Average Precision": ap_xgb_inicial_u3},
]).set_index("Modelo")
cumple_precision_xgb_u3 = bool(pd.notna(precision_xgb_inicial_u3) and precision_xgb_inicial_u3 >= META_PRECISION)
cumple_recall_xgb_u3 = bool(recall_xgb_inicial_u3 >= META_RECALL)
kpi_xgb_inicial_u3 = pd.DataFrame([
    {"KPI": "Precisión >=30%", "Resultado": precision_xgb_inicial_u3, "Estado": "Cumple" if cumple_precision_xgb_u3 else "No cumple"},
    {"KPI": "Recall >=60%", "Resultado": recall_xgb_inicial_u3, "Estado": "Cumple" if cumple_recall_xgb_u3 else "No cumple"},
    {"KPI": "Ambas metas simultáneas", "Resultado": np.nan, "Estado": "Cumple" if cumple_precision_xgb_u3 and cumple_recall_xgb_u3 else "No cumple"},
])
display(matriz_xgb_inicial_u3)
display(comparacion_xgb_referencia_u3.round(6))
display(kpi_xgb_inicial_u3.round({"Resultado": 6}))
print(f"Detecta {int(tp_xgb):,} de {int(tp_xgb + fn_xgb):,} positivas; omite {int(fn_xgb):,}.")
print(f"Señala {int(tp_xgb + fp_xgb):,} canciones, con {int(fp_xgb):,} falsas alarmas.")
print(f"Precisión {precision_xgb_inicial_u3:.2%}; recall {recall_xgb_inicial_u3:.2%}; F1 {f1_xgb_inicial_u3:.4f}.")
print(f"Diferencia de accuracy frente a referencia: {(accuracy_xgb_inicial_u3 - accuracy_referencia_u3) * 100:+.3f} puntos porcentuales.")

,Predice 0,Predice 1
Real 0: popularidad <60,63610,257
Real 1: popularidad >=60,7145,716


,Umbral,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision
Modelo,,,,,,,
Mayoritaria,No aplica: clase constante,NaN,0.000000,0.000000,0.890405,0.500000,0.109595
XGBoost inicial sin artista,0.50,0.735868,0.091083,0.162101,0.896805,0.839099,0.435580


,KPI,Resultado,Estado
0,Precisión >=30%,0.735868,Cumple
1,Recall >=60%,0.091083,No cumple
2,Ambas metas simultáneas,NaN,No cumple


Detecta 716 de 7,861 positivas; omite 7,145.
Señala 973 canciones, con 257 falsas alarmas.
Precisión 73.59%; recall 9.11%; F1 0.1621.
Diferencia de accuracy frente a referencia: +0.640 puntos porcentuales.


### Explicación de la comparación y los KPI

1. Las decisiones completas se convierten a enteros y se comparan con las respuestas alineadas por ID. La matriz reúne todas las validaciones sin duplicar canciones.
2. Se calculan precisión TP/(TP+FP), recall TP/(TP+FN), F1, accuracy, ROC-AUC y Average Precision sobre las predicciones OOF globales. No son necesariamente el promedio de las cinco métricas por ronda.
3. `pd.DataFrame` muestra la matriz con clases reales en filas y decisiones en columnas. Los cuatro conteos deben sumar 71.728.
4. La comparación utiliza la referencia OOF anterior, sobre exactamente las mismas canciones y pliegues. No se reutilizan resultados de tres pliegues o de test de la Unidad 2.
5. La precisión de la referencia aparece como `NaN`, es decir, no definida por no señalar positivos. No se presenta como un valor numérico válido igual a cero. En la fila «Ambas metas simultáneas», el resultado numérico no aplica; su interpretación está en la columna Estado.
6. Las condiciones comparan precisión y recall con las metas fijadas. Una AUC alta o una mejora de accuracy no reemplaza el cumplimiento de ambas metas.
7. Las salidas finales traducen proporciones a porcentajes. La diferencia de accuracy se expresa en puntos porcentuales; no es una mejora relativa ni una prueba de significación.

**Cómo interpretar las métricas:** ROC-AUC mayor que 0,5 indica capacidad de ordenar positivas por encima de negativas en estas predicciones; no significa ese porcentaje de aciertos. Average Precision debe leerse junto a la prevalencia positiva de aproximadamente 0,1096 y resume el ordenamiento en precisión–recall, sin ser el porcentaje de canciones acertadas. Precisión y recall dependen del corte 0,50; AUC y Average Precision no se calculan usando solo ese corte.

No atribuimos incumplimientos a una única causa comprobada. Un recall bajo al corte 0,50 es compatible con un corte exigente y con el problema desbalanceado; también intervienen configuración y señales disponibles. Estudiar umbrales o ponderaciones requerirá experimentos internos separados, no modificar ahora el resultado para hacerlo aprobar.

### 12.2. Alcance de la evidencia y continuidad

Este bloque evalúa una configuración y un corte fijados previamente, usando solo entrenamiento. No utiliza test, artista, balanceo, búsqueda de hiperparámetros ni selección de umbral. Las predicciones OOF son válidas para este procedimiento inicial bajo la separación por clave; no representan una evaluación anidada del proceso de optimización futuro ni una prueba externa nueva.

Quedan disponibles `probabilidades_xgb_inicial_u3`, `predicciones_xgb_inicial_u3`, `resultados_rondas_xgb_inicial_u3`, `matriz_xgb_inicial_u3`, `comparacion_xgb_referencia_u3` y `kpi_xgb_inicial_u3`. No hay un modelo final ajustado sobre todo entrenamiento.

**Actualización tras el bloque 13:** ambas búsquedas sin artista se ejecutaron con validación anidada; sus resultados y costos se documentan a continuación. La elección de configuraciones y cortes se hará en validación interna; las tres rondas externas evaluarán el procedimiento completo. No se elegirá un umbral aprovechando sus resultados externos.

Fuentes: [interfaz Python de XGBoost](https://xgboost.readthedocs.io/en/release_2.1.0/python/python_api.html) y [parámetros de XGBoost](https://xgboost.readthedocs.io/en/release_2.1.0/parameter.html).

**Resultados ejecutados al corte 0,50:** 716 positivas detectadas (TP), 7,145 omitidas (FN), 257 falsas alarmas (FP) y 63,610 negativas correctamente descartadas (TN). Precisión 73.59%, recall 9.11%, F1 0.1621, accuracy 89.68%, ROC-AUC 0.8391 y Average Precision 0.4356.

La precisión cumple el mínimo de 30%; el recall no cumple el mínimo de 60%. Por tanto, el cumplimiento simultáneo es **No cumple**. De cada cien canciones señaladas, aproximadamente 73.6 son positivas reales; de cada cien positivas reales se detectan aproximadamente 9.1. Estos valores describen este corte y configuración, no un límite definitivo de XGBoost.


## 13. GridSearchCV y RandomizedSearchCV con validación anidada

### 13.1. Qué se elige y qué se evalúa

En el bloque 12 evaluamos una configuración fija. Ahora elegiremos configuraciones, por lo que necesitamos separar **datos para decidir** y **datos para evaluar lo decidido**. Todo sigue dentro de los 71.728 IDs de entrenamiento; el test previo no interviene.

1. Crear tres pliegues externos agrupados. Reservar uno para evaluación y usar los otros dos para desarrollo.
2. Dentro de desarrollo, crear cinco pliegues internos nuevos: no se toman los cinco globales del bloque 10.
3. Comparar GridSearchCV (16 configuraciones) y RandomizedSearchCV (10 del mismo espacio), usando idénticos pliegues internos y Average Precision media para seleccionar parámetros. Se informa también ROC-AUC.
4. Para cada ganador generar puntuaciones OOF internas y elegir su corte sin respuestas externas. Se conservan ambas estrategias para compararlas; adicionalmente se registra la estrategia elegida por AP interna, antes de ver resultados externos.
5. Aplicar los ganadores reajustados con todo desarrollo al pliegue externo. Evaluar tanto corte fijo 0,50 como corte elegido internamente.
6. Reunir las predicciones externas, una por ID, y revisar métricas y KPI.

**Por qué dos cortes:** el fijo permite observar el cambio de hiperparámetros; el interno evalúa el procedimiento completo de selección de parámetros más umbral. No se atribuye toda mejora de recall exclusivamente a los hiperparámetros si también cambió el corte.

La regla de corte se fija ahora, antes de ejecutar: candidatos 0,00; 0,05; 0,10; 0,15; 0,20; 0,25; 0,30; 0,35; 0,40; 0,45; 0,50; 0,60; 0,70; 0,80; 0,90; 1,00. Entre los que cumplan precisión ≥0,30 y recall ≥0,60, mayor F1; empate: mayor precisión y después mayor corte. Si ninguno cumple, se usa 0,50 y se registra que no hubo corte viable interno.

Las OOF internas para el corte usan parámetros elegidos con esos mismos datos de desarrollo: son evidencia de desarrollo, no una estimación independiente. Las respuestas externas permanecen excluidas de toda esa selección. No hay early stopping, ponderación, artista ni pruebas de semillas alternativas.

**Comparación justa:** Random toma un subconjunto de la malla de Grid; con el mismo modelo determinista y particiones, el mejor AP interno de Grid no debería ser menor que el de Random, salvo diferencias numéricas. Eso no garantiza que Grid tenga mejor resultado externo. Comparamos presupuestos de 16 frente a 10 configuraciones, no búsquedas con igual costo.

In [18]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_predict

espacio_pipeline_xgb_u3 = {f"modelo__{nombre}": valores for nombre, valores in ESPACIO_XGBOOST.items()}
CORTES_BUSQUEDA_U3 = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]
metricas_busqueda_u3 = {"ap": "average_precision", "auc": "roc_auc"}

def indice_ganador_u3(resultados):
    puntuaciones = np.asarray(resultados["mean_test_ap"])
    if not np.isfinite(puntuaciones).all():
        raise ValueError("Hay puntuaciones de búsqueda no finitas.")
    return min(range(len(puntuaciones)), key=lambda i: (
        -puntuaciones[i],
        resultados["params"][i]["modelo__n_estimators"],
        resultados["params"][i]["modelo__max_depth"],
        resultados["params"][i]["modelo__learning_rate"],
        resultados["params"][i]["modelo__subsample"],
    ))

def metricas_binarias_u3(y_real, probabilidades, corte):
    decisiones = (np.asarray(probabilidades) >= corte).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_real, decisiones, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = recall_score(y_real, decisiones, zero_division=0)
    return {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Precisión": precision, "Recall": recall,
        "F1": f1_score(y_real, decisiones, zero_division=0),
        "Accuracy": accuracy_score(y_real, decisiones),
        "ROC-AUC": roc_auc_score(y_real, probabilidades),
        "Average Precision": average_precision_score(y_real, probabilidades),
        "Cumple ambos KPI": bool(pd.notna(precision) and precision >= META_PRECISION and recall >= META_RECALL),
    }

def elegir_corte_interno_u3(y_interno, proba_interna):
    tabla = pd.DataFrame([
        {"Corte": corte, **metricas_binarias_u3(y_interno, proba_interna, corte)}
        for corte in CORTES_BUSQUEDA_U3
    ])
    elegibles = tabla.loc[tabla["Cumple ambos KPI"]]
    if elegibles.empty:
        return 0.50, False, tabla
    ganador = elegibles.sort_values(["F1", "Precisión", "Corte"], ascending=[False, False, False]).iloc[0]
    return float(ganador["Corte"]), True, tabla

cv_externo_u3 = StratifiedGroupKFold(n_splits=PLIEGUES_EXTERNOS, shuffle=True, random_state=SEMILLA)
particiones_externas_u3 = list(cv_externo_u3.split(
    X_train_sin_artista_u3, y_train_u3, groups=claves_train_u3,
))
print(f"Protocolo: {len(particiones_externas_u3)} rondas externas; {PLIEGUES_BUSQUEDA} pliegues internos nuevos por ronda.")
print("Criterio: AP media interna; las respuestas externas no eligen parámetros ni cortes.")

Protocolo: 3 rondas externas; 5 pliegues internos nuevos por ronda.
Criterio: AP media interna; las respuestas externas no eligen parámetros ni cortes.


### Explicación de la configuración y funciones

1. Se importan las dos búsquedas y `cross_val_predict`, que generará probabilidades internas sin ajustar con la fila que se predice.
2. La comprensión de diccionario añade `modelo__` a cada hiperparámetro. Ese prefijo indica que debe cambiarse en el paso `modelo` del Pipeline, manteniendo la preparación dentro de cada ajuste.
3. La lista de cortes y el diccionario de métricas fijan las reglas antes de ver resultados. La búsqueda usa puntuaciones continuas, no F1 al corte 0,50.
4. `indice_ganador_u3` comprueba resultados finitos y devuelve el índice de mayor AP media. El signo negativo permite ordenar de mayor a menor mediante `min`. En empates exactos se prefieren menos árboles, menor profundidad, menor tasa y menor subsample, en ese orden. Los dos últimos son desempates deterministas, no una afirmación universal de simplicidad.
5. Esta función se pasa como `refit`. La búsqueda reajusta al ganador con todos los datos de desarrollo. Como `refit` es una función, leeremos su AP desde `cv_results_` y `best_index_`, sin suponer que existe `best_score_`.
6. `metricas_binarias_u3` obtiene decisiones con el corte indicado, calcula los cuatro conteos y las métricas. Devuelve un diccionario; precisión indefinida queda como NaN y no cumple KPI.
7. `elegir_corte_interno_u3` evalúa la lista fijada, conserva cortes viables y aplica los desempates. Si no hay ninguno, devuelve 0,50, el indicador falso y toda la tabla para documentarlo. No cambia metas para conseguir aprobación.
8. `StratifiedGroupKFold` externo crea tres pares de posiciones sobre todo entrenamiento. La agrupación impide separar claves; la estratificación intenta equilibrar clases. Estas posiciones externas son distintas de las cinco del bloque 10.

No se ejecuta aquí una búsqueda: se definen herramientas y separaciones. Las funciones no reciben las respuestas del pliegue externo para seleccionar parámetros o corte.

### 13.2. Ejecución dentro de cada ronda externa

El siguiente bloque puede tardar varios minutos. Se usa `n_jobs=1` en búsquedas y modelos para controlar memoria y evitar paralelismo anidado. Cada candidato recibe un Pipeline completo, por lo que ni mediana ni categorías se ajustan antes de dividir.

Se calculan además mayoritaria y XGBoost inicial en los **mismos tres pliegues externos**. Esto permite comparar con el procedimiento optimizado sin mezclar los tamaños de ajuste de tres pliegues con los resultados anteriores de cinco pliegues. Las cifras anteriores se conservan como antecedentes.

Si se quiere evaluar la regla que elige entre Grid y Random, debe decidir dentro de cada desarrollo. La regla predefinida será mayor AP interna; si empatan exactamente, se prefiere Random por su menor presupuesto. Los resultados externos de la estrategia seleccionada se guardarán como procedimiento adicional, no como un modelo distinto.

In [19]:
nombres_estrategias_u3 = ["Mayoritaria externa", "XGBoost inicial externo", "Grid", "Random", "Selección interna"]
proba_externa_u3 = {nombre: pd.Series(np.nan, index=y_train_u3.index, dtype="float64") for nombre in nombres_estrategias_u3}
pred_externa_u3 = {nombre: pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64") for nombre in nombres_estrategias_u3}
veces_externo_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
filas_controles_anidados_u3 = []
filas_busquedas_u3 = []
filas_metricas_externas_u3 = []
tablas_candidatos_u3 = {}
tablas_cortes_internos_u3 = {}
conteo_ajustes_anidados_u3 = 0

for ronda, (pos_desarrollo, pos_externo) in enumerate(particiones_externas_u3, start=1):
    X_desarrollo = X_train_sin_artista_u3.iloc[pos_desarrollo]
    y_desarrollo = y_train_u3.iloc[pos_desarrollo]
    grupos_desarrollo = claves_train_u3.iloc[pos_desarrollo]
    X_externo = X_train_sin_artista_u3.iloc[pos_externo]
    y_externo = y_train_u3.iloc[pos_externo]
    grupos_externos = claves_train_u3.iloc[pos_externo]
    if set(grupos_desarrollo).intersection(set(grupos_externos)) or len(X_desarrollo.index.union(X_externo.index).intersection(ids_test_u2)):
        raise ValueError("La separación externa comparte claves o incluye test.")
    if set(y_desarrollo.unique()) != {0, 1} or set(y_externo.unique()) != {0, 1}:
        raise ValueError("Falta una clase en una partición externa.")
    cv_interno = StratifiedGroupKFold(n_splits=PLIEGUES_BUSQUEDA, shuffle=True, random_state=SEMILLA)
    particiones_internas = list(cv_interno.split(X_desarrollo, y_desarrollo, groups=grupos_desarrollo))
    cobertura_interna = np.zeros(len(X_desarrollo), dtype=int)
    for interna, (pos_fit, pos_val) in enumerate(particiones_internas, start=1):
        claves_fit = set(grupos_desarrollo.iloc[pos_fit])
        claves_val = set(grupos_desarrollo.iloc[pos_val])
        if claves_fit.intersection(claves_val) or len(X_desarrollo.index[pos_fit].intersection(X_desarrollo.index[pos_val])):
            raise ValueError("La separación interna comparte IDs o claves.")
        if len(X_desarrollo.index[pos_fit].union(X_desarrollo.index[pos_val])) != len(X_desarrollo):
            raise ValueError("La separación interna no cubre desarrollo.")
        if X_desarrollo.iloc[pos_fit][variables_numericas_u3].isna().all().any():
            raise ValueError("Una variable numérica está totalmente ausente en ajuste interno.")
        if set(y_desarrollo.iloc[pos_fit].unique()) != {0, 1} or set(y_desarrollo.iloc[pos_val].unique()) != {0, 1}:
            raise ValueError("Falta una clase en una partición interna.")
        cobertura_interna[pos_val] += 1
        filas_controles_anidados_u3.append({
            "Externa": ronda, "Interna": interna, "IDs desarrollo": len(X_desarrollo),
            "IDs evaluación externa": len(X_externo), "IDs fit interno": len(pos_fit), "IDs validación interna": len(pos_val),
            "% positivas interna": y_desarrollo.iloc[pos_val].mean() * 100,
            "Claves compartidas internas": 0, "Claves compartidas externas": 0, "IDs test": 0,
        })
    if not (cobertura_interna == 1).all():
        raise ValueError("La cobertura interna no es una validación por ID de desarrollo.")
    print(f"Ronda externa {ronda}: {len(X_desarrollo)} IDs para desarrollo; {len(X_externo)} reservados.", flush=True)
    estrategias_ronda = {}
    for metodo in ["Grid", "Random"]:
        argumentos = dict(
            estimator=clone(flujo_xgb_inicial_u3), cv=particiones_internas,
            scoring=metricas_busqueda_u3, refit=indice_ganador_u3,
            n_jobs=1, error_score="raise", return_train_score=False,
        )
        if metodo == "Grid":
            busqueda = GridSearchCV(param_grid=espacio_pipeline_xgb_u3, **argumentos)
        else:
            busqueda = RandomizedSearchCV(param_distributions=espacio_pipeline_xgb_u3, n_iter=10, random_state=SEMILLA, **argumentos)
        inicio_busqueda = perf_counter()
        busqueda.fit(X_desarrollo, y_desarrollo)
        segundos_busqueda = perf_counter() - inicio_busqueda
        candidatos = pd.DataFrame(busqueda.cv_results_)
        tablas_candidatos_u3[(ronda, metodo)] = candidatos
        mejor = busqueda.best_index_
        ap_interna = float(candidatos.loc[mejor, "mean_test_ap"])
        inicio_cortes = perf_counter()
        proba_interna = cross_val_predict(
            clone(busqueda.best_estimator_), X_desarrollo, y_desarrollo,
            cv=particiones_internas, method="predict_proba", n_jobs=1,
        )[:, 1]
        if not np.isfinite(proba_interna).all():
            raise ValueError("Las probabilidades internas no son finitas.")
        corte, corte_viable, tabla_cortes = elegir_corte_interno_u3(y_desarrollo, proba_interna)
        segundos_cortes = perf_counter() - inicio_cortes
        tablas_cortes_internos_u3[(ronda, metodo)] = tabla_cortes
        estrategias_ronda[metodo] = {
            "estimador": busqueda.best_estimator_, "AP interna": ap_interna,
            "corte": corte, "viable": corte_viable,
        }
        ajustes_busqueda = len(candidatos) * PLIEGUES_BUSQUEDA
        conteo_ajustes_anidados_u3 += ajustes_busqueda + 1 + PLIEGUES_BUSQUEDA
        parametros_legibles = {nombre.replace("modelo__", ""): valor for nombre, valor in busqueda.best_params_.items()}
        filas_busquedas_u3.append({
            "Externa": ronda, "Método": metodo, "Configuraciones": len(candidatos),
            "Ajustes CV": ajustes_busqueda, "Reajustes ganador": 1, "Ajustes OOF corte": PLIEGUES_BUSQUEDA,
            "AP media interna": ap_interna, "DE AP interna": float(candidatos.loc[mejor, "std_test_ap"]),
            "AUC media interna": float(candidatos.loc[mejor, "mean_test_auc"]),
            "Corte interno": corte, "Corte interno viable": corte_viable,
            "Segundos búsqueda y refit": segundos_busqueda, "Segundos OOF y corte": segundos_cortes,
            "Parámetros": parametros_legibles,
        })
        print(f"  {metodo}: {len(candidatos)} configuraciones; AP interna {ap_interna:.4f}; corte {corte:.2f}; {segundos_busqueda:.1f}s búsqueda/refit.", flush=True)
    elegido_interno = max(["Random", "Grid"], key=lambda nombre: estrategias_ronda[nombre]["AP interna"])
    # La elección anterior no usa ninguna métrica externa.
    for metodo in ["Grid", "Random"]:
        estrategia = estrategias_ronda[metodo]
        proba = estrategia["estimador"].predict_proba(X_externo)[:, 1]
        if not np.isfinite(proba).all() or not ((proba >= 0) & (proba <= 1)).all():
            raise ValueError("Puntuaciones externas no válidas.")
        proba_externa_u3[metodo].loc[y_externo.index] = proba
        pred_externa_u3[metodo].loc[y_externo.index] = (proba >= estrategia["corte"]).astype("int8")
        for tipo, corte_eval in [("Fijo 0.50", 0.50), ("Elegido interno", estrategia["corte"])]:
            filas_metricas_externas_u3.append({"Externa": ronda, "Método": metodo, "Decisión": tipo,
                "Corte": corte_eval, **metricas_binarias_u3(y_externo, proba, corte_eval)})
    proba_elegida = proba_externa_u3[elegido_interno].loc[y_externo.index]
    corte_elegido = estrategias_ronda[elegido_interno]["corte"]
    proba_externa_u3["Selección interna"].loc[y_externo.index] = proba_elegida
    pred_externa_u3["Selección interna"].loc[y_externo.index] = pred_externa_u3[elegido_interno].loc[y_externo.index]
    filas_metricas_externas_u3.append({"Externa": ronda, "Método": "Selección interna", "Decisión": elegido_interno,
        "Corte": corte_elegido, **metricas_binarias_u3(y_externo, proba_elegida, corte_elegido)})
    for nombre in ["Mayoritaria externa", "XGBoost inicial externo"]:
        if nombre == "Mayoritaria externa":
            flujo_base = Pipeline([("preparacion", clone(preparador_sin_artista_u3)), ("modelo", DummyClassifier(strategy="most_frequent"))])
        else:
            flujo_base = clone(flujo_xgb_inicial_u3)
        flujo_base.fit(X_desarrollo, y_desarrollo)
        conteo_ajustes_anidados_u3 += 1
        proba_base = flujo_base.predict_proba(X_externo)[:, 1]
        proba_externa_u3[nombre].loc[y_externo.index] = proba_base
        pred_externa_u3[nombre].loc[y_externo.index] = (proba_base >= 0.50).astype("int8")
        filas_metricas_externas_u3.append({"Externa": ronda, "Método": nombre, "Decisión": "Fijo 0.50",
            "Corte": 0.50, **metricas_binarias_u3(y_externo, proba_base, 0.50)})
        del flujo_base
    veces_externo_u3.loc[y_externo.index] += 1
    del estrategias_ronda, busqueda
    print(f"  Ronda {ronda} evaluada; estrategia elegida internamente: {elegido_interno}.", flush=True)

if not veces_externo_u3.eq(1).all():
    raise ValueError("Cada ID debe evaluarse externamente una vez.")
for nombre in nombres_estrategias_u3:
    if proba_externa_u3[nombre].isna().any() or pred_externa_u3[nombre].isna().any():
        raise ValueError(f"Predicciones externas incompletas: {nombre}.")
controles_anidados_u3 = pd.DataFrame(filas_controles_anidados_u3)
resumen_busquedas_u3 = pd.DataFrame(filas_busquedas_u3)
metricas_rondas_anidadas_u3 = pd.DataFrame(filas_metricas_externas_u3)
print(f"Ajustes ejecutados en este bloque: {conteo_ajustes_anidados_u3}.")
display(controles_anidados_u3.round(3))
display(resumen_busquedas_u3.round(4))

Ronda externa 1: 47709 IDs para desarrollo; 24019 reservados.
  Grid: 16 configuraciones; AP interna 0.4824; corte 0.20; 69.8s búsqueda/refit.
  Random: 10 configuraciones; AP interna 0.4824; corte 0.20; 40.1s búsqueda/refit.
  Ronda 1 evaluada; estrategia elegida internamente: Random.
Ronda externa 2: 47900 IDs para desarrollo; 23828 reservados.
  Grid: 16 configuraciones; AP interna 0.4731; corte 0.20; 69.8s búsqueda/refit.
  Random: 10 configuraciones; AP interna 0.4731; corte 0.20; 40.4s búsqueda/refit.
  Ronda 2 evaluada; estrategia elegida internamente: Random.
Ronda externa 3: 47847 IDs para desarrollo; 23881 reservados.
  Grid: 16 configuraciones; AP interna 0.4765; corte 0.20; 73.3s búsqueda/refit.
  Random: 10 configuraciones; AP interna 0.4765; corte 0.20; 44.8s búsqueda/refit.
  Ronda 3 evaluada; estrategia elegida internamente: Random.
Ajustes ejecutados en este bloque: 432.


,Externa,Interna,IDs desarrollo,IDs evaluación externa,IDs fit interno,IDs validación interna,% positivas interna,Claves compartidas internas,Claves compartidas externas,IDs test
0,1,1,47709,24019,38107,9602,10.821,0,0,0
1,1,2,47709,24019,38225,9484,11.419,0,0,0
2,1,3,47709,24019,38163,9546,10.800,0,0,0
3,1,4,47709,24019,38206,9503,11.144,0,0,0
4,1,5,47709,24019,38135,9574,11.051,0,0,0
5,2,1,47900,23828,38312,9588,11.066,0,0,0
6,2,2,47900,23828,38305,9595,11.006,0,0,0
7,2,3,47900,23828,38302,9598,11.294,0,0,0
8,2,4,47900,23828,38376,9524,10.773,0,0,0
9,2,5,47900,23828,38305,9595,10.610,0,0,0


,Externa,Método,Configuraciones,Ajustes CV,Reajustes ganador,Ajustes OOF corte,AP media interna,DE AP interna,AUC media interna,Corte interno,Corte interno viable,Segundos búsqueda y refit,Segundos OOF y corte,Parámetros
0,1,Grid,16,80,1,5,0.4824,0.0145,0.8701,0.2,True,69.7682,7.5662,"{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}"
1,1,Random,10,50,1,5,0.4824,0.0145,0.8701,0.2,True,40.1135,6.0059,"{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}"
2,2,Grid,16,80,1,5,0.4731,0.0188,0.8726,0.2,True,69.7540,6.1443,"{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}"
3,2,Random,10,50,1,5,0.4731,0.0188,0.8726,0.2,True,40.4091,5.9623,"{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}"
4,3,Grid,16,80,1,5,0.4765,0.0111,0.8716,0.2,True,73.3076,6.3127,"{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}"
5,3,Random,10,50,1,5,0.4765,0.0111,0.8716,0.2,True,44.8248,7.1094,"{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}"


### Explicación del bucle anidado

1. Los diccionarios guardan una serie por estrategia con el índice de entrenamiento. Se reservan puntuaciones, decisiones y un contador de evaluación externa. Las listas guardan controles, candidatos, cortes y resultados para auditarlos.
2. El bucle externo separa desarrollo y evaluación por `.iloc`. Sus claves no se intersectan; ningún ID pertenece a test. Se exige presencia de ambas clases.
3. Se crean cinco particiones internas **sobre desarrollo**, con sus claves correspondientes. Sus posiciones son locales a desarrollo, no posiciones sobre todo entrenamiento. Se comprueban claves e IDs separados, cobertura, ambas clases y que ninguna variable numérica esté completamente ausente en ajuste.
4. El contador interno exige que cada fila de desarrollo se valide una vez. La tabla registra tamaños reales; no se supone que sean los 14.346 de la búsqueda sobre todo entrenamiento.
5. `argumentos` declara el Pipeline completo, las mismas particiones, AP y AUC, el desempate y errores que detienen ejecución. `return_train_score=False` evita calcular puntuaciones de ajuste que no necesitamos para seleccionar.
6. Grid revisa las 16 combinaciones; Random selecciona diez sin reemplazo porque el espacio contiene listas finitas. La semilla se fija, sin repetir búsquedas aleatorias para escoger una favorable.
7. `fit(X_desarrollo, y_desarrollo)` realiza validación interna y reajusta el ganador con todo desarrollo. En cada ajuste se aprende de nuevo el Pipeline. La matriz externa no se entrega a búsqueda ni early stopping.
8. `cv_results_` conserva todas las configuraciones y puntuaciones; `best_index_` identifica al ganador. La media y desviación de AP son de cinco validaciones internas; la desviación no es un intervalo de confianza.
9. `cross_val_predict(clone(best_estimator_), ..., cv=particiones_internas)` reajusta el ganador cinco veces para obtener puntuaciones internas sin entrenar con cada fila que predice. Usa parámetros seleccionados con todo desarrollo: ese resultado interno puede ser optimista y sirve para elegir el corte, no para confirmar rendimiento.
10. La función de corte consulta solo `y_desarrollo`. Se guardan las 16 alternativas y si encontró una viable. El ganador que ya reajustó la búsqueda no necesita otro fit para predecir externo; el umbral solo convierte sus puntuaciones en decisiones.
11. Se suman ajustes de búsqueda, un refit y cinco ajustes para el corte. Los tiempos de búsqueda incluyen refit; el tiempo de cortes incluye OOF y selección. No son estimaciones de producción.
12. Se elige el método por AP interna antes de predecir externo. `max` conserva Random en empate porque aparece primero. Es una regla declarada, no una elección por sus resultados externos.
13. Ambos ganadores predicen externo. Se evalúa el corte fijo y el interno; la selección interna reutiliza las predicciones de su ganador, sin otro entrenamiento. Las métricas externas no modifican las decisiones tomadas.
14. Mayoritaria y XGBoost inicial se reajustan en el mismo desarrollo y se evalúan en el mismo externo con corte fijo. Se añaden dos ajustes por ronda. Las probabilidades constantes de mayoritaria producen siempre clase cero; su corte es una representación técnica de esa decisión constante, no un umbral elegido.
15. Al terminar cada ronda se eliminan sus modelos y se incrementa el contador externo. Las comprobaciones finales exigen una predicción por ID y estrategia. Los resultados guardados son tablas y predicciones; no hay un modelo final de todo entrenamiento.

**Costo ejecutable previsto:** 390 ajustes de candidatos + 6 reajustes de ganadores + 30 ajustes OOF internos para cortes + 3 ajustes XGBoost inicial + 3 mayoritarias = **432 ajustes completos**. No incluye los ajustes de los bloques previos. La selección entre Grid y Random no añade ajustes. Se ejecutan ambos aunque compartan candidatos para comparar las herramientas; el costo no es una comparación con presupuestos idénticos.

In [20]:
filas_resumen_externo_u3 = []
matrices_anidadas_u3 = {}
for nombre in nombres_estrategias_u3:
    proba = proba_externa_u3[nombre]
    decisiones = pred_externa_u3[nombre].astype("int64")
    tn, fp, fn, tp = confusion_matrix(y_train_u3, decisiones, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = recall_score(y_train_u3, decisiones, zero_division=0)
    fila = {"Procedimiento": nombre, "Decisión": "Interna por ronda" if nombre in ["Grid", "Random", "Selección interna"] else "Fijo 0.50",
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Precisión": precision, "Recall": recall, "F1": f1_score(y_train_u3, decisiones, zero_division=0),
        "Accuracy": accuracy_score(y_train_u3, decisiones), "ROC-AUC": roc_auc_score(y_train_u3, proba),
        "Average Precision": average_precision_score(y_train_u3, proba),
        "Cumple precisión": bool(pd.notna(precision) and precision >= META_PRECISION),
        "Cumple recall": bool(recall >= META_RECALL)}
    fila["KPI simultáneos"] = "Cumple" if fila["Cumple precisión"] and fila["Cumple recall"] else "No cumple"
    filas_resumen_externo_u3.append(fila)
    matrices_anidadas_u3[nombre] = pd.DataFrame([[tn, fp], [fn, tp]], index=["Real 0", "Real 1"], columns=["Predice 0", "Predice 1"])
for nombre in ["Grid", "Random"]:
    m = metricas_binarias_u3(y_train_u3, proba_externa_u3[nombre], 0.50)
    filas_resumen_externo_u3.append({"Procedimiento": nombre, "Decisión": "Fijo 0.50", **m,
        "Cumple precisión": bool(pd.notna(m["Precisión"]) and m["Precisión"] >= META_PRECISION),
        "Cumple recall": bool(m["Recall"] >= META_RECALL),
        "KPI simultáneos": "Cumple" if m["Cumple ambos KPI"] else "No cumple"})
resumen_externo_anidado_u3 = pd.DataFrame(filas_resumen_externo_u3).drop(columns=["Cumple ambos KPI"], errors="ignore")
comparacion_costos_busqueda_u3 = resumen_busquedas_u3.groupby("Método").agg(
    Configuraciones_evaluadas=("Configuraciones", "sum"), Ajustes_CV=("Ajustes CV", "sum"),
    Reajustes_ganador=("Reajustes ganador", "sum"), Ajustes_OOF_corte=("Ajustes OOF corte", "sum"),
    Segundos_busqueda_y_refit=("Segundos búsqueda y refit", "sum"), Segundos_OOF_y_corte=("Segundos OOF y corte", "sum"),
)
display(metricas_rondas_anidadas_u3.round(4))
display(resumen_externo_anidado_u3.round(6))
display(comparacion_costos_busqueda_u3.round(2))
for nombre in ["Grid", "Random", "Selección interna"]:
    print(f"Matriz externa de {nombre} con corte interno:")
    display(matrices_anidadas_u3[nombre])
print(f"Cobertura externa: {veces_externo_u3.eq(1).sum():,} IDs evaluados una vez por procedimiento.")

,Externa,Método,Decisión,Corte,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI
0,1,Grid,Fijo 0.50,0.5,21107,321,2088,503,0.6104,0.1941,0.2946,0.8997,0.8753,0.4727,False
1,1,Grid,Elegido interno,0.2,19097,2331,1009,1582,0.4043,0.6106,0.4865,0.8609,0.8753,0.4727,True
2,1,Random,Fijo 0.50,0.5,21107,321,2088,503,0.6104,0.1941,0.2946,0.8997,0.8753,0.4727,False
3,1,Random,Elegido interno,0.2,19097,2331,1009,1582,0.4043,0.6106,0.4865,0.8609,0.8753,0.4727,True
4,1,Selección interna,Random,0.2,19097,2331,1009,1582,0.4043,0.6106,0.4865,0.8609,0.8753,0.4727,True
5,1,Mayoritaria externa,Fijo 0.50,0.5,21428,0,2591,0,NaN,0.0000,0.0000,0.8921,0.5000,0.1079,False
6,1,XGBoost inicial externo,Fijo 0.50,0.5,21319,109,2329,262,0.7062,0.1011,0.1769,0.8985,0.8363,0.4242,False
7,2,Grid,Fijo 0.50,0.5,20924,288,2097,519,0.6431,0.1984,0.3032,0.8999,0.8688,0.4870,False
8,2,Grid,Elegido interno,0.2,18820,2392,1000,1616,0.4032,0.6177,0.4879,0.8576,0.8688,0.4870,True
9,2,Random,Fijo 0.50,0.5,20924,288,2097,519,0.6431,0.1984,0.3032,0.8999,0.8688,0.4870,False


,Procedimiento,Decisión,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple precisión,Cumple recall,KPI simultáneos
0,Mayoritaria externa,Fijo 0.50,63867,0,7861,0,NaN,0.000000,0.000000,0.890405,0.500000,0.109595,False,False,No cumple
1,XGBoost inicial externo,Fijo 0.50,63608,259,7102,759,0.745580,0.096553,0.170965,0.897376,0.837677,0.436654,True,False,No cumple
2,Grid,Interna por ronda,56916,6951,3033,4828,0.409882,0.614171,0.491650,0.860807,0.873512,0.484730,True,True,Cumple
3,Random,Interna por ronda,56916,6951,3033,4828,0.409882,0.614171,0.491650,0.860807,0.873512,0.484730,True,True,Cumple
4,Selección interna,Interna por ronda,56916,6951,3033,4828,0.409882,0.614171,0.491650,0.860807,0.873512,0.484730,True,True,Cumple
5,Grid,Fijo 0.50,63027,840,6315,1546,0.647946,0.196667,0.301747,0.900248,0.873512,0.484730,True,False,No cumple
6,Random,Fijo 0.50,63027,840,6315,1546,0.647946,0.196667,0.301747,0.900248,0.873512,0.484730,True,False,No cumple


,Configuraciones_evaluadas,Ajustes_CV,Reajustes_ganador,Ajustes_OOF_corte,Segundos_busqueda_y_refit,Segundos_OOF_y_corte
Método,,,,,,
Grid,48,240,3,15,212.83,20.02
Random,30,150,3,15,125.35,19.08


Matriz externa de Grid con corte interno:


,Predice 0,Predice 1
Real 0,56916,6951
Real 1,3033,4828


Matriz externa de Random con corte interno:


,Predice 0,Predice 1
Real 0,56916,6951
Real 1,3033,4828


Matriz externa de Selección interna con corte interno:


,Predice 0,Predice 1
Real 0,56916,6951
Real 1,3033,4828


Cobertura externa: 71,728 IDs evaluados una vez por procedimiento.


### Explicación de la evaluación externa y comparación

1. Para cada procedimiento se reúnen las decisiones externas ya fijadas y sus puntuaciones. Las métricas globales se calculan por ID, no promediando métricas de rondas.
2. TN/FP/FN/TP muestran si el procedimiento intercambia falsas alarmas por detecciones. Cuando el corte cambia por ronda se usan las decisiones guardadas, no se impone un corte global después de ver respuestas.
3. Se calculan estados de precisión, recall y cumplimiento simultáneo. La precisión indefinida de mayoritaria permanece NaN y no cumple; no se confunde con un cero válido.
4. Las filas adicionales de Grid y Random al corte fijo 0,50 usan las mismas puntuaciones externas. Permiten distinguir cuánto cambia la decisión con el corte elegido internamente, sin nuevos ajustes. AUC y AP son iguales entre los dos cortes de un mismo método.
5. `groupby(...).agg(...)` suma configuraciones, ajustes y tiempos por herramienta. Las 48 de Grid son 16 repetidas en tres desarrollos, no 48 configuraciones únicas. Random evalúa 30 en total, con diez por ronda.
6. Los tiempos corresponden a este equipo y esta ejecución, con Grid antes de Random. El orden, calentamiento y procesos del equipo influyen; no se afirman ventajas temporales universales. El número de ajustes es una medida de presupuesto más estable.
7. La tabla por ronda describe variación externa y muestra qué método eligió la regla interna. Una diferencia pequeña entre dos resultados no demuestra superioridad estadística.
8. Las matrices tienen filas reales y columnas predichas; cada una suma 71.728. El conteo final verifica una evaluación externa por ID, sin test.

**Interpretación metodológica:** las métricas externas corresponden al proceso de búsqueda y elección de corte. El AP medio interno del ganador no es su AP externo. El AUC/AP global externo reúne puntuaciones generadas por tres modelos distintos; se reportan también por ronda porque las escalas pueden variar. No se presenta esa dispersión como un intervalo de confianza.

La fila «Selección interna» evalúa la regla de elegir herramienta usando desarrollo. Elegir después el mejor procedimiento según esta tabla externa y presentar esa misma cifra como nueva confirmación independiente sería reutilizar la evaluación para seleccionar. Tampoco se afirma que exista una única configuración o corte final: cada ronda puede elegir valores diferentes.

### 13.3. Qué queda pendiente

Se conserva el test histórico sin consulta. Las búsquedas usan solo entrenamiento, con versiones juntas en cada nivel, preparación dentro de cada fit y una evaluación externa por ID. La exploración global de la Unidad 1 y decisiones previas sobre entrenamiento siguen siendo limitaciones históricas: la validación anidada no crea datos externos nuevos.

No se ha ajustado aún un modelo final sobre todo entrenamiento ni se ha aplicado balanceo. La incorporación de artista continúa en el bloque 14. La siguiente decisión será revisar si los KPI se cumplen y qué limitaciones muestran los errores, antes de ampliar el experimento. Si se realizan nuevos cambios motivados por estas métricas externas, deberán documentarse como desarrollo adicional.

Fuentes: [validación anidada](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html), [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) y [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html).

**Resultados ejecutados:** se completaron 432 ajustes de este bloque. Las tablas anteriores muestran los resultados reales externos, los parámetros y cortes elegidos dentro de cada desarrollo. El cumplimiento se determina exclusivamente con las condiciones fijadas, sin cambiar metas ni umbrales a partir de la evaluación externa.


### 13.4. Lectura simple de los resultados ejecutados

Las cifras siguientes corresponden a las predicciones externas de las tres rondas. No proceden de test ni de elegir parámetros con las respuestas externas.

**Grid:** cortes elegidos internamente por ronda: 0.20, 0.20, 0.20. Detectó 4,828 de 7.861 positivas, omitió 3,033 y generó 6,951 falsas alarmas. Precisión 40.99%, recall 61.42% y F1 0.4916. El resultado global de KPI es **Cumple**; 3 de las tres rondas externas cumplieron ambas metas. El cumplimiento global no garantiza que toda partición o población futura las cumpla.
Al mantener corte 0,50, ese mismo procedimiento obtuvo precisión 64.79%, recall 19.67% y F1 0.3017. La diferencia frente al corte interno muestra el efecto de cambiar la regla de decisión sobre las mismas puntuaciones, no de volver a entrenar otros árboles. AP externa global 0.4847 y ROC-AUC 0.8735 se mantienen entre ambos cortes.

**Random:** cortes elegidos internamente por ronda: 0.20, 0.20, 0.20. Detectó 4,828 de 7.861 positivas, omitió 3,033 y generó 6,951 falsas alarmas. Precisión 40.99%, recall 61.42% y F1 0.4916. El resultado global de KPI es **Cumple**; 3 de las tres rondas externas cumplieron ambas metas. El cumplimiento global no garantiza que toda partición o población futura las cumpla.
Al mantener corte 0,50, ese mismo procedimiento obtuvo precisión 64.79%, recall 19.67% y F1 0.3017. La diferencia frente al corte interno muestra el efecto de cambiar la regla de decisión sobre las mismas puntuaciones, no de volver a entrenar otros árboles. AP externa global 0.4847 y ROC-AUC 0.8735 se mantienen entre ambos cortes.

**Referencia inicial comparable:** XGBoost inicial, reajustado en estas tres rondas externas y con corte fijo 0,50, obtuvo AP 0.4367, AUC 0.8377, precisión 74.56% y recall 9.66%. Es la referencia adecuada para este contraste; el bloque 12 utilizó cinco pliegues con más filas por ajuste y por eso conserva cifras diferentes.

**Comparación de herramientas en esta ejecución:** las puntuaciones externas de Grid y Random fueron exactamente iguales. Random revisó 10 de las 16 configuraciones por ronda: el ahorro de ajustes de candidatos fue 37,5% (150 frente a 240 en las tres rondas). Eso no garantiza encontrar siempre la misma solución con otro espacio, semilla o dataset.

**Costo Grid:** 240 ajustes de candidatos, 3 reajustes de ganadores y 15 ajustes internos para cortes. Búsqueda más reajuste: 212.8 segundos acumulados; OOF y corte: 20.0 segundos. Son tiempos de esta ejecución y equipo, no una promesa de rendimiento.

**Costo Random:** 150 ajustes de candidatos, 3 reajustes de ganadores y 15 ajustes internos para cortes. Búsqueda más reajuste: 125.3 segundos acumulados; OOF y corte: 19.1 segundos. Son tiempos de esta ejecución y equipo, no una promesa de rendimiento.

**Interpretación y límite:** se compararon ambos procedimientos sin modificar metas a partir del resultado externo. Una diferencia entre sus cifras no prueba superioridad estadística; una igualdad tampoco demuestra equivalencia universal. La variante con artista se incorpora y evalúa en el bloque 14; todavía no se ha ajustado un modelo final. Si se cambian reglas a partir de esta revisión, se documentará como desarrollo adicional, no como nueva evaluación independiente.


**Parámetros elegidos en las tres rondas por ambas herramientas:** 200 árboles, profundidad máxima 6, tasa de aprendizaje 0,10 y subsample 0,8. El resto conserva la configuración inicial. Los dos primeros están en el extremo superior de la malla ensayada: son los mejores valores encontrados en este espacio, no un óptimo global demostrado. Esto tampoco obliga a ampliar la búsqueda después de ver la evaluación externa.

**Costo de los errores:** al corte interno se detectaron 4.828 positivas y se omitieron 3.033; hubo 6.951 falsas alarmas. La accuracy baja a 86,08% frente al 89,04% de la referencia mayoritaria, pero el procedimiento ahora detecta positivas. Esa reducción de accuracy no invalida por sí sola el cumplimiento del objetivo: refleja que aceptamos más falsas alarmas para recuperar más positivas. Con precisión de 40,99%, aproximadamente 59 de cada cien canciones señaladas siguen siendo negativas reales. Con recall de 61,42%, todavía se omite cerca del 39% de las positivas. Cumplir KPI no significa resolver completamente el problema.


## 14. XGBoost con artista: comparación anidada sin fuga de información

### 14.1. Qué cambia y qué se conserva

Añadimos **una característica numérica de artista**, calculada a partir de la etiqueta binaria, no del puntaje continuo. Se mantienen entrenamiento, clase positiva ≥60, KPI, tres particiones externas, cinco internas, preparación musical, espacio de hiperparámetros y regla de elección de corte.

Se usa **RandomizedSearchCV con las mismas diez configuraciones** de la versión sin artista. Esta decisión se documenta antes de ejecutar: en el bloque 13 encontró el mismo resultado que Grid con menos ajustes. La comparación principal será Random sin artista frente a Random con artista, no una búsqueda con distinto presupuesto. No repetimos Grid con artista en este bloque ni afirmamos que se haya ejecutado.

Cada variante selecciona sus parámetros y umbral **dentro de su desarrollo**. Comparar sus resultados externos evalúa dos procedimientos de búsqueda con las mismas reglas; no demuestra el efecto de añadir artista a un único modelo con parámetros congelados. La extensión estaba prevista en el objetivo, pero se realiza después de observar resultados anteriores: no es una prueba externa nueva.

### 14.2. Qué significa el valor de artista

Se recuperan nombres desde la parte de artista de la clave normalizada, separando colaboradores por `;` y eliminando nombres repetidos dentro de una canción. El título y la clave no se entregan al modelo.

Para cada artista, usando exclusivamente respuestas permitidas:

**valor = (positivas del artista + 10 × proporción positiva global permitida) / (canciones del artista + 10).**

El suavizado 10 se conserva de la Unidad 2 y no se optimiza aquí. Ejemplo ilustrativo, no resultado del dataset: con 2 positivas entre 5 canciones y una proporción global de 0,10, el valor es (2 + 10 × 0,10)/(5 + 10) = 0,20. El respaldo reduce extremos basados en pocos ejemplos.

Para colaboradores se promedian sus valores. Un nombre desconocido recibe la proporción global del ajuste permitido; una canción con un artista conocido y otro desconocido promedia ambos. No es una valoración del talento ni una probabilidad calibrada de éxito futuro.

**Protección necesaria en tres niveles:** evaluación externa; validación interna para parámetros y corte; y cinco grupos adicionales de codificación dentro de cada fit. Las filas usadas para entrenar árboles reciben un valor calculado sin su grupo de codificación, incluida su propia clave y todas sus versiones. Validación recibe el mapa completo del ajuste permitido, sin consultar sus respuestas.

Los cinco grupos de codificación **no son otros cinco XGBoost**: construyen una característica. No se calcula una columna global antes de dividir. La separación es por clave de canción, no por artista completo: artistas pueden aparecer en ajuste y validación; evaluar artistas enteramente nuevos sería otra pregunta.

In [21]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class ArtistaCruzadoU3(TransformerMixin, BaseEstimator):
    def __init__(self, suavizado=10.0, grupos_codigo=5, random_state=42):
        self.suavizado = suavizado
        self.grupos_codigo = grupos_codigo
        self.random_state = random_state

    def _artistas(self, X):
        parte = X["clave_cancion"].astype("string").str.split(" || ", regex=False).str[0]
        nombres = parte.str.split(";").map(lambda lista: list(dict.fromkeys(nombre.strip() for nombre in lista if nombre.strip())))
        if not nombres.map(len).gt(0).all():
            raise ValueError("Una clave no contiene nombres de artista utilizables.")
        return nombres

    def _mapa(self, artistas, y):
        respaldo = float(y.mean())
        detalle = pd.DataFrame({"Artista": artistas, "Positiva": y}).explode("Artista")
        conteos = detalle.groupby("Artista")["Positiva"].agg(["sum", "count"])
        mapa = ((conteos["sum"] + self.suavizado * respaldo) / (conteos["count"] + self.suavizado)).to_dict()
        return mapa, respaldo

    def _aplicar(self, artistas, mapa, respaldo):
        detalle = artistas.explode().map(mapa).fillna(respaldo)
        return detalle.groupby(level=0, sort=False).mean().reindex(artistas.index)

    def _salida(self, X, valores):
        if valores.isna().any() or not valores.between(0, 1).all():
            raise ValueError("La codificación de artista no es válida.")
        salida = X.drop(columns="clave_cancion").copy()
        salida["artista_clase_1"] = valores
        return salida

    def _etiquetas(self, X, y):
        if isinstance(y, pd.Series) and not X.index.equals(y.index):
            raise ValueError("Artista requiere respuestas e IDs alineados.")
        valores = np.asarray(y)
        if valores.ndim != 1 or len(valores) != len(X):
            raise ValueError("La cantidad o forma de las respuestas no coincide con X.")
        if not pd.Series(valores).isin([0, 1]).all():
            raise ValueError("Las respuestas deben ser binarias antes de convertir su tipo.")
        return pd.Series(valores, index=X.index, dtype="int8")

    def fit(self, X, y):
        y = self._etiquetas(X, y)
        if X.index.has_duplicates or X["clave_cancion"].isna().any() or not y.isin([0, 1]).all():
            raise ValueError("Hay IDs, claves o respuestas inválidos.")
        if self.suavizado <= 0 or self.grupos_codigo < 2:
            raise ValueError("Configuración de codificación inválida.")
        self.mapa_, self.respaldo_ = self._mapa(self._artistas(X), y)
        self.ids_ajuste_ = X.index.copy()
        self.claves_ajuste_ = frozenset(X["clave_cancion"])
        self.n_features_in_ = X.shape[1]
        return self

    def fit_transform(self, X, y=None, **fit_params):
        y = self._etiquetas(X, y)
        self.fit(X, y)
        claves = X["clave_cancion"]
        unicas = claves.drop_duplicates().sort_values().sample(frac=1, random_state=self.random_state).reset_index(drop=True)
        if len(unicas) < self.grupos_codigo:
            raise ValueError("No hay suficientes claves para codificación cruzada.")
        asignacion = claves.map(pd.Series(np.arange(len(unicas)) % self.grupos_codigo, index=unicas))
        artistas = self._artistas(X)
        valores = pd.Series(np.nan, index=X.index, dtype="float64")
        visitas = pd.Series(0, index=X.index, dtype="int64")
        for grupo in range(self.grupos_codigo):
            consulta = asignacion.eq(grupo)
            fuente = ~consulta
            if set(claves.loc[consulta]).intersection(set(claves.loc[fuente])):
                raise ValueError("La propia clave participa en su codificación.")
            mapa, respaldo = self._mapa(artistas.loc[fuente], y.loc[fuente])
            valores.loc[consulta] = self._aplicar(artistas.loc[consulta], mapa, respaldo)
            visitas.loc[consulta] += 1
        if not visitas.eq(1).all():
            raise ValueError("La codificación cruzada no cubre exactamente cada ID.")
        self.cobertura_cruzada_ = int(visitas.eq(1).sum())
        return self._salida(X, valores)

    def transform(self, X):
        check_is_fitted(self, ["mapa_", "respaldo_", "ids_ajuste_"])
        valores = self._aplicar(self._artistas(X), self.mapa_, self.respaldo_)
        return self._salida(X, valores)

print("Transformador definido: fit_transform cruzado para ajuste; transform con mapa permitido para consulta.")

Transformador definido: fit_transform cruzado para ajuste; transform con mapa permitido para consulta.


### Explicación de cada parte del codificador

1. `BaseEstimator` permite que scikit-learn clone y configure la clase; `TransformerMixin` la integra como transformación del Pipeline. `__init__` guarda únicamente los tres valores fijados, sin aprender datos.
2. `_artistas` toma el texto anterior a ` || ` y lo divide por `;`. `dict.fromkeys` elimina colaboradores repetidos preservando orden; `strip` retira espacios. No recuperamos nombres desde una fuente externa ni identificadores oficiales: hay limitaciones por nombres iguales o inconsistentes.
3. `_mapa` calcula la proporción positiva del conjunto recibido. `explode` crea una fila por colaborador y conserva el ID; una colaboración aporta una observación a cada artista. `groupby` obtiene positivas y canciones para aplicar la fórmula suavizada. El respaldo se calcula por canciones, no por cantidad de colaboradores.
4. `_aplicar` consulta el mapa para cada nombre, sustituye desconocidos por respaldo y promedia por ID. No recibe respuestas de las filas consultadas. `reindex` recupera el orden exacto de entrada.
5. `_salida` exige valores entre 0 y 1 y sin faltantes. Retira `clave_cancion` y añade `artista_clase_1`. La clave puede entrar a la transformación para identificar grupos, pero no llega a los árboles.
6. `_etiquetas` verifica alineación si las respuestas son una Serie; si scikit-learn las entrega como arreglo, verifica forma y cantidad y les asigna el índice de X en el orden recibido. No ordena etiquetas por separado. `fit` exige además IDs únicos, claves válidas y etiquetas binarias. Aprende un mapa completo con su ajuste permitido y registra IDs y claves para controles. El guion bajo final identifica atributos aprendidos.
7. `fit_transform` primero conserva el mapa completo para consultas posteriores. Para las **filas de ajuste**, genera aparte cinco grupos de claves: las ordena, mezcla con semilla fija y asigna el resto `%5`. No utiliza las respuestas para decidir esos grupos.
8. Cada grupo se codifica con los otros cuatro. La intersección verifica que ninguna versión de su clave participe. También el respaldo se calcula solo con la fuente de esa codificación, evitando que la propia respuesta influya indirectamente a través de la proporción global.
9. `visitas` comprueba cobertura única y `_salida` devuelve las filas de ajuste con sus valores cruzados. No usa el mapa completo para producir esos valores de entrenamiento.
10. `transform` comprueba que hubo ajuste y aplica el mapa completo permitido a filas de consulta, sin respuestas. Esta diferencia deliberada entre `fit_transform` y `fit().transform()` protege el entrenamiento; **no deben intercambiarse al construir las características de ajuste**.

El mapa completo ve más filas que los mapas cruzados y puede dar valores diferentes para artistas raros. Las métricas externas evaluarán también esa diferencia. No se calcularán métricas de ajuste usando el mapa completo como si esas predicciones fueran OOF.

In [22]:
# Control pequeño de exclusión: no es una evaluación de rendimiento.
X_control_artista_u3 = pd.DataFrame({"clave_cancion": [
    "comun || a", "comun || a", "comun || b", "comun || c",
    "comun || d", "comun || e", "comun || f", "comun || g",
    "comun || h", "comun || i", "comun || j", "comun || k",
]})
y_control_artista_u3 = pd.Series([0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], dtype="int8")
es_propia_clave_u3 = X_control_artista_u3["clave_cancion"].eq("comun || a")
y_control_cambiado_u3 = y_control_artista_u3.copy()
y_control_cambiado_u3.loc[es_propia_clave_u3] = 1
codigo_control_1 = ArtistaCruzadoU3().fit_transform(X_control_artista_u3, y_control_artista_u3)
codigo_control_2 = ArtistaCruzadoU3().fit_transform(X_control_artista_u3, y_control_cambiado_u3)
codigo_control_array = ArtistaCruzadoU3().fit_transform(X_control_artista_u3, y_control_artista_u3.to_numpy())
if not np.allclose(codigo_control_1["artista_clase_1"], codigo_control_array["artista_clase_1"]):
    raise ValueError("La representación de y como arreglo alteró la codificación.")
if not np.allclose(codigo_control_1.loc[es_propia_clave_u3, "artista_clase_1"], codigo_control_2.loc[es_propia_clave_u3, "artista_clase_1"]):
    raise ValueError("Cambiar las respuestas propias alteró su codificación.")
codificador_control_u3 = ArtistaCruzadoU3().fit(X_control_artista_u3, y_control_artista_u3)
consulta_desconocida_u3 = pd.DataFrame({"clave_cancion": ["nombre nuevo || otro titulo"]}, index=["consulta"])
valor_desconocido_u3 = codificador_control_u3.transform(consulta_desconocida_u3)["artista_clase_1"].iloc[0]
if not np.isclose(valor_desconocido_u3, y_control_artista_u3.mean()):
    raise ValueError("El artista desconocido no recibió el respaldo permitido.")

X_train_con_artista_u3 = X_train_sin_artista_u3.copy()
X_train_con_artista_u3["clave_cancion"] = claves_train_u3
preparador_con_artista_u3 = ColumnTransformer([
    ("numericas", SimpleImputer(strategy="median"), variables_numericas_u3 + ["artista_clase_1"]),
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), variables_categoricas_u3),
    ("generos", "passthrough", variables_genero_u3),
], remainder="drop")
flujo_xgb_con_artista_u3 = Pipeline([
    ("artista", ArtistaCruzadoU3(suavizado=10.0, grupos_codigo=5, random_state=SEMILLA)),
    ("preparacion", preparador_con_artista_u3),
    ("modelo", clone(modelo_xgb_inicial_u3)),
])
print("Control de exclusión de la propia clave: cumple.")
print("Control de artista desconocido y respaldo: cumple.")
print("El Pipeline crea la característica de artista dentro de cada fit; todavía no hay resultados nuevos.")

Control de exclusión de la propia clave: cumple.
Control de artista desconocido y respaldo: cumple.
El Pipeline crea la característica de artista dentro de cada fit; todavía no hay resultados nuevos.


### Explicación de controles y Pipeline

1. Se crea un ejemplo ficticio con dos versiones de la clave `comun || a` y otras diez claves. No se mezcla con las canciones reales.
2. Se cambian las respuestas de ambas versiones de esa clave. Se codifican otra vez las mismas filas con la misma regla y se exige que **su propio valor cruzado no cambie**. Esto comprueba exclusión directa y del respaldo de la propia clave, no demuestra por sí solo toda ausencia de fuga del proyecto.
Además, se comprueba que entregar las mismas respuestas como Serie o arreglo conserve los valores cruzados. El codificador valida que sean binarias antes de convertir su tipo.

3. Se consulta un artista que no existe en el mapa y se comprueba que reciba la proporción global del ajuste ficticio. Es un control funcional, no una métrica predictiva.
4. Se copia la matriz sin artista y se añade la clave solo como dato auxiliar del transformador. La etiqueta no se añade a X.
5. El preparador añade `artista_clase_1` a las diez variables numéricas. Conserva categorías y géneros, sin escalamiento ni selección de variables nuevas por resultados externos.
6. El Pipeline ejecutará codificación cruzada, preparación y XGBoost en ese orden. La búsqueda cambia solo los cuatro parámetros `modelo__...` ya definidos. El clasificador inicial y sus demás parámetros se conservan.
7. No se aprende el artista al crear el Pipeline. `clone` y los ajustes de búsqueda crearán mapas nuevos en cada candidato, pliegue y refit. En estas rondas se esperan 133 columnas finales, una más que las 132 sin artista.

Se registran controles ficticios por ser una transformación supervisada sensible a fuga. Sus resultados «cumple» son metodológicos, no los KPI de precisión y recall.

### 14.3. Búsqueda con artista y evaluación externa

Se conservan `particiones_externas_u3` y se reproducen exactamente las cinco internas de cada desarrollo. Antes de ejecutar se comprueba que las diez configuraciones muestreadas coincidan con Random sin artista. Se reutilizan la métrica AP interna, los desempates y los 16 cortes del bloque 13.

Por cada ronda externa se realizan 50 ajustes de búsqueda, un reajuste del ganador y cinco ajustes para probabilidades OOF internas del corte: **56 ajustes de Pipeline, 168 en total**. Cada uno incluye cinco separaciones adicionales para codificación del artista, pero no cinco modelos XGBoost extra. El tiempo incluirá ese trabajo; no es directamente el tiempo de una búsqueda sin artista.

Se guardarán puntuaciones y decisiones externas, tamaños, cobertura de artistas, parámetros, cortes y matrices. La comparación de variantes utiliza los resultados Random sin artista ya calculados, sin volver a consultar test ni repetir su búsqueda para escoger una más favorable.

In [23]:
from sklearn.model_selection import ParameterSampler

candidatos_random_artista_u3 = list(ParameterSampler(espacio_pipeline_xgb_u3, n_iter=10, random_state=SEMILLA))
for ronda in range(1, PLIEGUES_EXTERNOS + 1):
    anteriores = tablas_candidatos_u3[(ronda, "Random")]["params"].tolist()
    if candidatos_random_artista_u3 != anteriores:
        raise ValueError("La lista de candidatos no coincide con Random sin artista.")

proba_externa_artista_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64")
pred_externa_artista_u3 = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64")
veces_externo_artista_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
filas_busqueda_artista_u3 = []
filas_rondas_artista_u3 = []
filas_controles_artista_u3 = []
candidatos_con_artista_u3 = {}
cortes_con_artista_u3 = {}

for ronda, (pos_desarrollo, pos_externo) in enumerate(particiones_externas_u3, start=1):
    X_desarrollo = X_train_con_artista_u3.iloc[pos_desarrollo]
    y_desarrollo = y_train_u3.iloc[pos_desarrollo]
    grupos_desarrollo = claves_train_u3.iloc[pos_desarrollo]
    X_externo = X_train_con_artista_u3.iloc[pos_externo]
    y_externo = y_train_u3.iloc[pos_externo]
    if set(grupos_desarrollo).intersection(set(claves_train_u3.iloc[pos_externo])) or len(X_desarrollo.index.union(X_externo.index).intersection(ids_test_u2)):
        raise ValueError("La separación externa comparte claves o incluye test.")
    cv_interno = StratifiedGroupKFold(n_splits=PLIEGUES_BUSQUEDA, shuffle=True, random_state=SEMILLA)
    particiones_internas = list(cv_interno.split(X_desarrollo, y_desarrollo, groups=grupos_desarrollo))
    visitas_internas = np.zeros(len(X_desarrollo), dtype=int)
    for interna, (pos_fit, pos_val) in enumerate(particiones_internas, start=1):
        if set(grupos_desarrollo.iloc[pos_fit]).intersection(set(grupos_desarrollo.iloc[pos_val])):
            raise ValueError("Claves compartidas en validación interna.")
        if set(y_desarrollo.iloc[pos_fit].unique()) != {0, 1} or set(y_desarrollo.iloc[pos_val].unique()) != {0, 1}:
            raise ValueError("Falta una clase en la validación interna.")
        control_previo = controles_anidados_u3.loc[(controles_anidados_u3["Externa"] == ronda) & (controles_anidados_u3["Interna"] == interna)].iloc[0]
        if len(pos_fit) != control_previo["IDs fit interno"] or len(pos_val) != control_previo["IDs validación interna"]:
            raise ValueError("Los tamaños internos cambiaron respecto a la comparación anterior.")
        visitas_internas[pos_val] += 1
    if not (visitas_internas == 1).all():
        raise ValueError("Cobertura interna incorrecta.")
    print(f"Artista, ronda externa {ronda}: búsqueda Random de 10 configuraciones.", flush=True)
    busqueda_artista = RandomizedSearchCV(
        clone(flujo_xgb_con_artista_u3), param_distributions=espacio_pipeline_xgb_u3,
        n_iter=10, cv=particiones_internas, scoring=metricas_busqueda_u3,
        refit=indice_ganador_u3, random_state=SEMILLA, n_jobs=1,
        error_score="raise", return_train_score=False,
    )
    inicio = perf_counter()
    busqueda_artista.fit(X_desarrollo, y_desarrollo)
    segundos_busqueda = perf_counter() - inicio
    candidatos = pd.DataFrame(busqueda_artista.cv_results_)
    candidatos_con_artista_u3[ronda] = candidatos
    ganador = busqueda_artista.best_index_
    inicio = perf_counter()
    proba_interna = cross_val_predict(
        clone(busqueda_artista.best_estimator_), X_desarrollo, y_desarrollo,
        cv=particiones_internas, method="predict_proba", n_jobs=1,
    )[:, 1]
    if not np.isfinite(proba_interna).all() or not ((proba_interna >= 0) & (proba_interna <= 1)).all():
        raise ValueError("Las puntuaciones internas no son válidas.")
    corte, corte_viable, tabla_cortes = elegir_corte_interno_u3(y_desarrollo, proba_interna)
    segundos_corte = perf_counter() - inicio
    cortes_con_artista_u3[ronda] = tabla_cortes
    ganador_ajustado = busqueda_artista.best_estimator_
    codificador = ganador_ajustado.named_steps["artista"]
    if not codificador.ids_ajuste_.equals(X_desarrollo.index) or codificador.cobertura_cruzada_ != len(X_desarrollo):
        raise ValueError("El codificador no se ajustó exactamente con desarrollo y cobertura cruzada completa.")
    if codificador.claves_ajuste_.intersection(set(claves_train_u3.iloc[pos_externo])):
        raise ValueError("El mapa de artista contiene claves externas.")
    if not np.isclose(codificador.respaldo_, y_desarrollo.mean()):
        raise ValueError("El respaldo no coincide con desarrollo.")
    nombres_externos = codificador._artistas(X_externo)
    sin_conocidos = nombres_externos.map(lambda nombres: all(nombre not in codificador.mapa_ for nombre in nombres))
    valores_artista_externo = codificador.transform(X_externo)["artista_clase_1"]
    if sin_conocidos.any() and not np.allclose(valores_artista_externo.loc[sin_conocidos], codificador.respaldo_):
        raise ValueError("Los desconocidos no recibieron respaldo de desarrollo.")
    proba = ganador_ajustado.predict_proba(X_externo)[:, 1]
    if not np.isfinite(proba).all() or not ((proba >= 0) & (proba <= 1)).all():
        raise ValueError("Las puntuaciones externas no son válidas.")
    proba_externa_artista_u3.loc[y_externo.index] = proba
    pred_externa_artista_u3.loc[y_externo.index] = (proba >= corte).astype("int8")
    veces_externo_artista_u3.loc[y_externo.index] += 1
    filas_busqueda_artista_u3.append({
        "Externa": ronda, "AP media interna": float(candidatos.loc[ganador, "mean_test_ap"]),
        "DE AP interna": float(candidatos.loc[ganador, "std_test_ap"]),
        "Corte interno": corte, "Corte viable interno": corte_viable,
        "Parámetros": {k.replace("modelo__", ""): v for k, v in busqueda_artista.best_params_.items()},
        "Segundos búsqueda y refit": segundos_busqueda, "Segundos OOF y corte": segundos_corte,
    })
    filas_controles_artista_u3.append({
        "Externa": ronda, "IDs fit mapa": len(codificador.ids_ajuste_),
        "IDs externos": len(X_externo), "IDs con codificación cruzada": codificador.cobertura_cruzada_,
        "Claves externas en fit": 0, "IDs test": 0,
        "% consulta sin artistas conocidos": sin_conocidos.mean() * 100,
        "Respaldo de desarrollo": codificador.respaldo_,
        "Columnas finales": ganador_ajustado.named_steps["preparacion"].transform(codificador.transform(X_externo.iloc[:1])).shape[1],
    })
    for tipo, corte_evaluado in [("Fijo 0.50", 0.50), ("Elegido interno", corte)]:
        filas_rondas_artista_u3.append({"Externa": ronda, "Decisión": tipo, "Corte": corte_evaluado,
            **metricas_binarias_u3(y_externo, proba, corte_evaluado)})
    print(f"  Ronda {ronda}: AP interna {candidatos.loc[ganador, 'mean_test_ap']:.4f}; corte {corte:.2f}; búsqueda/refit {segundos_busqueda:.1f}s.", flush=True)
    del busqueda_artista, ganador_ajustado, codificador

if not veces_externo_artista_u3.eq(1).all() or proba_externa_artista_u3.isna().any() or pred_externa_artista_u3.isna().any():
    raise ValueError("OOF externa incompleta o repetida.")
resumen_busqueda_artista_u3 = pd.DataFrame(filas_busqueda_artista_u3)
controles_artista_anidado_u3 = pd.DataFrame(filas_controles_artista_u3)
metricas_rondas_artista_u3 = pd.DataFrame(filas_rondas_artista_u3)
display(resumen_busqueda_artista_u3.round(4))
display(controles_artista_anidado_u3.round(4))
display(metricas_rondas_artista_u3.round(4))
print("168 ajustes de Pipeline con artista; cobertura externa única de 71.728 IDs.")

Artista, ronda externa 1: búsqueda Random de 10 configuraciones.
  Ronda 1: AP interna 0.5936; corte 0.30; búsqueda/refit 73.3s.
Artista, ronda externa 2: búsqueda Random de 10 configuraciones.
  Ronda 2: AP interna 0.5875; corte 0.30; búsqueda/refit 80.7s.
Artista, ronda externa 3: búsqueda Random de 10 configuraciones.
  Ronda 3: AP interna 0.5813; corte 0.25; búsqueda/refit 76.5s.


,Externa,AP media interna,DE AP interna,Corte interno,Corte viable interno,Parámetros,Segundos búsqueda y refit,Segundos OOF y corte
0,1,0.5936,0.0074,0.30,True,"{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}",73.2627,10.1716
1,2,0.5875,0.0138,0.30,True,"{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}",80.6702,9.1485
2,3,0.5813,0.0132,0.25,True,"{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}",76.4622,9.3633


,Externa,IDs fit mapa,IDs externos,IDs con codificación cruzada,Claves externas en fit,IDs test,% consulta sin artistas conocidos,Respaldo de desarrollo,Columnas finales
0,1,47709,24019,47709,0,0,8.8680,0.1105,133
1,2,47900,23828,47900,0,0,8.9978,0.1095,133
2,3,47847,23881,47847,0,0,8.9485,0.1088,133


,Externa,Decisión,Corte,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI
0,1,Fijo 0.50,0.50,20764,664,1520,1071,0.6173,0.4134,0.4951,0.9091,0.9101,0.5964,False
1,1,Elegido interno,0.30,19643,1785,878,1713,0.4897,0.6611,0.5627,0.8891,0.9101,0.5964,True
2,2,Fijo 0.50,0.50,20649,563,1522,1094,0.6602,0.4182,0.5121,0.9125,0.9082,0.6076,False
3,2,Elegido interno,0.30,19402,1810,857,1759,0.4929,0.6724,0.5688,0.8881,0.9082,0.6076,True
4,3,Fijo 0.50,0.50,20640,587,1542,1112,0.6545,0.4190,0.5109,0.9108,0.9137,0.6137,False
5,3,Elegido interno,0.25,19187,2040,766,1888,0.4807,0.7114,0.5737,0.8825,0.9137,0.6137,True


168 ajustes de Pipeline con artista; cobertura externa única de 71.728 IDs.


### Explicación del procedimiento anidado con artista

1. `ParameterSampler` reproduce las diez configuraciones de Random y las compara con las listas anteriores; no se sortean candidatos adicionales para favorecer al artista.
2. Se reservan puntuaciones, decisiones y conteo externo por ID. El bucle reutiliza los mismos pares externos y genera los internos con los mismos datos, orden, clases, claves y semilla que sin artista. La columna auxiliar no cambia cómo se dividen los grupos.
3. Se verifica separación externa, ausencia de test, separación interna, ambas clases y cobertura interna única. Los tamaños se contrastan con el bloque anterior; la reproducción exacta procede además del mismo separador y entradas de agrupación.
4. `RandomizedSearchCV` recibe el Pipeline completo con el codificador. En **cada fit interno** se construye el artista cruzado con sus propias respuestas permitidas; la validación interna se transforma con el mapa de ese ajuste interno. No se reutiliza una característica calculada con todo desarrollo antes de dividir.
5. La búsqueda selecciona por AP media y reajusta el ganador con todo desarrollo. `cv_results_` conserva todos los candidatos y `best_index_` permite recuperar el AP ganador, sin usar las respuestas externas.
6. `cross_val_predict` clona ese Pipeline ganador y lo ajusta cinco veces. El codificador vuelve a hacer sus cinco grupos de codificación en cada fit. Se elige el corte con las probabilidades y etiquetas internas, conservando la regla de KPI y el respaldo 0,50 si no hay corte viable.
7. Se comprueba que el mapa del ganador final contiene exactamente IDs de desarrollo, ninguna clave externa, cobertura cruzada completa y proporción de respaldo correcta. La función interna del codificador comprueba exclusión de claves cada vez que se ajusta, también durante búsqueda.
8. Se identifica qué canciones externas no tienen ningún colaborador conocido. Se comprueba que su valor sea el respaldo de desarrollo. Un porcentaje pequeño de desconocidos no demuestra rendimiento en artistas totalmente nuevos; la evaluación seguirá mezclando artistas conocidos y desconocidos.
9. `predict_proba(X_externo)` transforma sin recibir sus respuestas. Se guardan puntuaciones y decisiones con el corte ya elegido. No se cambian parámetros o cortes según los KPI externos.
10. Las tablas registran parámetros, AP interna, cortes, tiempos y controles. La cantidad de columnas se comprueba transformando una fila externa, sin reajustar el preparador. La cobertura externa exige una predicción por cada uno de los 71.728 IDs.
11. Se calculan métricas externas al corte fijo e interno. Se eliminan modelos ajustados al terminar la ronda; no queda un modelo final de todo entrenamiento.

La columna de artista emplea respuestas del catálogo permitido, no fechas históricas verificadas. Por tanto, no equivale a disponer de información que necesariamente hubiera existido antes del lanzamiento de una canción. Esta limitación debe acompañar las conclusiones.

In [24]:
filas_comparacion_artista_u3 = []
matrices_comparacion_artista_u3 = {}
for variante, probabilidades, decisiones_internas in [
    ("Sin artista", proba_externa_u3["Random"], pred_externa_u3["Random"]),
    ("Con artista", proba_externa_artista_u3, pred_externa_artista_u3),
]:
    for tipo, decisiones in [
        ("Fijo 0.50", (probabilidades >= 0.50).astype("int8")),
        ("Elegido interno", decisiones_internas.astype("int64")),
    ]:
        tn, fp, fn, tp = confusion_matrix(y_train_u3, decisiones, labels=[0, 1]).ravel()
        precision = tp / (tp + fp) if tp + fp > 0 else np.nan
        recall = recall_score(y_train_u3, decisiones, zero_division=0)
        cumple_p = bool(pd.notna(precision) and precision >= META_PRECISION)
        cumple_r = bool(recall >= META_RECALL)
        filas_comparacion_artista_u3.append({
            "Variante": variante, "Decisión": tipo,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
            "Precisión": precision, "Recall": recall,
            "F1": f1_score(y_train_u3, decisiones, zero_division=0),
            "Accuracy": accuracy_score(y_train_u3, decisiones),
            "ROC-AUC": roc_auc_score(y_train_u3, probabilidades),
            "Average Precision": average_precision_score(y_train_u3, probabilidades),
            "KPI precisión": "Cumple" if cumple_p else "No cumple",
            "KPI recall": "Cumple" if cumple_r else "No cumple",
            "KPI simultáneos": "Cumple" if cumple_p and cumple_r else "No cumple",
        })
        matrices_comparacion_artista_u3[(variante, tipo)] = pd.DataFrame([[tn, fp], [fn, tp]], index=["Real 0", "Real 1"], columns=["Predice 0", "Predice 1"])
comparacion_anidada_artista_u3 = pd.DataFrame(filas_comparacion_artista_u3)
base_cambios = comparacion_anidada_artista_u3.set_index(["Variante", "Decisión"])
filas_cambios = []
for tipo in ["Fijo 0.50", "Elegido interno"]:
    sin = base_cambios.loc[("Sin artista", tipo)]
    con = base_cambios.loc[("Con artista", tipo)]
    filas_cambios.append({"Decisión": tipo,
        "Cambio precisión (pp)": (con["Precisión"] - sin["Precisión"]) * 100,
        "Cambio recall (pp)": (con["Recall"] - sin["Recall"]) * 100,
        "Cambio F1": con["F1"] - sin["F1"],
        "Cambio AUC": con["ROC-AUC"] - sin["ROC-AUC"],
        "Cambio AP": con["Average Precision"] - sin["Average Precision"],
        "Positivas detectadas adicionales": int(con["TP"] - sin["TP"]),
        "Cambio falsas alarmas": int(con["FP"] - sin["FP"]),
    })
cambios_por_artista_u3 = pd.DataFrame(filas_cambios)
display(comparacion_anidada_artista_u3.round(6))
display(cambios_por_artista_u3.round(4))
print("Matriz externa con artista y corte elegido internamente:")
display(matrices_comparacion_artista_u3[("Con artista", "Elegido interno")])

,Variante,Decisión,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,KPI precisión,KPI recall,KPI simultáneos
0,Sin artista,Fijo 0.50,63027,840,6315,1546,0.647946,0.196667,0.301747,0.900248,0.873512,0.484730,Cumple,No cumple,No cumple
1,Sin artista,Elegido interno,56916,6951,3033,4828,0.409882,0.614171,0.491650,0.860807,0.873512,0.484730,Cumple,Cumple,Cumple
2,Con artista,Fijo 0.50,62053,1814,4584,3277,0.643685,0.416868,0.506022,0.910802,0.910567,0.605341,Cumple,No cumple,No cumple
3,Con artista,Elegido interno,58232,5635,2501,5360,0.487494,0.681847,0.568519,0.886571,0.910567,0.605341,Cumple,Cumple,Cumple


,Decisión,Cambio precisión (pp),Cambio recall (pp),Cambio F1,Cambio AUC,Cambio AP,Positivas detectadas adicionales,Cambio falsas alarmas
0,Fijo 0.50,-0.4261,22.0201,0.2043,0.0371,0.1206,1731,974
1,Elegido interno,7.7612,6.7676,0.0769,0.0371,0.1206,532,-1316


Matriz externa con artista y corte elegido internamente:


,Predice 0,Predice 1
Real 0,58232,5635
Real 1,2501,5360


### Explicación de la comparación y los KPI

1. Se comparan las probabilidades externas Random sin artista del bloque 13 con las nuevas con artista, sobre exactamente los mismos IDs. No se reemplazan las primeras por métricas de cinco pliegues del modelo inicial.
2. Cada variante tiene dos filas: decisión al corte fijo 0,50 y decisión con el corte elegido en cada desarrollo. Para esta última se usan las decisiones guardadas: no se elige un corte global mirando todas las respuestas externas.
3. La matriz agrega verdaderos negativos, falsas alarmas, positivas omitidas y detectadas. Sus cuatro conteos deben sumar 71.728. Precisión y recall se contrastan por separado y simultáneamente con las metas originales.
4. AP y ROC-AUC se calculan con puntuaciones y por eso no cambian entre los dos cortes de la misma variante. Describen ordenamiento, no porcentaje de aciertos ni calibración. Se reportan también métricas por ronda porque las puntuaciones proceden de tres modelos distintos.
5. `set_index` permite seleccionar cada par variante/decisión. Restar con menos sin da el cambio. Para precisión y recall se multiplica por cien y se informa en puntos porcentuales; cambios F1/AUC/AP conservan su escala. Los conteos adicionales son diferencias absolutas de canciones, no porcentajes.
6. Un cambio positivo de recall indica más positivas recuperadas; un cambio positivo de falsas alarmas indica más negativas señaladas. La interpretación debe considerar ambos, sin decidir solo por accuracy.

Esta comparación evalúa procedimientos con parámetros seleccionados dentro de cada variante, no la causalidad del artista sobre la popularidad. Los KPI externos describen estas particiones de entrenamiento; si no se cumplen, se informa y explica sin redefinirlos. No se mezclan artistas nuevos y conocidos para afirmar una garantía sobre artistas nuevos.

### 14.4. Alcance y continuidad

Se añadió artista mediante una codificación supervisada cruzada dentro del Pipeline, manteniendo las versiones juntas en todos los niveles. Los controles de separación, respaldo y cobertura aportan evidencia metodológica; no constituyen una garantía absoluta sobre cualquier forma de fuga o sobre versiones que la clave no reconozca.

Las métricas externas son de comparación interna del proyecto, no de un test externo nuevo. La Unidad 1 ya exploró la base completa y el experimento continúa decisiones históricas. Elegir ahora una variante a partir de estas métricas no produce una confirmación independiente adicional.

No se hizo balanceo, Grid con artista, entrenamiento final sobre todo entrenamiento ni evaluación en test. El siguiente paso será revisar la elección provisional y sus errores, interpretabilidad y limitaciones antes de cerrar clasificación o continuar con clustering.

Referencias: [codificación de objetivo y cross-fitting, scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html#target-encoder) y [validación anidada](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html). La implementación adapta el principio de cross-fitting a grupos de versiones y colaboradores; no usa directamente TargetEncoder por filas.

### Lectura de los resultados ejecutados

Con artista y cortes internos se obtuvo precisión **48.75%**, recall **68.18%**, F1 **0.5685**, ROC-AUC **0.9106** y Average Precision **0.6053**. El KPI de precisión **cumple**, el de recall **cumple** y el cumplimiento simultáneo es **Cumple**. En 3 de tres rondas externas se cumplieron ambas metas. El cumplimiento global no garantiza cumplimiento en toda población futura.

Se detectaron **5,360** positivas de 7.861, se omitieron **2,501** y hubo **5,635** falsas alarmas. De cada cien canciones señaladas aproximadamente 48.7 son positivas reales; de cada cien positivas reales se detectan aproximadamente 68.2. Cumplir KPI no elimina errores ni demuestra calibración.

Sin artista, bajo el mismo procedimiento Random, se obtuvo precisión 40.99%, recall 61.42%, F1 0.4916, AP 0.4847 y AUC 0.8735. El cambio de F1 es +0.0769, el de AP +0.1206 y el de AUC +0.0371. Estas diferencias describen la comparación, sin una prueba de superioridad estadística ni evidencia causal.

Los cortes internos con artista por ronda fueron 0.30, 0.30, 0.25; se eligieron antes de usar las respuestas externas. No se impone el 0,20 sin artista ni se selecciona uno nuevo usando el resumen externo. La tabla al corte fijo 0,50 permite observar el resultado sin cambiar simultáneamente la regla de decisión.


**Parámetros seleccionados con artista:** en las tres rondas se eligieron 200 árboles, profundidad 6, tasa 0,10 y subsample 1,0. Sin artista se había elegido subsample 0,8. Por eso la comparación principal es entre procedimientos optimizados; no atribuimos toda la diferencia a añadir una columna con los demás parámetros congelados. Son los mejores candidatos encontrados en el espacio probado, no un óptimo global demostrado.

**Tamaño del cambio con cortes internos:** se detectaron 532 positivas adicionales y hubo 1.316 falsas alarmas menos. Precisión subió aproximadamente 7,76 puntos porcentuales y recall 6,77 puntos. Al corte fijo 0,50, el recall con artista fue 41,69% frente a 19,67% sin artista: la señal mejora, pero ambas variantes siguen incumpliendo el recall mínimo a ese corte. Con los cortes elegidos internamente, la variante con artista cumple ambos KPI en las tres rondas externas.

**Artistas desconocidos:** entre 8,87% y 9,00% de las canciones externas no tenían ningún colaborador conocido en su desarrollo y recibieron el respaldo permitido. No significa que el modelo se equivocara en ese porcentaje; es un control de cobertura de información. No se evaluó aquí por separado el desempeño sobre ese subgrupo.


## 15. Random Forest inicial con artista

### 15.1. Objetivo y alcance de la comparación

Incorporamos `RandomForestClassifier` como segundo ensamble de la Unidad 3. A diferencia del boosting secuencial de XGBoost, Random Forest aprende árboles con muestras bootstrap y selección aleatoria de características en cada división. En scikit-learn combina sus probabilidades mediante un promedio; después aplicamos el corte fijado para obtener la decisión.

Se conservan las 71.728 canciones de entrenamiento, la clase positiva popularidad ≥60, precisión mínima 30% y recall mínimo 60%. Se utiliza el mismo Pipeline de artista cruzado y preparación musical, y las **mismas tres particiones externas** del bloque 14. Se trata de tres ajustes de una configuración fija; no hay búsqueda interna de hiperparámetros ni de corte en este bloque.

**Corte inicial: 0,50, fijado antes de ejecutar.** Se mostrará Random Forest junto con XGBoost **optimizado** con artista, al corte 0,50 y con sus cortes elegidos internamente. La comparación describe los procedimientos disponibles, pero no prueba superioridad de un algoritmo con igual esfuerzo de optimización: Random Forest aún no está optimizado. La comparación definitiva de ensambles corresponde al bloque posterior de hiperparámetros. No se presenta una ejecución inexistente de XGBoost inicial con artista.

Los resultados de XGBoost se recuperan de las variables calculadas en el bloque 14, sobre exactamente los mismos IDs y rondas. No se utiliza test ni se repiten las búsquedas de XGBoost para elegir otra favorable. La referencia mayoritaria externa aporta contexto sobre desbalance, no basta por sí sola para demostrar mejora frente a un clasificador competente.

### 15.2. Configuración fijada y su justificación

| Hiperparámetro | Valor inicial | Decisión |
|---|---|---|
| n_estimators | 100 | Bosque acotado para un punto de partida reproducible. No son 100 rondas de validación. |
| max_depth | 10 | Limitar la complejidad de cada árbol antes de optimizar. No se afirma que sea el mejor valor. |
| min_samples_leaf | 3 | Evitar hojas basadas en una sola observación. |
| min_samples_split | 2 | Condición inicial mínima para intentar dividir; también deben cumplirse las restricciones de hoja y profundidad. |
| max_features | sqrt | Considerar aproximadamente la raíz del número de columnas en cada división; favorece diversidad entre árboles. |
| criterion | gini | Medir mezcla de clases en las divisiones; no es la métrica externa del proyecto. |
| bootstrap | True | Cada árbol recibe una muestra con reemplazo del ajuste permitido. |
| class_weight | None | Sin ponderación adicional de clases para este punto de partida. |
| oob_score | False | No usar evaluación out-of-bag por filas; mantenemos evaluación agrupada por versiones. |
| random_state | 42 | Reproducir la aleatoriedad con los mismos datos y versión. |
| n_jobs | 1 | Controlar recursos de ejecución. |

Son decisiones de configuración inicial, no valores exigidos por la rúbrica. El bloque posterior probará otras configuraciones. No se aplican SMOTE, sobremuestreo ni submuestreo de balanceo. El bootstrap propio del bosque es un mecanismo del ensamble, no una estrategia que iguale las clases.

Con 133 columnas transformadas, `sqrt` corresponde aproximadamente a once características candidatas por división. Esto puede hacer que diferentes árboles utilicen variables distintas. No equivale a conservar permanentemente once columnas ni garantiza una mejora.

La codificación de artista mantiene cinco grupos de claves dentro de cada ajuste para excluir la propia respuesta y sus versiones. Esos cinco grupos calculan una característica; no entrenan cinco bosques adicionales.

In [25]:
from sklearn.ensemble import RandomForestClassifier

UMBRAL_RF_INICIAL_U3 = 0.50
modelo_rf_inicial_u3 = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=3,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    oob_score=False,
    random_state=SEMILLA,
    n_jobs=1,
)
flujo_rf_inicial_artista_u3 = Pipeline([
    ("artista", ArtistaCruzadoU3(suavizado=10.0, grupos_codigo=5, random_state=SEMILLA)),
    ("preparacion", clone(preparador_con_artista_u3)),
    ("modelo", modelo_rf_inicial_u3),
])
print(f"scikit-learn: {sklearn.__version__}")
print(f"Random Forest inicial: 100 árboles, profundidad 10, hoja mínima 3; umbral {UMBRAL_RF_INICIAL_U3:.2f}.")
print("Pipeline sin ajustar: artista cruzado, preparación y bosque.")

scikit-learn: 1.6.1
Random Forest inicial: 100 árboles, profundidad 10, hoja mínima 3; umbral 0.50.
Pipeline sin ajustar: artista cruzado, preparación y bosque.


### Explicación de la definición del modelo

1. La importación incorpora `RandomForestClassifier`; no entrena por sí misma.
2. `UMBRAL_RF_INICIAL_U3 = 0.50` define la regla de decisión antes de ver sus resultados. El valor 60 sigue definiendo la clase real y no se cambia.
3. El constructor guarda la configuración de la tabla. `n_estimators` controla cantidad de árboles; profundidad y mínimos controlan divisiones y hojas. `sqrt` actúa durante la selección de variables de cada división, no sobre la lista original de columnas.
4. `criterion="gini"` dirige las divisiones buscando nodos con clases menos mezcladas. Precisión, recall, F1, AP y AUC se calcularán después sobre validación.
5. `bootstrap=True` puede repetir observaciones dentro del ajuste de cada árbol. Toda esa aleatoriedad queda dentro del ajuste permitido; no añade filas del pliegue externo.
6. `class_weight=None` conserva pesos sin modificación adicional. `oob_score=False` evita sustituir la evaluación agrupada por una evaluación de filas fuera de las muestras bootstrap, que no controla versiones mediante la clave.
7. `random_state` y `n_jobs` fijan semilla y paralelismo. No probamos distintas semillas para seleccionar resultados.
8. `Pipeline` encadena artista cruzado, preparación y bosque. El preparador se clona sin aprendizaje previo; después se clonará todo el flujo en cada ronda. La clave solo llega al codificador y es retirada antes de los árboles.
9. `print` registra versión y decisiones. No es evidencia de rendimiento; el modelo todavía está sin ajustar.

In [26]:
probabilidades_rf_inicial_artista_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64", name="probabilidad_rf_inicial")
predicciones_rf_inicial_artista_u3 = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64", name="prediccion_rf_inicial")
visitas_rf_inicial_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
filas_rf_inicial_u3 = []
filas_controles_rf_u3 = []

for ronda, (pos_ajuste, pos_validacion) in enumerate(particiones_externas_u3, start=1):
    X_ajuste = X_train_con_artista_u3.iloc[pos_ajuste]
    X_validacion = X_train_con_artista_u3.iloc[pos_validacion]
    y_ajuste = y_train_u3.iloc[pos_ajuste]
    y_validacion = y_train_u3.iloc[pos_validacion]
    if not X_ajuste.index.equals(y_ajuste.index) or not X_validacion.index.equals(y_validacion.index):
        raise ValueError("Entradas y respuestas no están alineadas.")
    claves_ajuste = set(claves_train_u3.iloc[pos_ajuste])
    claves_validacion = set(claves_train_u3.iloc[pos_validacion])
    if claves_ajuste.intersection(claves_validacion) or len(X_ajuste.index.intersection(X_validacion.index)):
        raise ValueError("Hay IDs o versiones compartidos entre ajuste y validación.")
    if len(X_ajuste.index.union(X_validacion.index)) != len(y_train_u3) or len(X_ajuste.index.union(X_validacion.index).intersection(ids_test_u2)):
        raise ValueError("La cobertura no es entrenamiento completo o incluye test.")
    if set(y_ajuste.unique()) != {0, 1} or set(y_validacion.unique()) != {0, 1}:
        raise ValueError("Falta una clase en una partición.")
    if X_ajuste[variables_numericas_u3].isna().all().any():
        raise ValueError("Hay una variable numérica completamente ausente en ajuste.")
    flujo_rf_ronda = clone(flujo_rf_inicial_artista_u3)
    inicio = perf_counter()
    flujo_rf_ronda.fit(X_ajuste, y_ajuste)
    segundos_fit = perf_counter() - inicio
    codificador_rf = flujo_rf_ronda.named_steps["artista"]
    preparador_rf = flujo_rf_ronda.named_steps["preparacion"]
    bosque_rf = flujo_rf_ronda.named_steps["modelo"]
    if not codificador_rf.ids_ajuste_.equals(X_ajuste.index) or codificador_rf.cobertura_cruzada_ != len(X_ajuste):
        raise ValueError("El artista no usó exactamente el ajuste con codificación cruzada completa.")
    if codificador_rf.claves_ajuste_.intersection(claves_validacion):
        raise ValueError("El mapa de artista contiene claves de validación.")
    if not np.isclose(codificador_rf.respaldo_, y_ajuste.mean()):
        raise ValueError("El respaldo no proviene del ajuste permitido.")
    medianas_audio = preparador_rf.named_transformers_["numericas"].statistics_[:len(variables_numericas_u3)]
    if not np.allclose(medianas_audio, X_ajuste[variables_numericas_u3].median().to_numpy(dtype=float)):
        raise ValueError("Las medianas no corresponden al ajuste de esta ronda.")
    if len(bosque_rf.estimators_) != 100:
        raise ValueError("La cantidad de árboles no coincide con la configuración.")
    columna_positiva = int(np.flatnonzero(bosque_rf.classes_ == 1)[0])
    proba = flujo_rf_ronda.predict_proba(X_validacion)[:, columna_positiva]
    if not np.isfinite(proba).all() or not ((proba >= 0) & (proba <= 1)).all():
        raise ValueError("Las puntuaciones no son válidas.")
    decisiones = (proba >= UMBRAL_RF_INICIAL_U3).astype("int8")
    probabilidades_rf_inicial_artista_u3.loc[y_validacion.index] = proba
    predicciones_rf_inicial_artista_u3.loc[y_validacion.index] = decisiones
    visitas_rf_inicial_u3.loc[y_validacion.index] += 1
    nombres_consulta = codificador_rf._artistas(X_validacion)
    sin_conocidos = nombres_consulta.map(lambda nombres: all(nombre not in codificador_rf.mapa_ for nombre in nombres))
    valores_consulta = codificador_rf.transform(X_validacion)["artista_clase_1"]
    if sin_conocidos.any() and not np.allclose(valores_consulta.loc[sin_conocidos], codificador_rf.respaldo_):
        raise ValueError("Un artista desconocido no recibió el respaldo permitido.")
    filas_rf_inicial_u3.append({"Ronda": ronda, "Corte": UMBRAL_RF_INICIAL_U3,
        **metricas_binarias_u3(y_validacion, proba, UMBRAL_RF_INICIAL_U3), "Segundos fit": segundos_fit})
    filas_controles_rf_u3.append({
        "Ronda": ronda, "IDs ajuste": len(X_ajuste), "IDs validación": len(X_validacion),
        "IDs codificados con exclusión": codificador_rf.cobertura_cruzada_,
        "Claves compartidas": 0, "IDs test": 0,
        "Columnas de los árboles": bosque_rf.n_features_in_, "Árboles": len(bosque_rf.estimators_),
        "% sin artistas conocidos": sin_conocidos.mean() * 100,
        "Respaldo permitido": codificador_rf.respaldo_, "Medianas verificadas": True,
    })
    print(f"Ronda {ronda} completada: {len(X_ajuste):,} para ajustar; {len(X_validacion):,} para validar; {segundos_fit:.2f}s de fit.", flush=True)
    del flujo_rf_ronda, codificador_rf, preparador_rf, bosque_rf

if not visitas_rf_inicial_u3.eq(1).all() or probabilidades_rf_inicial_artista_u3.isna().any() or predicciones_rf_inicial_artista_u3.isna().any():
    raise ValueError("Faltan predicciones OOF o algún ID se evaluó más de una vez.")
resultados_rondas_rf_inicial_u3 = pd.DataFrame(filas_rf_inicial_u3).set_index("Ronda")
controles_rf_inicial_u3 = pd.DataFrame(filas_controles_rf_u3).set_index("Ronda")
display(controles_rf_inicial_u3.round(4))
display(resultados_rondas_rf_inicial_u3.round(6))
print(f"Cobertura OOF: {visitas_rf_inicial_u3.eq(1).sum():,} IDs evaluados exactamente una vez.")

Ronda 1 completada: 47,709 para ajustar; 24,019 para validar; 4.00s de fit.
Ronda 2 completada: 47,900 para ajustar; 23,828 para validar; 3.91s de fit.
Ronda 3 completada: 47,847 para ajustar; 23,881 para validar; 3.86s de fit.


,IDs ajuste,IDs validación,IDs codificados con exclusión,Claves compartidas,IDs test,Columnas de los árboles,Árboles,% sin artistas conocidos,Respaldo permitido,Medianas verificadas
Ronda,,,,,,,,,,
1,47709,24019,47709,0,0,133,100,8.8680,0.1105,True
2,47900,23828,47900,0,0,133,100,8.9978,0.1095,True
3,47847,23881,47847,0,0,133,100,8.9485,0.1088,True


,Corte,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI,Segundos fit
Ronda,,,,,,,,,,,,,
1,0.5,21400,28,2350,241,0.895911,0.093014,0.168531,0.900995,0.893059,0.540900,False,3.998210
2,0.5,21172,40,2341,275,0.873016,0.105122,0.187649,0.900076,0.894258,0.558942,False,3.912008
3,0.5,21203,24,2395,259,0.915194,0.097589,0.176370,0.898706,0.900535,0.575073,False,3.856338


Cobertura OOF: 71,728 IDs evaluados exactamente una vez.


### Explicación del ajuste y los controles por ronda

1. Se crean puntuaciones y decisiones inicialmente faltantes, un contador OOF en cero y listas para guardar evidencia. Los índices son los IDs de entrenamiento.
2. El bucle utiliza **los mismos tres pares externos guardados**, sin crear otra partición. `.iloc` selecciona filas por posición y se exige alineación con sus respuestas.
3. Se comprueban IDs y claves separados, cobertura de todo entrenamiento, cero IDs de test y ambas clases presentes. Una columna totalmente faltante en ajuste detiene el proceso porque no tiene mediana aprendible.
4. `clone` crea un flujo limpio por ronda. `fit` llama al codificador cruzado con las respuestas permitidas, aprende la preparación y entrena los cien árboles. No se calculó antes una columna global de artista.
5. `perf_counter` mide preparación y entrenamiento juntos; no incluye toda la predicción, controles o cálculo de métricas. No se comparará este tiempo con el de una búsqueda que hizo decenas de ajustes.
6. Los controles del codificador exigen IDs exactos de ajuste, cobertura cruzada completa, ninguna clave de validación y respaldo igual a la proporción positiva de ajuste. Sus grupos internos ya comprueban exclusión de la propia clave.
7. `statistics_` recupera las medianas aprendidas. Se comparan las diez primeras —duración y audio— con las del ajuste. La característica de artista también está en el bloque numérico, pero proviene de su codificación cruzada y no de un mapa global.
8. `estimators_` contiene los árboles entrenados. Se exige que sean cien. `classes_` identifica cuál columna de `predict_proba` corresponde a la clase positiva.
9. `predict_proba(X_validacion)` transforma y predice sin recibir respuestas de validación. Las puntuaciones deben ser finitas y estar entre cero y uno; no se consideran por eso probabilidades calibradas.
10. El corte fijo convierte puntuaciones en decisiones. `.loc` guarda los resultados en sus IDs y suma una visita OOF. Cada canción se evalúa con un flujo que no ajustó con ella ni con las versiones reconocidas por su clave.
11. Se comprueba también el respaldo de artistas desconocidos. Ese porcentaje informa cobertura de nombres, no porcentaje de errores ni una evaluación separada de artistas nuevos.
12. `metricas_binarias_u3` calcula matriz y métricas con las respuestas de validación **después de predecir**. Las tablas muestran controles y resultados por ronda; `round` solo cambia su presentación.
13. `del` retira los modelos al terminar cada ronda. Ninguno se presenta como un modelo final ajustado con todo entrenamiento. El control final exige exactamente una OOF por ID.

Los cinco grupos del codificador permanecen dentro de cada ajuste externo. No se realiza búsqueda en los cinco pliegues internos del protocolo anidado porque esta etapa evalúa una configuración fija. Se usarán al optimizar Random Forest en el bloque de hiperparámetros.

In [27]:
metricas_rf_inicial_u3 = metricas_binarias_u3(y_train_u3, probabilidades_rf_inicial_artista_u3, UMBRAL_RF_INICIAL_U3)
matriz_rf_inicial_u3 = pd.DataFrame(
    [[metricas_rf_inicial_u3["TN"], metricas_rf_inicial_u3["FP"]],
     [metricas_rf_inicial_u3["FN"], metricas_rf_inicial_u3["TP"]]],
    index=["Real 0: popularidad <60", "Real 1: popularidad >=60"],
    columns=["Predice 0", "Predice 1"],
)
precision_rf_u3 = metricas_rf_inicial_u3["Precisión"]
recall_rf_u3 = metricas_rf_inicial_u3["Recall"]
kpi_rf_inicial_u3 = pd.DataFrame([
    {"KPI": "Precisión >=30%", "Resultado": precision_rf_u3,
     "Estado": "Cumple" if pd.notna(precision_rf_u3) and precision_rf_u3 >= META_PRECISION else "No cumple"},
    {"KPI": "Recall >=60%", "Resultado": recall_rf_u3,
     "Estado": "Cumple" if recall_rf_u3 >= META_RECALL else "No cumple"},
    {"KPI": "Ambas metas simultáneas", "Resultado": np.nan,
     "Estado": "Cumple" if metricas_rf_inicial_u3["Cumple ambos KPI"] else "No cumple"},
])
xgb_fijo_comparable = comparacion_anidada_artista_u3.loc[
    (comparacion_anidada_artista_u3["Variante"] == "Con artista") &
    (comparacion_anidada_artista_u3["Decisión"] == "Fijo 0.50")
].iloc[0]
xgb_corte_comparable = comparacion_anidada_artista_u3.loc[
    (comparacion_anidada_artista_u3["Variante"] == "Con artista") &
    (comparacion_anidada_artista_u3["Decisión"] == "Elegido interno")
].iloc[0]
columnas_metricas_comparar = ["Precisión", "Recall", "F1", "Accuracy", "ROC-AUC", "Average Precision"]
comparacion_rf_xgb_inicial_u3 = pd.DataFrame([
    {"Procedimiento": "Random Forest inicial con artista", "Decisión": "Fijo 0.50",
     **{m: metricas_rf_inicial_u3[m] for m in columnas_metricas_comparar},
     "KPI simultáneos": "Cumple" if metricas_rf_inicial_u3["Cumple ambos KPI"] else "No cumple"},
    {"Procedimiento": "XGBoost optimizado con artista", "Decisión": "Fijo 0.50",
     **{m: xgb_fijo_comparable[m] for m in columnas_metricas_comparar}, "KPI simultáneos": xgb_fijo_comparable["KPI simultáneos"]},
    {"Procedimiento": "XGBoost optimizado con artista", "Decisión": "Corte interno por ronda",
     **{m: xgb_corte_comparable[m] for m in columnas_metricas_comparar}, "KPI simultáneos": xgb_corte_comparable["KPI simultáneos"]},
])
display(matriz_rf_inicial_u3)
display(kpi_rf_inicial_u3.round({"Resultado": 6}))
display(comparacion_rf_xgb_inicial_u3.round(6))
print(f"Random Forest detecta {metricas_rf_inicial_u3['TP']:,} de {int(y_train_u3.sum()):,} positivas.")
print(f"Omite {metricas_rf_inicial_u3['FN']:,} positivas y genera {metricas_rf_inicial_u3['FP']:,} falsas alarmas.")
print(f"Recall al corte fijo: {recall_rf_u3:.2%}; accuracy: {metricas_rf_inicial_u3['Accuracy']:.2%}.")

,Predice 0,Predice 1
Real 0: popularidad <60,63775,92
Real 1: popularidad >=60,7086,775


,KPI,Resultado,Estado
0,Precisión >=30%,0.893887,Cumple
1,Recall >=60%,0.098588,No cumple
2,Ambas metas simultáneas,NaN,No cumple


,Procedimiento,Decisión,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,KPI simultáneos
0,Random Forest inicial con artista,Fijo 0.50,0.893887,0.098588,0.177589,0.899928,0.895779,0.557724,No cumple
1,XGBoost optimizado con artista,Fijo 0.50,0.643685,0.416868,0.506022,0.910802,0.910567,0.605341,No cumple
2,XGBoost optimizado con artista,Corte interno por ronda,0.487494,0.681847,0.568519,0.886571,0.910567,0.605341,Cumple


Random Forest detecta 775 de 7,861 positivas.
Omite 7,086 positivas y genera 92 falsas alarmas.
Recall al corte fijo: 9.86%; accuracy: 89.99%.


### Explicación de las métricas globales, comparación y KPI

1. `metricas_binarias_u3` reúne las 71.728 predicciones OOF de Random Forest a 0,50. Las métricas globales no son necesariamente iguales al promedio simple de las tres rondas.
2. La matriz tiene clases reales en filas y predicciones en columnas. TN son negativas descartadas, FP falsas alarmas, FN positivas omitidas y TP positivas detectadas. Los cuatro conteos deben sumar todo entrenamiento.
3. La tabla de KPI compara precisión y recall con sus metas fijadas. Una precisión indefinida no cumple. La fila de cumplimiento simultáneo no tiene resultado numérico propio; se interpreta su Estado.
4. `.loc` recupera las filas ya ejecutadas de XGBoost con artista: una al corte fijo y otra con cortes internos. No se reajusta XGBoost ni se elige un corte para Random Forest con esas cifras.
5. La lista de métricas evita comparar columnas incompatibles. Los diccionarios crean una fila por procedimiento y `round(6)` limita decimales mostrados.
6. Precisión representa cuántas señaladas son realmente positivas. Recall representa cuántas positivas reales recuperamos. F1 combina ambas; accuracy cuenta aciertos de ambas clases y puede ocultar baja detección positiva.
7. ROC-AUC y Average Precision evalúan ordenamiento con puntuaciones y no se limitan al corte 0,50. AUC 0,5 y AP aproximadamente 0,1096 son referencias de puntuación constante en este entrenamiento, no metas universales.
8. Comparar ambos a 0,50 iguala el corte y conserva particiones y entradas. Todavía difiere la etapa de optimización. La fila de XGBoost con corte interno muestra el procedimiento más completo ya disponible, no un corte que Random Forest deba heredar.
9. Los KPI se informan tal como resulten. Si el recall no alcanza 60%, una accuracy alta o AUC favorable no lo reemplaza. Un incumplimiento inicial no demuestra que todo Random Forest sea inadecuado; parámetros y corte aún están por estudiar.

Si la precisión es alta y el recall bajo, se señalan pocas canciones y se dejan positivas fuera. El desbalance y un corte exigente son posibles factores; no se afirma una causa única comprobada. Tampoco se promete que optimizar vaya a resolverlo: habrá que medirlo.

### 15.3. Cierre de la etapa inicial

Se evaluó una configuración inicial de Random Forest con artista en las mismas tres particiones externas. La preparación y codificación se aprendieron en ajuste; cada ID recibió una predicción OOF. No hubo búsqueda, balanceo adicional, elección de umbral, evaluación en test ni entrenamiento final sobre todo entrenamiento.

Los resultados permiten documentar el punto de partida antes del bloque de hiperparámetros. El orden acordado continúa con **LightGBM inicial con artista** y después la optimización y comparación de ensambles. No se declarará ganador definitivo enfrentando un Random Forest inicial con un XGBoost ya optimizado.

La comparación conserva las limitaciones de la clave para reconocer versiones y del artista como información del catálogo permitido. No demuestra causalidad ni disponibilidad temporal previa al lanzamiento.

Fuente: [RandomForestClassifier, scikit-learn 1.6.1](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

### Lectura de los resultados ejecutados

Random Forest inicial al corte 0,50 obtuvo precisión **89.39%**, recall **9.86%**, F1 **0.1776**, accuracy **89.99%**, ROC-AUC **0.8958** y Average Precision **0.5577**. El cumplimiento simultáneo de KPI es **No cumple**.

Detectó **775** de 7.861 positivas y omitió **7,086**. Hubo **92** falsas alarmas y **63,775** negativas correctamente descartadas. De cada cien canciones señaladas, aproximadamente 89.4 son positivas reales; de cada cien positivas reales recupera aproximadamente 9.9.

La precisión cumple el mínimo de 30% y el recall no cumple el mínimo de 60%. No se cambia el corte para modificar este resultado. Las métricas describen este punto de partida; no representan el resultado de Random Forest optimizado.


**Por qué no cumple el objetivo:** solo señaló 867 de las 71.728 canciones (aproximadamente 1,21%). De ellas, 775 fueron positivas reales y 92 falsas alarmas, lo que explica la precisión de 89,39%. Sin embargo, omitió 7.086 de las 7.861 positivas, aproximadamente el 90,14%. El recall de 9,86% queda muy por debajo de la meta de 60%, en las tres rondas. La accuracy de 89,99% y AUC de 0,8958 no sustituyen ese requisito.

**Comparación al mismo corte:** XGBoost optimizado con artista a 0,50 obtuvo precisión 64,37%, recall 41,69% y F1 0,5060; Random Forest inicial tiene mayor precisión entre las pocas canciones que señala, pero recupera muchas menos positivas. Ambos fallan el recall mínimo a ese corte. XGBoost con cortes internos alcanzó recall 68,18% y sí cumplió ambos KPI. Esa última fila incluye selección de parámetros y de corte; no sería correcto atribuir toda su ventaja al nombre del algoritmo. Random Forest todavía debe pasar por el bloque de optimización.


## 16. LightGBM inicial con artista

### 16.1. Objetivo y comparación prevista

Se incorpora `LGBMClassifier` como tercer ensamble de clasificación. LightGBM construye árboles de boosting de forma secuencial y utiliza histogramas. Su crecimiento por hojas prioriza divisiones que mejoran la función objetivo; cantidad de hojas, profundidad y mínimo de observaciones limitan su complejidad.

Se mantienen las 71.728 canciones de entrenamiento, clases, KPI, características musicales, codificación cruzada del artista y **las mismas tres particiones externas agrupadas**. No se utiliza test. Se fijan una configuración y el corte **0,50 antes de ejecutar**; en esta etapa no se eligen hiperparámetros ni umbrales con los resultados.

La comparación principal de configuraciones iniciales será LightGBM frente a Random Forest, ambos con artista y a 0,50. También se incluirá XGBoost optimizado con artista al corte fijo y con cortes internos como referencia del procedimiento ya desarrollado. **XGBoost está en una etapa de optimización distinta**: no se atribuirá superioridad definitiva al algoritmo a partir de esa comparación. La evaluación conjunta después de optimizar los tres corresponde al último bloque de hiperparámetros.

**Dependencias:** la ejecución verificada usa LightGBM 4.6.0 y scikit-learn 1.6.1. Si falta LightGBM puede instalarse en una celda auxiliar con `%pip install lightgbm==4.6.0`; después puede ser necesario reiniciar el kernel y ejecutar en orden. En macOS puede requerir OpenMP. Las celdas de modelamiento no cambian la instalación. El paquete utilizado procede de [PyPI, versión 4.6.0](https://pypi.org/project/lightgbm/4.6.0/).

### 16.2. Configuración inicial y motivos

| Hiperparámetro | Valor | Interpretación y decisión |
|---|---|---|
| boosting_type | gbdt | Boosting tradicional de árboles. |
| objective | binary | Aprender la clase binaria 0/1. |
| n_estimators | 100 | Hasta 100 etapas fijadas, sin early stopping elegido con validación. |
| learning_rate | 0,10 | Magnitud de la contribución de cada nueva etapa. |
| num_leaves | 31 | Máximo de hojas por árbol; no exige que todos tengan exactamente 31. |
| max_depth | 6 | Limitar además la profundidad. 31 hojas es compatible con el límite de 64 hojas de un árbol binario de profundidad 6. |
| min_child_samples | 20 | Mínimo de observaciones por hoja según el criterio de LightGBM; actúa como restricción adicional. |
| subsample / subsample_freq | 1,0 / 0 | Sin submuestreo de filas para bagging en este punto de partida. La frecuencia 0 lo desactiva. |
| colsample_bytree | 1,0 | Considerar todas las columnas por árbol. |
| reg_alpha / reg_lambda | 0 / 0 | Sin penalización L1/L2 adicional explícita en esta configuración; profundidad, hojas y mínimos también restringen complejidad. |
| class_weight / scale_pos_weight / is_unbalance | None / 1 / False | Sin ponderación adicional ni balanceo automático. |
| random_state / n_jobs | 42 / 1 | Reproducibilidad y recursos controlados. |

Son valores iniciales justificados, no una configuración óptima ni un requisito numérico de la pauta. Cien árboles no implican igual capacidad que cien árboles de otro algoritmo. No se pretende igualar artificialmente parámetros con significados diferentes.

Se añaden `deterministic=True` y `force_col_wise=True` para controlar la ejecución en CPU, sin probar alternativas según las métricas. `verbosity=-1` reduce mensajes internos; se registrará el número real de iteraciones y se validarán las predicciones para no asumir que entrenar es suficiente.

No se usa `eval_set` ni callbacks de parada temprana. El número real de etapas puede ser menor si el algoritmo no puede continuar con divisiones útiles; se informa el valor observado, sin presentarlo como una optimización de validación.

In [28]:
import lightgbm
from lightgbm import LGBMClassifier

UMBRAL_LGB_INICIAL_U3 = 0.50
modelo_lgb_inicial_u3 = LGBMClassifier(
    boosting_type="gbdt",
    objective="binary",
    n_estimators=100,
    learning_rate=0.10,
    num_leaves=31,
    max_depth=6,
    min_child_samples=20,
    subsample=1.0,
    subsample_freq=0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    class_weight=None,
    scale_pos_weight=1.0,
    is_unbalance=False,
    random_state=SEMILLA,
    n_jobs=1,
    deterministic=True,
    force_col_wise=True,
    verbosity=-1,
)
preparador_lgb_inicial_u3 = clone(preparador_con_artista_u3).set_output(transform="pandas")
flujo_lgb_inicial_artista_u3 = Pipeline([
    ("artista", ArtistaCruzadoU3(suavizado=10.0, grupos_codigo=5, random_state=SEMILLA)),
    ("preparacion", preparador_lgb_inicial_u3),
    ("modelo", modelo_lgb_inicial_u3),
])
print(f"LightGBM: {lightgbm.__version__}; scikit-learn: {sklearn.__version__}")
print(f"Configuración inicial: 100 etapas, hasta 31 hojas, profundidad 6; corte {UMBRAL_LGB_INICIAL_U3:.2f}.")
print("Pipeline definido, aún sin ajustar; artista cruzado dentro de cada fit.")

LightGBM: 4.6.0; scikit-learn: 1.6.1
Configuración inicial: 100 etapas, hasta 31 hojas, profundidad 6; corte 0.50.
Pipeline definido, aún sin ajustar; artista cruzado dentro de cada fit.


### Explicación de las instrucciones de configuración

1. `import lightgbm` permite registrar la versión y `LGBMClassifier` importa el clasificador. Ninguna de esas líneas entrena un modelo.
2. `UMBRAL_LGB_INICIAL_U3 = 0.50` fija la decisión positiva cuando la puntuación sea ≥0,50. Popularidad ≥60 sigue definiendo la respuesta real, en otra escala.
3. El constructor declara el tipo de boosting y objetivo binario. Las siguientes líneas fijan cantidad de etapas, tasa, hojas, profundidad y mínimo de observaciones, con los motivos de la tabla.
4. `subsample=1.0` y `subsample_freq=0` dejan sin bagging de filas este punto de partida. Reducir solo subsample sin activar una frecuencia no bastaría para habilitar ese mecanismo en LightGBM; lo tendremos en cuenta al definir búsquedas.
5. `colsample_bytree=1.0` conserva todas las características. Los parámetros de regularización y ponderación quedan explícitos para no confundir esta ejecución con una variante balanceada.
6. Semilla e hilo de ejecución controlan aleatoriedad y recursos. Las dos opciones de ejecución determinista afectan el cómputo; no son hiperparámetros elegidos para maximizar resultados.
7. El preparador se clona sin aprendizaje previo. `.set_output(transform="pandas")` conserva nombres de columnas al transformar, para comprobar que ajuste y predicción usen la misma representación. No modifica la imputación, categorías, géneros ni sus valores; cambia el contenedor de salida de matriz a tabla con nombres.
8. El Pipeline aplica el codificador cruzado, la preparación y el clasificador en ese orden. La clave y el título no llegan a los árboles; el codificador retira la clave y aporta una característica numérica de artista.
9. Los `print` documentan la configuración, no resultados predictivos. En cada ronda se clonará de nuevo todo el Pipeline para impedir reutilizar mapas, medianas o árboles de otra ronda.

In [29]:
probabilidades_lgb_inicial_artista_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64", name="probabilidad_lgb_inicial")
predicciones_lgb_inicial_artista_u3 = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64", name="prediccion_lgb_inicial")
visitas_lgb_inicial_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
filas_lgb_inicial_u3 = []
filas_controles_lgb_u3 = []

for ronda, (pos_ajuste, pos_validacion) in enumerate(particiones_externas_u3, start=1):
    X_ajuste = X_train_con_artista_u3.iloc[pos_ajuste]
    X_validacion = X_train_con_artista_u3.iloc[pos_validacion]
    y_ajuste = y_train_u3.iloc[pos_ajuste]
    y_validacion = y_train_u3.iloc[pos_validacion]
    if not X_ajuste.index.equals(y_ajuste.index) or not X_validacion.index.equals(y_validacion.index):
        raise ValueError("Entradas y respuestas no están alineadas.")
    claves_ajuste = set(claves_train_u3.iloc[pos_ajuste])
    claves_validacion = set(claves_train_u3.iloc[pos_validacion])
    if claves_ajuste.intersection(claves_validacion) or len(X_ajuste.index.intersection(X_validacion.index)):
        raise ValueError("Hay IDs o versiones compartidos.")
    if len(X_ajuste.index.union(X_validacion.index)) != len(y_train_u3) or len(X_ajuste.index.union(X_validacion.index).intersection(ids_test_u2)):
        raise ValueError("La cobertura no es entrenamiento completo o incluye test.")
    if set(y_ajuste.unique()) != {0, 1} or set(y_validacion.unique()) != {0, 1}:
        raise ValueError("Falta una clase en una partición.")
    if X_ajuste[variables_numericas_u3].isna().all().any():
        raise ValueError("Hay una variable numérica completamente ausente en ajuste.")
    flujo_lgb_ronda = clone(flujo_lgb_inicial_artista_u3)
    inicio = perf_counter()
    flujo_lgb_ronda.fit(X_ajuste, y_ajuste)
    segundos_fit = perf_counter() - inicio
    codificador_lgb = flujo_lgb_ronda.named_steps["artista"]
    preparador_lgb = flujo_lgb_ronda.named_steps["preparacion"]
    clasificador_lgb = flujo_lgb_ronda.named_steps["modelo"]
    if not codificador_lgb.ids_ajuste_.equals(X_ajuste.index) or codificador_lgb.cobertura_cruzada_ != len(X_ajuste):
        raise ValueError("La codificación no corresponde exactamente al ajuste permitido.")
    if codificador_lgb.claves_ajuste_.intersection(claves_validacion) or not np.isclose(codificador_lgb.respaldo_, y_ajuste.mean()):
        raise ValueError("El mapa o el respaldo de artista no respetan la separación.")
    medianas_audio = preparador_lgb.named_transformers_["numericas"].statistics_[:len(variables_numericas_u3)]
    if not np.allclose(medianas_audio, X_ajuste[variables_numericas_u3].median().to_numpy(dtype=float)):
        raise ValueError("Las medianas no proceden del ajuste permitido.")
    entradas_validacion = preparador_lgb.transform(codificador_lgb.transform(X_validacion))
    if not entradas_validacion.index.equals(X_validacion.index) or not np.isfinite(entradas_validacion.to_numpy(dtype=float)).all():
        raise ValueError("Entradas de validación desalineadas o no finitas.")
    if clasificador_lgb.booster_.feature_name() != list(entradas_validacion.columns):
        raise ValueError("Nombres u orden de características incompatibles.")
    if not 0 < clasificador_lgb.n_estimators_ <= modelo_lgb_inicial_u3.n_estimators:
        raise ValueError("Cantidad de etapas entrenadas inesperada.")
    columna_positiva = int(np.flatnonzero(clasificador_lgb.classes_ == 1)[0])
    proba = flujo_lgb_ronda.predict_proba(X_validacion)[:, columna_positiva]
    if not np.isfinite(proba).all() or not ((proba >= 0) & (proba <= 1)).all():
        raise ValueError("Las puntuaciones no son válidas.")
    decisiones = (proba >= UMBRAL_LGB_INICIAL_U3).astype("int8")
    probabilidades_lgb_inicial_artista_u3.loc[y_validacion.index] = proba
    predicciones_lgb_inicial_artista_u3.loc[y_validacion.index] = decisiones
    visitas_lgb_inicial_u3.loc[y_validacion.index] += 1
    nombres_consulta = codificador_lgb._artistas(X_validacion)
    sin_conocidos = nombres_consulta.map(lambda nombres: all(nombre not in codificador_lgb.mapa_ for nombre in nombres))
    valores_consulta = codificador_lgb.transform(X_validacion)["artista_clase_1"]
    if sin_conocidos.any() and not np.allclose(valores_consulta.loc[sin_conocidos], codificador_lgb.respaldo_):
        raise ValueError("Los artistas desconocidos no recibieron el respaldo permitido.")
    filas_lgb_inicial_u3.append({"Ronda": ronda, "Corte": UMBRAL_LGB_INICIAL_U3,
        **metricas_binarias_u3(y_validacion, proba, UMBRAL_LGB_INICIAL_U3), "Segundos fit": segundos_fit})
    filas_controles_lgb_u3.append({
        "Ronda": ronda, "IDs ajuste": len(X_ajuste), "IDs validación": len(X_validacion),
        "IDs codificados con exclusión": codificador_lgb.cobertura_cruzada_,
        "Claves compartidas": 0, "IDs test": 0,
        "Columnas": clasificador_lgb.n_features_in_, "Etapas reales": clasificador_lgb.n_estimators_,
        "% sin artistas conocidos": sin_conocidos.mean() * 100,
        "Respaldo permitido": codificador_lgb.respaldo_,
        "Medianas verificadas": True, "Nombres y valores verificados": True,
    })
    print(f"Ronda {ronda} completada: {len(X_ajuste):,} para ajustar; {len(X_validacion):,} para validar; {clasificador_lgb.n_estimators_} etapas; {segundos_fit:.2f}s de fit.", flush=True)
    del flujo_lgb_ronda, codificador_lgb, preparador_lgb, clasificador_lgb, entradas_validacion

if not visitas_lgb_inicial_u3.eq(1).all() or probabilidades_lgb_inicial_artista_u3.isna().any() or predicciones_lgb_inicial_artista_u3.isna().any():
    raise ValueError("Faltan predicciones OOF o hay repeticiones.")
resultados_rondas_lgb_inicial_u3 = pd.DataFrame(filas_lgb_inicial_u3).set_index("Ronda")
controles_lgb_inicial_u3 = pd.DataFrame(filas_controles_lgb_u3).set_index("Ronda")
display(controles_lgb_inicial_u3.round(4))
display(resultados_rondas_lgb_inicial_u3.round(6))
print(f"Cobertura OOF: {visitas_lgb_inicial_u3.eq(1).sum():,} IDs evaluados exactamente una vez.")

Ronda 1 completada: 47,709 para ajustar; 24,019 para validar; 100 etapas; 1.50s de fit.
Ronda 2 completada: 47,900 para ajustar; 23,828 para validar; 100 etapas; 1.28s de fit.
Ronda 3 completada: 47,847 para ajustar; 23,881 para validar; 100 etapas; 1.39s de fit.


,IDs ajuste,IDs validación,IDs codificados con exclusión,Claves compartidas,IDs test,Columnas,Etapas reales,% sin artistas conocidos,Respaldo permitido,Medianas verificadas,Nombres y valores verificados
Ronda,,,,,,,,,,,
1,47709,24019,47709,0,0,133,100,8.8680,0.1105,True,True
2,47900,23828,47900,0,0,133,100,8.9978,0.1095,True,True
3,47847,23881,47847,0,0,133,100,8.9485,0.1088,True,True


,Corte,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI,Segundos fit
Ronda,,,,,,,,,,,,,
1,0.5,20812,616,1598,993,0.617154,0.383250,0.472857,0.907823,0.905162,0.585366,False,1.495448
2,0.5,20690,522,1580,1036,0.664955,0.396024,0.496406,0.911784,0.904468,0.600046,False,1.275458
3,0.5,20679,548,1594,1060,0.659204,0.399397,0.497419,0.910305,0.909627,0.606701,False,1.389759


Cobertura OOF: 71,728 IDs evaluados exactamente una vez.


### Explicación de las rondas y sus controles

1. Se reservan series por ID para puntuaciones, decisiones y número de validaciones; las dos primeras comienzan faltantes para detectar omisiones.
2. El bucle usa exactamente `particiones_externas_u3`. `.iloc` recupera entradas y respuestas en el orden de las posiciones guardadas. No se crean otras particiones para mejorar resultados.
3. Se comprueban alineación, versiones e IDs separados, cobertura de entrenamiento, ausencia de test, ambas clases y medianas calculables.
4. `clone` crea artista, preparación y modelo nuevos. `fit` aprende todo exclusivamente con ajuste. Dentro del codificador, cada fila de ajuste recibe un valor calculado sin su grupo de claves ni sus versiones; la validación recibirá el mapa completo de ese ajuste, sin sus respuestas.
5. El tiempo medido incluye preparación y aprendizaje, no toda la predicción o revisión. No se utiliza como comparación controlada de velocidad entre librerías ni se compara con una búsqueda de muchos ajustes.
6. Se verifican IDs del mapa, cobertura cruzada, ausencia de claves externas, respaldo y medianas. Una coincidencia de números aislada no sustituye la evidencia del flujo fit/transform.
7. `transform` obtiene la tabla de validación con nombres, sin reajustar. Se exige el índice original y valores finitos. `booster_.feature_name()` comprueba nombre y orden de las características que aprendió el modelo.
8. `n_estimators_` informa cuántas etapas se entrenaron realmente; se contrasta con las 100 solicitadas y se registra. No se eligieron mediante el resultado de validación.
9. `classes_` identifica la columna positiva. `predict_proba` transforma y predice sin recibir `y_validacion`; se exigen puntuaciones finitas entre cero y uno. No se afirma calibración por cumplir ese rango.
10. Se aplica únicamente corte 0,50 y se guarda por ID. El contador aumenta una vez. Se verifica además el respaldo de nombres desconocidos, sin calcular sus valores con respuestas externas.
11. La función de métricas compara predicciones con respuestas después de predecir. Se guardan conteos, métricas, tiempos y controles; `display` presenta resultados reales y `round` solo modifica la visualización.
12. Se retiran los objetos ajustados de cada ronda. No se guarda el último como un modelo final de todo entrenamiento. Al terminar, cada ID debe tener una sola OOF.

Aquí las tres rondas externas evalúan una configuración fija. Los cinco pliegues internos para búsqueda se utilizarán en el bloque posterior; los cinco grupos de codificación del artista sí se ejecutan dentro de cada fit, pero no entrenan cinco LightGBM extra.

In [30]:
metricas_lgb_inicial_u3 = metricas_binarias_u3(y_train_u3, probabilidades_lgb_inicial_artista_u3, UMBRAL_LGB_INICIAL_U3)
matriz_lgb_inicial_u3 = pd.DataFrame(
    [[metricas_lgb_inicial_u3["TN"], metricas_lgb_inicial_u3["FP"]],
     [metricas_lgb_inicial_u3["FN"], metricas_lgb_inicial_u3["TP"]]],
    index=["Real 0: popularidad <60", "Real 1: popularidad >=60"],
    columns=["Predice 0", "Predice 1"],
)
precision_lgb_u3 = metricas_lgb_inicial_u3["Precisión"]
recall_lgb_u3 = metricas_lgb_inicial_u3["Recall"]
kpi_lgb_inicial_u3 = pd.DataFrame([
    {"KPI": "Precisión >=30%", "Resultado": precision_lgb_u3,
     "Estado": "Cumple" if pd.notna(precision_lgb_u3) and precision_lgb_u3 >= META_PRECISION else "No cumple"},
    {"KPI": "Recall >=60%", "Resultado": recall_lgb_u3,
     "Estado": "Cumple" if recall_lgb_u3 >= META_RECALL else "No cumple"},
    {"KPI": "Ambas metas simultáneas", "Resultado": np.nan,
     "Estado": "Cumple" if metricas_lgb_inicial_u3["Cumple ambos KPI"] else "No cumple"},
])
columnas_metricas_ensambles_u3 = ["Precisión", "Recall", "F1", "Accuracy", "ROC-AUC", "Average Precision"]
filas_comparacion_ensambles_iniciales_u3 = []
for nombre, etapa, metricas in [
    ("Random Forest con artista", "Inicial; corte 0.50", metricas_rf_inicial_u3),
    ("LightGBM con artista", "Inicial; corte 0.50", metricas_lgb_inicial_u3),
]:
    filas_comparacion_ensambles_iniciales_u3.append({"Modelo": nombre, "Etapa y decisión": etapa,
        **{m: metricas[m] for m in columnas_metricas_ensambles_u3},
        "KPI simultáneos": "Cumple" if metricas["Cumple ambos KPI"] else "No cumple"})
for tipo, descripcion in [("Fijo 0.50", "Optimizado; corte 0.50"), ("Elegido interno", "Optimizado; cortes internos")]:
    fila_xgb = comparacion_anidada_artista_u3.loc[
        (comparacion_anidada_artista_u3["Variante"] == "Con artista") &
        (comparacion_anidada_artista_u3["Decisión"] == tipo)
    ].iloc[0]
    filas_comparacion_ensambles_iniciales_u3.append({"Modelo": "XGBoost con artista", "Etapa y decisión": descripcion,
        **{m: fila_xgb[m] for m in columnas_metricas_ensambles_u3}, "KPI simultáneos": fila_xgb["KPI simultáneos"]})
comparacion_ensambles_iniciales_u3 = pd.DataFrame(filas_comparacion_ensambles_iniciales_u3)
display(matriz_lgb_inicial_u3)
display(kpi_lgb_inicial_u3.round({"Resultado": 6}))
display(comparacion_ensambles_iniciales_u3.round(6))
print(f"LightGBM detecta {metricas_lgb_inicial_u3['TP']:,} de {int(y_train_u3.sum()):,} positivas.")
print(f"Omite {metricas_lgb_inicial_u3['FN']:,} positivas y genera {metricas_lgb_inicial_u3['FP']:,} falsas alarmas.")
print(f"Precisión {precision_lgb_u3:.2%}; recall {recall_lgb_u3:.2%}; F1 {metricas_lgb_inicial_u3['F1']:.4f}.")

,Predice 0,Predice 1
Real 0: popularidad <60,62181,1686
Real 1: popularidad >=60,4772,3089


,KPI,Resultado,Estado
0,Precisión >=30%,0.646911,Cumple
1,Recall >=60%,0.392953,No cumple
2,Ambas metas simultáneas,NaN,No cumple


,Modelo,Etapa y decisión,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,KPI simultáneos
0,Random Forest con artista,Inicial; corte 0.50,0.893887,0.098588,0.177589,0.899928,0.895779,0.557724,No cumple
1,LightGBM con artista,Inicial; corte 0.50,0.646911,0.392953,0.488921,0.909965,0.906320,0.597222,No cumple
2,XGBoost con artista,Optimizado; corte 0.50,0.643685,0.416868,0.506022,0.910802,0.910567,0.605341,No cumple
3,XGBoost con artista,Optimizado; cortes internos,0.487494,0.681847,0.568519,0.886571,0.910567,0.605341,Cumple


LightGBM detecta 3,089 de 7,861 positivas.
Omite 4,772 positivas y genera 1,686 falsas alarmas.
Precisión 64.69%; recall 39.30%; F1 0.4889.


### Explicación de métricas, comparación y KPI

1. Se reúnen las 71.728 predicciones externas de LightGBM. Cada una procede de un Pipeline que excluyó su clave de canción durante ajuste. Las métricas globales no son necesariamente el promedio simple de métricas por ronda.
2. La matriz tiene clases reales en filas y decisiones en columnas. TN son negativas descartadas, FP falsas alarmas, FN positivas omitidas y TP positivas detectadas. Los cuatro conteos deben sumar 71.728.
3. Precisión = TP/(TP+FP) y recall = TP/(TP+FN). Si no hay predicciones positivas, la precisión queda no definida y no cumple el KPI. La condición simultánea exige ambas metas; no se reemplaza por accuracy, AUC o F1.
4. La fila de cumplimiento simultáneo no tiene un resultado numérico propio; se lee la columna Estado. Los valores de precisión y recall se muestran en escala 0–1; multiplicarlos por cien da porcentajes.
5. Se recuperan métricas de Random Forest y XGBoost ya calculadas sobre los mismos IDs y particiones. No se usan resultados de test, ni los cinco pliegues del XGBoost inicial sin artista.
6. Random Forest y LightGBM tienen configuración inicial y corte 0,50. XGBoost aporta dos referencias optimizadas claramente identificadas; sus parámetros y cortes se eligieron dentro de desarrollo. Eso permite contextualizar, no adjudicar una comparación definitiva con igual esfuerzo de optimización.
7. F1 resume precisión y recall. Accuracy cuenta ambos tipos de acierto y puede ser alta aunque se omitan positivas. ROC-AUC y Average Precision describen ordenamiento de puntuaciones; no son porcentajes de canciones acertadas ni garantizan calibración.
8. Las dos filas de XGBoost tienen las mismas AUC/AP porque provienen de las mismas puntuaciones con decisiones diferentes. No se copia su corte al LightGBM para intentar cumplir metas después de ver resultados.
9. Si el recall es insuficiente, se informa cuántas positivas se omitieron. Un corte exigente y el desbalance pueden influir, pero no se afirma una única causa ni que optimizar garantice mejorar. Tampoco se declara inútil todo el algoritmo por una configuración inicial.

Las métricas son de comparación dentro del entrenamiento del proyecto. Los errores y las diferencias entre rondas deben acompañar cualquier elección provisional; no constituyen una prueba estadística de superioridad ni una confirmación externa nueva.

### 16.3. Cierre de las configuraciones iniciales y siguiente paso

Quedan incorporados Random Forest y LightGBM iniciales con artista, junto a la evidencia de XGBoost. En LightGBM se verificaron separación por clave, aprendizaje del artista y preparación dentro de cada ajuste, nombres de características y una OOF por ID. Los CSV y la partición previa no se modificaron.

No se hizo búsqueda, balanceo adicional, selección de corte, evaluación en test ni ajuste final sobre todo entrenamiento. No se declarará ganador definitivo con modelos en distintas etapas de optimización.

**Siguiente paso:** definir el bloque conjunto de hiperparámetros y validación para los ensambles, conservando los puntos de partida. Los espacios y presupuestos de Grid/Random deben fijarse y justificarse; la elección de parámetros y cortes se hará en los cinco pliegues internos y la evaluación en las mismas tres rondas externas, documentando las decisiones históricas ya observadas.

Se conservan además los pendientes de aplicación acotada de Leave-P-Out, análisis de errores e importancia de variables, cierre de clasificación, optimización de clustering y entregables de la pauta. El informe PDF final recogerá decisiones, resultados reales, KPI y limitaciones.

Fuentes: [LGBMClassifier](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.LGBMClassifier.html) y [parámetros de LightGBM](https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html).

### Lectura de los resultados ejecutados

LightGBM inicial al corte 0,50 obtuvo precisión **64.69%**, recall **39.30%**, F1 **0.4889**, accuracy **91.00%**, ROC-AUC **0.9063** y Average Precision **0.5972**. El cumplimiento simultáneo de KPI es **No cumple**. La precisión cumple el mínimo de 30%; el recall no cumple el mínimo de 60%.

Detectó **3,089** de 7.861 positivas y omitió **4,772**. Hubo **1,686** falsas alarmas y **62,181** negativas correctamente descartadas. De cada cien canciones señaladas aproximadamente 64.7 son positivas reales; de cada cien positivas reales se detectan aproximadamente 39.3.

Frente a Random Forest inicial, con el mismo corte y particiones, el cambio de recall fue +29.44 puntos porcentuales y el de F1 +0.3113. Se detectaron +2,314 positivas y hubo +1,594 falsas alarmas de diferencia. Hay que considerar ambos cambios, no solo una métrica. La comparación es descriptiva de estas configuraciones iniciales y no demuestra superioridad universal.

No se modificó el corte ni la configuración para alterar el cumplimiento. Las cifras anteriores quedan como punto de partida de LightGBM antes de optimizar.


**Por qué falla el recall:** al corte 0,50 se señalaron 4.775 canciones, de las cuales 3.089 fueron positivas reales y 1.686 falsas alarmas. Sin embargo, 4.772 de las 7.861 positivas quedaron fuera: se omite aproximadamente el 60,70%. El recall de 39,30% no alcanza la meta de 60% en ninguna de las tres rondas. La precisión de 64,69% sí supera 30%; la accuracy de 91,00% y AUC de 0,9063 no reemplazan el requisito incumplido.

**Cómo leer la diferencia entre ensambles:** LightGBM inicial detectó 2.314 positivas más que Random Forest inicial, con 1.594 falsas alarmas adicionales. A igual corte 0,50, XGBoost optimizado tuvo recall 41,69% y F1 0,5060 frente a 39,30% y 0,4889 de LightGBM inicial. Esa proximidad es descriptiva; todavía no compara modelos con el mismo esfuerzo de búsqueda. Solo el procedimiento XGBoost con cortes internos de la tabla cumple ambos KPI por ahora. El umbral de LightGBM no se cambia a partir de estas respuestas externas.

**Controles ejecutados:** tres ajustes completos, 100 etapas reales y 133 columnas en cada ronda; 71.728 OOF únicas; cero IDs de test y cero claves compartidas entre ajuste y validación. Los tiempos de fit registrados incluyen la preparación de artista y características. Las bibliotecas y configuraciones iniciales difieren entre ensambles, por lo que estos tiempos no prueban una ventaja universal de velocidad.


## 17. Optimización conjunta de Random Forest, LightGBM y XGBoost con artista

### 17.1. Objetivo y reglas fijadas antes de ejecutar

Se comparan los tres ensambles con artista mediante GridSearchCV y RandomizedSearchCV. Se mantiene la clase popularidad ≥60, los KPI de precisión ≥30% y recall ≥60%, la preparación dentro de cada fit y la exclusión de versiones. Se usa solo entrenamiento, con las mismas tres rondas externas y cinco internas agrupadas de los bloques anteriores.

La **AP media interna** selecciona hiperparámetros. Después, las puntuaciones OOF internas del ganador eligen el corte con la misma lista y regla del bloque 13: cumplir ambas metas, maximizar F1 y desempatar por precisión y corte; si ninguno cumple, corte 0,50 y registro de falta de viabilidad interna. Las métricas internas son de desarrollo porque sus datos participaron en la selección de parámetros. La evaluación externa mide el procedimiento completo.

Se informan dos decisiones sobre las mismas puntuaciones externas: **corte fijo 0,50** y **corte elegido internamente**. Así se distingue el efecto de optimizar parámetros del cambio de umbral. No se mueve un corte después de observar errores externos para hacerlo aprobar.

Además se evalúa una regla de selección conjunta: en cada desarrollo elegir la pareja modelo/método con mayor AP media interna; empate exacto: menos configuraciones y luego orden alfabético del nombre de modelo y método. Esta regla se aplica antes de consultar respuestas externas. Elegir después una fila por su métrica externa sería otra selección y no una nueva confirmación independiente.

Se conservan los resultados anteriores: no se reescriben los puntos de partida. El test histórico no se utiliza y las decisiones históricas de exploración siguen siendo limitaciones. Esta ampliación no crea una prueba externa nueva.

### 17.2. Espacios de búsqueda y costo

| Modelo | Espacio fijado | Combinaciones Grid | Configuraciones Random por ronda |
|---|---|---:|---:|
| Random Forest | árboles 100/200; profundidad 5/10/20; mínimo de hoja 1/3 | 12 | 10 |
| LightGBM | etapas 100/200; hojas 15/31; tasa 0,05/0,10; mínimo por hoja 10/20 | 16 | 10 |
| XGBoost | árboles 100/200; profundidad 3/6; tasa 0,05/0,10; subsample 0,8/1,0 | 16 | 10 |

Random Forest conserva `max_features="sqrt"`, bootstrap y ausencia de ponderación. LightGBM conserva profundidad 6, sin bagging de filas ni ponderación: sus hojas 15/31 son compatibles con ese límite. XGBoost conserva el resto de su configuración previa. Los valores de la guía se acotan para que el costo sea revisable; no se afirma cubrir todo el espacio posible ni encontrar un óptimo global.

Grid evalúa todo el espacio y Random diez configuraciones sin reemplazo, con semilla 42. Random es un subconjunto de Grid; sus presupuestos no son iguales. Cada pareja modelo/método usa las mismas cinco particiones internas de cada desarrollo.

**Costo de candidatos en tres rondas:** RF: (12+10)×5×3=330; LightGBM: (16+10)×5×3=390; XGBoost: (16+10)×5×3=390. Total: **1.110 ajustes de candidatos**, de los que 150 corresponden a Random de XGBoost con artista ya ejecutado. Faltan contabilizar refits y generación OOF para cortes; las tablas distinguirán costo representado y ajustes nuevos.

**Reutilización verificable:** Random de XGBoost del bloque 14 ya tiene candidatos, parámetros, cortes y puntuaciones externas. Solo se reutiliza tras comprobar datos/respuestas, particiones, versiones y coincidencia de puntuaciones internas contra la nueva búsqueda Grid. Si Grid elige sus mismos parámetros y reproduce sus puntuaciones externas, se conserva también la tabla de cortes ya calculada. Si algo no coincide, se detiene el bloque en vez de presentar evidencia antigua como compatible.

Entre Grid y Random de RF o LightGBM se puede reutilizar una tabla de cortes si los parámetros ganadores son idénticos, dentro del mismo desarrollo y modelo. Se reutiliza una selección interna idéntica; no se decide con respuestas externas. Los candidatos de ambas búsquedas se ejecutan por separado.

Se permiten dos ajustes independientes en paralelo mediante hilos; cada estimador conserva `n_jobs=1`. Eso reduce espera sin convertir un pliegue en datos de otro. Los tiempos son descriptivos de este equipo, orden y paralelismo; los de XGBoost Random proceden de su ejecución anterior y no son una comparación controlada de velocidad.

In [31]:
from functools import partial
from math import prod
from joblib import parallel_backend

PARALELOS_BUSQUEDA_U3 = 2
espacios_conjuntos_u3 = {
    "Random Forest": {"modelo__n_estimators": [100, 200], "modelo__max_depth": [5, 10, 20], "modelo__min_samples_leaf": [1, 3]},
    "LightGBM": {"modelo__n_estimators": [100, 200], "modelo__num_leaves": [15, 31], "modelo__learning_rate": [0.05, 0.10], "modelo__min_child_samples": [10, 20]},
    "XGBoost": espacio_pipeline_xgb_u3,
}
plantillas_conjuntas_u3 = {
    "Random Forest": flujo_rf_inicial_artista_u3,
    "LightGBM": flujo_lgb_inicial_artista_u3,
    "XGBoost": flujo_xgb_con_artista_u3,
}
orden_desempate_conjunto_u3 = {
    "Random Forest": [("n_estimators", 1), ("max_depth", 1), ("min_samples_leaf", -1)],
    "LightGBM": [("n_estimators", 1), ("num_leaves", 1), ("learning_rate", 1), ("min_child_samples", -1)],
    "XGBoost": [("n_estimators", 1), ("max_depth", 1), ("learning_rate", 1), ("subsample", 1)],
}

def indice_ganador_conjunto_u3(resultados, modelo):
    ap = np.asarray(resultados["mean_test_ap"])
    if not np.isfinite(ap).all():
        raise ValueError("Una búsqueda produjo AP no finita.")
    return min(range(len(ap)), key=lambda i: (
        -ap[i],
        *[signo * resultados["params"][i][f"modelo__{nombre}"] for nombre, signo in orden_desempate_conjunto_u3[modelo]],
    ))

def clave_parametros_conjuntos_u3(parametros):
    return tuple(sorted(parametros.items()))

plan_busquedas_conjuntas_u3 = pd.DataFrame([
    {"Modelo": modelo, "Configuraciones Grid": prod(len(v) for v in espacio.values()),
     "Configuraciones Random": 10, "CV Grid por externa": prod(len(v) for v in espacio.values()) * PLIEGUES_BUSQUEDA,
     "CV Random por externa": 10 * PLIEGUES_BUSQUEDA}
    for modelo, espacio in espacios_conjuntos_u3.items()
])
display(plan_busquedas_conjuntas_u3)
print("Métrica de selección: AP media interna. KPI y regla de cortes conservados.")
print("Cada estimador usa un hilo; la búsqueda permite dos ajustes independientes en paralelo.")

,Modelo,Configuraciones Grid,Configuraciones Random,CV Grid por externa,CV Random por externa
0,Random Forest,12,10,60,50
1,LightGBM,16,10,80,50
2,XGBoost,16,10,80,50


Métrica de selección: AP media interna. KPI y regla de cortes conservados.
Cada estimador usa un hilo; la búsqueda permite dos ajustes independientes en paralelo.


### Explicación del plan y los desempates

1. `partial` permite fijar el nombre de modelo en una función de selección; `prod` multiplica cantidades de opciones; `parallel_backend` limita el tipo de paralelismo.
2. Los diccionarios declaran espacios y plantillas. El prefijo `modelo__` cambia parámetros del clasificador dentro del Pipeline completo: artista y preparación siguen aprendiendo dentro de cada ajuste.
3. Cada lista de desempate solo actúa si la AP media es exactamente igual. Se prefieren menos árboles y menor profundidad o cantidad de hojas; el signo -1 prioriza un mínimo de hoja mayor. Tasa y subsample conservan los desempates previos de XGBoost. No se afirma que cada desempate garantice mejor calidad.
4. La función exige puntuaciones finitas, ordena por AP descendente y después por la regla declarada. `partial(..., modelo=...)` se entregará como `refit` para reajustar al candidato elegido con todo desarrollo.
5. `clave_parametros_conjuntos_u3` crea una representación ordenada de parámetros para detectar configuraciones idénticas; no ordena canciones ni incorpora respuestas externas.
6. La comprensión crea una fila por modelo. `prod` cuenta combinaciones y multiplicar por cinco da ajustes CV por ronda externa. Los números todavía no son tiempos ni resultados de los modelos.
7. `display` y `print` registran las decisiones previas. El mayor puntaje interno será desarrollo; se evaluará después sobre la parte externa reservada.

In [32]:
def ejecutar_busqueda_conjunta_u3(modelo, metodo, X_dev, y_dev, X_ext, grupos_dev, grupos_ext, particiones_int, cache_cortes, anterior_xgb=None):
    espacio = espacios_conjuntos_u3[modelo]
    argumentos = dict(
        estimator=clone(plantillas_conjuntas_u3[modelo]), cv=particiones_int,
        scoring=metricas_busqueda_u3, refit=partial(indice_ganador_conjunto_u3, modelo=modelo),
        n_jobs=PARALELOS_BUSQUEDA_U3, error_score="raise", return_train_score=False,
    )
    busqueda = (GridSearchCV(param_grid=espacio, **argumentos) if metodo == "Grid" else
                RandomizedSearchCV(param_distributions=espacio, n_iter=10, random_state=SEMILLA, **argumentos))
    inicio = perf_counter()
    with parallel_backend("threading", n_jobs=PARALELOS_BUSQUEDA_U3):
        busqueda.fit(X_dev, y_dev)
    segundos_busqueda = perf_counter() - inicio
    candidatos = pd.DataFrame(busqueda.cv_results_)
    parametros = busqueda.best_params_
    ganador = busqueda.best_index_
    modelo_ajustado = busqueda.best_estimator_
    codificador = modelo_ajustado.named_steps["artista"]
    preparador = modelo_ajustado.named_steps["preparacion"]
    if not codificador.ids_ajuste_.equals(X_dev.index) or codificador.cobertura_cruzada_ != len(X_dev):
        raise ValueError("El artista no se ajustó exclusivamente con desarrollo y cobertura cruzada completa.")
    if codificador.claves_ajuste_.intersection(set(grupos_ext)) or not np.isclose(codificador.respaldo_, y_dev.mean()):
        raise ValueError("Mapa de artista o respaldo incompatibles con la separación externa.")
    medianas = preparador.named_transformers_["numericas"].statistics_[:len(variables_numericas_u3)]
    if not np.allclose(medianas, X_dev[variables_numericas_u3].median().to_numpy(dtype=float)):
        raise ValueError("Medianas externas no aprendidas con desarrollo.")
    if modelo == "XGBoost" and anterior_xgb is not None:
        tabla_grid = {clave_parametros_conjuntos_u3(fila["params"]): fila for _, fila in candidatos.iterrows()}
        columnas_puntajes = [f"split{k}_test_{metrica}" for k in range(PLIEGUES_BUSQUEDA) for metrica in ["ap", "auc"]]
        for _, fila in anterior_xgb["candidatos"].iterrows():
            actual = tabla_grid[clave_parametros_conjuntos_u3(fila["params"])]
            if not np.allclose(fila[columnas_puntajes].to_numpy(dtype=float), actual[columnas_puntajes].to_numpy(dtype=float), rtol=0, atol=1e-10):
                raise ValueError("La evidencia Random de XGBoost no se reproduce con el protocolo actual.")
    llave = clave_parametros_conjuntos_u3(parametros)
    inicio = perf_counter()
    if llave in cache_cortes:
        corte_info = cache_cortes[llave]
        ajustes_corte_nuevos = 0
        origen_corte = "Reutilizado: mismos parámetros y desarrollo"
    else:
        with parallel_backend("threading", n_jobs=PARALELOS_BUSQUEDA_U3):
            proba_interna = cross_val_predict(clone(modelo_ajustado), X_dev, y_dev, cv=particiones_int,
                method="predict_proba", n_jobs=PARALELOS_BUSQUEDA_U3)[:, 1]
        if not np.isfinite(proba_interna).all() or not ((proba_interna >= 0) & (proba_interna <= 1)).all():
            raise ValueError("Puntuaciones internas inválidas.")
        corte, viable, tabla_cortes = elegir_corte_interno_u3(y_dev, proba_interna)
        corte_info = {"corte": corte, "viable": viable, "tabla": tabla_cortes}
        cache_cortes[llave] = corte_info
        ajustes_corte_nuevos = PLIEGUES_BUSQUEDA
        origen_corte = "Calculado: OOF interna"
    segundos_corte = perf_counter() - inicio
    columna_positiva = int(np.flatnonzero(modelo_ajustado.named_steps["modelo"].classes_ == 1)[0])
    proba_externa = pd.Series(modelo_ajustado.predict_proba(X_ext)[:, columna_positiva], index=X_ext.index)
    if not np.isfinite(proba_externa).all() or not proba_externa.between(0, 1).all():
        raise ValueError("Puntuaciones externas inválidas.")
    if modelo == "XGBoost" and anterior_xgb is not None and llave == clave_parametros_conjuntos_u3(anterior_xgb["parametros"]):
        if not np.allclose(proba_externa.to_numpy(), anterior_xgb["proba"].to_numpy(), rtol=0, atol=1e-12):
            raise ValueError("La reproducción externa de XGBoost no coincide con la evidencia reutilizada.")
    decision = proba_externa.ge(corte_info["corte"]).astype("int8")
    return {
        "parametros": parametros, "ap_interna": float(candidatos.loc[ganador, "mean_test_ap"]),
        "de_ap_interna": float(candidatos.loc[ganador, "std_test_ap"]),
        "auc_interna": float(candidatos.loc[ganador, "mean_test_auc"]),
        "candidatos": candidatos, "corte_info": corte_info, "proba": proba_externa, "decision": decision,
        "configuraciones": len(candidatos), "ajustes_cv": len(candidatos) * PLIEGUES_BUSQUEDA,
        "cv_nuevos": len(candidatos) * PLIEGUES_BUSQUEDA, "refits_nuevos": 1,
        "cortes_nuevos": ajustes_corte_nuevos, "segundos_busqueda": segundos_busqueda,
        "segundos_corte": segundos_corte, "origen": "Ejecutado en bloque 17", "origen_corte": origen_corte,
        "IDs desarrollo": len(X_dev), "IDs evaluación": len(X_ext),
        "IDs codificación cruzada": codificador.cobertura_cruzada_,
        "Claves externas en ajuste": 0, "IDs test": 0,
    }

### Explicación de la función de búsqueda

1. La función recibe desarrollo y sus respuestas, entradas externas **sin respuestas externas**, grupos y particiones internas. No puede elegir parámetros o corte usando `y_externo` porque no lo recibe.
2. Se clona todo el Pipeline para la búsqueda. La lista de pares internos ya comprobados se entrega a ambas herramientas; `error_score="raise"` impide ocultar ajustes fallidos.
3. `GridSearchCV` revisa todo el diccionario; Random toma diez combinaciones con semilla fija. El contexto de hilos permite dos ajustes independientes; cada modelo sigue con un hilo.
4. `fit` ejecuta CV interna y un refit del ganador. Se conservan `cv_results_`, parámetros y `best_index_`; al usar refit con función se recuperan puntuaciones desde la tabla, sin suponer que existe `best_score_`.
5. Se comprueban IDs del codificador, cobertura cruzada, ausencia de claves externas, respaldo y medianas. En cada fit interno el Pipeline también aplica exclusión de versiones en la codificación, no solo en el refit.
6. Para XGBoost, la nueva Grid contiene los diez candidatos de Random anterior. Se comparan AP y AUC de cada pliegue interno, tolerando únicamente diferencias numéricas diminutas. Una incompatibilidad detiene la reutilización.
7. Si ya se evaluó el corte para los mismos parámetros en ese desarrollo/modelo, se conserva esa tabla. De lo contrario, `cross_val_predict` clona y ajusta cinco Pipelines para producir OOF internas, y la función de corte aplica la regla previa. Estos ajustes no son una nueva evaluación externa.
8. Se predice externo con el ganador reajustado; se valida el rango. Si XGBoost coincide con los parámetros anteriores, se exige reproducir también sus puntuaciones externas antes de reutilizarlas como evidencia compatible. No se comparan respuestas externas para decidir qué conservar.
9. La decisión usa el corte ya elegido. El diccionario de salida guarda parámetros, candidatos, puntuaciones, cortes, tiempos y costo nuevo. No guarda modelos finales ni transforma el último modelo de una ronda en un modelo de todo entrenamiento.
10. El parámetro `grupos_dev` documenta el desarrollo recibido; la construcción y comprobación de las particiones se realiza en el siguiente bucle. Las comprobaciones de separación son parte del flujo completo, no una garantía derivada solo de un atributo.

Las tablas de cortes reutilizadas pertenecen al mismo conjunto de desarrollo y modelo, sin compartir respuestas externas. No se reutiliza una codificación de artista entre conjuntos de ajuste distintos.

In [33]:
evidencia_conjunta_u3 = {}
filas_particiones_conjuntas_u3 = []
filas_metricas_conjuntas_ronda_u3 = []
filas_seleccion_conjunta_u3 = []
visitas_conjuntas_u3 = pd.Series(0, index=y_train_u3.index, dtype="int64")
proba_seleccion_conjunta_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64")
pred_seleccion_conjunta_u3 = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64")
proba_xgb_inicial_artista_conjunta_u3 = pd.Series(np.nan, index=y_train_u3.index, dtype="float64")

for externa, (pos_dev, pos_ext) in enumerate(particiones_externas_u3, start=1):
    X_dev = X_train_con_artista_u3.iloc[pos_dev]
    y_dev = y_train_u3.iloc[pos_dev]
    grupos_dev = claves_train_u3.iloc[pos_dev]
    X_ext = X_train_con_artista_u3.iloc[pos_ext]
    y_ext = y_train_u3.iloc[pos_ext]
    grupos_ext = claves_train_u3.iloc[pos_ext]
    if set(grupos_dev).intersection(set(grupos_ext)) or len(X_dev.index.intersection(X_ext.index)):
        raise ValueError("IDs o claves compartidas externamente.")
    if len(X_dev.index.union(X_ext.index)) != len(y_train_u3) or len(X_dev.index.union(X_ext.index).intersection(ids_test_u2)):
        raise ValueError("Cobertura externa incorrecta o presencia de test.")
    if set(y_dev.unique()) != {0, 1} or set(y_ext.unique()) != {0, 1}:
        raise ValueError("Falta una clase externa.")
    cv_int = StratifiedGroupKFold(n_splits=PLIEGUES_BUSQUEDA, shuffle=True, random_state=SEMILLA)
    particiones_int = list(cv_int.split(X_dev, y_dev, groups=grupos_dev))
    visitas_int = np.zeros(len(X_dev), dtype=int)
    for interna, (pos_fit, pos_val) in enumerate(particiones_int, start=1):
        if set(grupos_dev.iloc[pos_fit]).intersection(set(grupos_dev.iloc[pos_val])):
            raise ValueError("Claves compartidas internamente.")
        if len(set(pos_fit).intersection(set(pos_val))) or len(set(pos_fit).union(set(pos_val))) != len(X_dev):
            raise ValueError("IDs internos repetidos o cobertura incorrecta.")
        if set(y_dev.iloc[pos_fit].unique()) != {0, 1} or set(y_dev.iloc[pos_val].unique()) != {0, 1}:
            raise ValueError("Falta una clase interna.")
        if X_dev.iloc[pos_fit][variables_numericas_u3].isna().all().any():
            raise ValueError("Una variable numérica no tiene mediana interna.")
        visitas_int[pos_val] += 1
        filas_particiones_conjuntas_u3.append({"Externa": externa, "Interna": interna,
            "IDs desarrollo": len(X_dev), "IDs externos": len(X_ext), "IDs fit interno": len(pos_fit),
            "IDs validación interna": len(pos_val), "Claves compartidas": 0, "IDs test": 0})
    if not (visitas_int == 1).all():
        raise ValueError("Cobertura OOF interna incorrecta.")
    resultados_de_la_ronda = {}
    for modelo in ["Random Forest", "LightGBM", "XGBoost"]:
        cache_cortes = {}
        anterior_xgb = None
        if modelo == "XGBoost":
            registro = resumen_busqueda_artista_u3.loc[resumen_busqueda_artista_u3["Externa"].eq(externa)].iloc[0]
            parametros_anteriores = {f"modelo__{k}": v for k, v in registro["Parámetros"].items()}
            anterior_xgb = {
                "parametros": parametros_anteriores, "candidatos": candidatos_con_artista_u3[externa],
                "proba": proba_externa_artista_u3.loc[y_ext.index],
            }
            cache_cortes[clave_parametros_conjuntos_u3(parametros_anteriores)] = {
                "corte": float(registro["Corte interno"]), "viable": bool(registro["Corte viable interno"]),
                "tabla": cortes_con_artista_u3[externa],
            }
        for metodo in ["Grid", "Random"]:
            print(f"INICIO: externa {externa}/3; {modelo}; {metodo}.", flush=True)
            if modelo == "XGBoost" and metodo == "Random":
                # Grid ya comprobó los diez candidatos anteriores dentro del protocolo actual.
                resultado = {
                    "parametros": parametros_anteriores, "ap_interna": float(registro["AP media interna"]),
                    "de_ap_interna": float(registro["DE AP interna"]),
                    "auc_interna": float(candidatos_con_artista_u3[externa]["mean_test_auc"].iloc[indice_ganador_conjunto_u3(candidatos_con_artista_u3[externa], "XGBoost")]),
                    "candidatos": candidatos_con_artista_u3[externa],
                    "corte_info": cache_cortes[clave_parametros_conjuntos_u3(parametros_anteriores)],
                    "proba": proba_externa_artista_u3.loc[y_ext.index].copy(),
                    "decision": pred_externa_artista_u3.loc[y_ext.index].astype("int8"),
                    "configuraciones": 10, "ajustes_cv": 50, "cv_nuevos": 0, "refits_nuevos": 0, "cortes_nuevos": 0,
                    "segundos_busqueda": float(registro["Segundos búsqueda y refit"]),
                    "segundos_corte": float(registro["Segundos OOF y corte"]),
                    "origen": "Reutilizado y verificado: bloque 14", "origen_corte": "Bloque 14",
                    "IDs desarrollo": len(X_dev), "IDs evaluación": len(X_ext),
                    "IDs codificación cruzada": len(X_dev), "Claves externas en ajuste": 0, "IDs test": 0,
                }
            else:
                resultado = ejecutar_busqueda_conjunta_u3(modelo, metodo, X_dev, y_dev, X_ext,
                    grupos_dev, grupos_ext, particiones_int, cache_cortes, anterior_xgb)
            evidencia_conjunta_u3[(externa, modelo, metodo)] = resultado
            resultados_de_la_ronda[(modelo, metodo)] = resultado
            print(f"COMPLETADO: externa {externa}; {modelo}; {metodo}; AP interna {resultado['ap_interna']:.4f}; corte {resultado['corte_info']['corte']:.2f}.", flush=True)
    # Elegir pareja por AP interna antes de calcular las métricas externas.
    pareja_elegida = min(resultados_de_la_ronda, key=lambda pareja: (
        -resultados_de_la_ronda[pareja]["ap_interna"], resultados_de_la_ronda[pareja]["configuraciones"], pareja[0], pareja[1]))
    elegida = resultados_de_la_ronda[pareja_elegida]
    proba_seleccion_conjunta_u3.loc[y_ext.index] = elegida["proba"]
    pred_seleccion_conjunta_u3.loc[y_ext.index] = elegida["decision"]
    filas_seleccion_conjunta_u3.append({"Externa": externa, "Modelo": pareja_elegida[0], "Método": pareja_elegida[1],
        "AP interna": elegida["ap_interna"], "Corte": elegida["corte_info"]["corte"]})
    for (modelo, metodo), resultado in resultados_de_la_ronda.items():
        for tipo, corte in [("Fijo 0.50", 0.50), ("Elegido interno", resultado["corte_info"]["corte"])]:
            filas_metricas_conjuntas_ronda_u3.append({"Externa": externa, "Modelo": modelo, "Método": metodo,
                "Decisión": tipo, "Corte": corte, **metricas_binarias_u3(y_ext, resultado["proba"], corte)})
    # Referencia inicial de XGBoost con artista, todavía no disponible como punto de partida homogéneo.
    inicial_xgb = clone(plantillas_conjuntas_u3["XGBoost"])
    inicial_xgb.fit(X_dev, y_dev)
    proba_xgb_inicial_artista_conjunta_u3.loc[y_ext.index] = inicial_xgb.predict_proba(X_ext)[:, 1]
    del inicial_xgb
    visitas_conjuntas_u3.loc[y_ext.index] += 1
    print(f"Ronda externa {externa} cerrada; selección interna: {pareja_elegida[0]} / {pareja_elegida[1]}.", flush=True)

if not visitas_conjuntas_u3.eq(1).all() or proba_seleccion_conjunta_u3.isna().any() or pred_seleccion_conjunta_u3.isna().any():
    raise ValueError("Cobertura de selección externa incompleta.")
controles_particiones_conjuntas_u3 = pd.DataFrame(filas_particiones_conjuntas_u3)
selecciones_conjuntas_u3 = pd.DataFrame(filas_seleccion_conjunta_u3)
metricas_conjuntas_por_ronda_u3 = pd.DataFrame(filas_metricas_conjuntas_ronda_u3)
filas_resumen_busquedas = []
for (externa, modelo, metodo), r in evidencia_conjunta_u3.items():
    filas_resumen_busquedas.append({"Externa": externa, "Modelo": modelo, "Método": metodo,
        "AP media interna": r["ap_interna"], "DE AP interna": r["de_ap_interna"],
        "Corte interno": r["corte_info"]["corte"], "Viable interno": r["corte_info"]["viable"],
        "Parámetros": {k.replace("modelo__", ""): v for k, v in r["parametros"].items()},
        "Configuraciones": r["configuraciones"], "CV representados": r["ajustes_cv"],
        "CV nuevos": r["cv_nuevos"], "Refits nuevos": r["refits_nuevos"], "OOF corte nuevos": r["cortes_nuevos"],
        "Segundos búsqueda y refit": r["segundos_busqueda"], "Segundos corte": r["segundos_corte"],
        "Origen": r["origen"], "Origen corte": r["origen_corte"]})
resumen_busquedas_conjuntas_u3 = pd.DataFrame(filas_resumen_busquedas)
display(resumen_busquedas_conjuntas_u3.round(4))
display(selecciones_conjuntas_u3.round(4))

INICIO: externa 1/3; Random Forest; Grid.
COMPLETADO: externa 1; Random Forest; Grid; AP interna 0.5871; corte 0.30.
INICIO: externa 1/3; Random Forest; Random.
COMPLETADO: externa 1; Random Forest; Random; AP interna 0.5871; corte 0.30.
INICIO: externa 1/3; LightGBM; Grid.
COMPLETADO: externa 1; LightGBM; Grid; AP interna 0.5877; corte 0.30.
INICIO: externa 1/3; LightGBM; Random.
COMPLETADO: externa 1; LightGBM; Random; AP interna 0.5877; corte 0.30.
INICIO: externa 1/3; XGBoost; Grid.
COMPLETADO: externa 1; XGBoost; Grid; AP interna 0.5936; corte 0.30.
INICIO: externa 1/3; XGBoost; Random.
COMPLETADO: externa 1; XGBoost; Random; AP interna 0.5936; corte 0.30.
Ronda externa 1 cerrada; selección interna: XGBoost / Random.
INICIO: externa 2/3; Random Forest; Grid.
COMPLETADO: externa 2; Random Forest; Grid; AP interna 0.5810; corte 0.30.
INICIO: externa 2/3; Random Forest; Random.
COMPLETADO: externa 2; Random Forest; Random; AP interna 0.5810; corte 0.30.
INICIO: externa 2/3; LightGBM;

,Externa,Modelo,Método,AP media interna,DE AP interna,Corte interno,Viable interno,Parámetros,Configuraciones,CV representados,CV nuevos,Refits nuevos,OOF corte nuevos,Segundos búsqueda y refit,Segundos corte,Origen,Origen corte
0,1,Random Forest,Grid,0.5871,0.0109,0.30,True,"{'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 200}",12,60,60,1,5,197.4583,33.7845,Ejecutado en bloque 17,Calculado: OOF interna
1,1,Random Forest,Random,0.5871,0.0109,0.30,True,"{'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 20}",10,50,50,1,0,183.2205,0.0000,Ejecutado en bloque 17,Reutilizado: mismos parámetros y desarrollo
2,1,LightGBM,Grid,0.5877,0.0067,0.30,True,"{'learning_rate': 0.1, 'min_child_samples': 10, 'n_estimators': 200, 'num_leaves': 31}",16,80,80,1,5,75.7502,5.8709,Ejecutado en bloque 17,Calculado: OOF interna
3,1,LightGBM,Random,0.5877,0.0067,0.30,True,"{'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 10, 'learning_rate': 0.1}",10,50,50,1,0,42.9517,0.0000,Ejecutado en bloque 17,Reutilizado: mismos parámetros y desarrollo
4,1,XGBoost,Grid,0.5936,0.0074,0.30,True,"{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1.0}",16,80,80,1,0,94.2832,0.0000,Ejecutado en bloque 17,Reutilizado: mismos parámetros y desarrollo
5,1,XGBoost,Random,0.5936,0.0074,0.30,True,"{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}",10,50,0,0,0,73.2627,10.1716,Reutilizado y verificado: bloque 14,Bloque 14
6,2,Random Forest,Grid,0.5810,0.0239,0.30,True,"{'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 200}",12,60,60,1,5,213.9197,34.8475,Ejecutado en bloque 17,Calculado: OOF interna
7,2,Random Forest,Random,0.5810,0.0239,0.30,True,"{'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 20}",10,50,50,1,0,187.0925,0.0000,Ejecutado en bloque 17,Reutilizado: mismos parámetros y desarrollo
8,2,LightGBM,Grid,0.5878,0.0128,0.30,True,"{'learning_rate': 0.1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 15}",16,80,80,1,5,68.8627,5.3414,Ejecutado en bloque 17,Calculado: OOF interna
9,2,LightGBM,Random,0.5878,0.0128,0.30,True,"{'num_leaves': 15, 'n_estimators': 200, 'min_child_samples': 20, 'learning_rate': 0.1}",10,50,50,1,0,43.0209,0.0000,Ejecutado en bloque 17,Reutilizado: mismos parámetros y desarrollo


,Externa,Modelo,Método,AP interna,Corte
0,1,XGBoost,Random,0.5936,0.30
1,2,LightGBM,Random,0.5878,0.30
2,3,XGBoost,Random,0.5813,0.25


### Explicación del bucle conjunto y la selección

1. Se reserva evidencia por ronda/modelo/método y series por ID para la regla de selección conjunta. Ninguna fila se transforma en test nuevo.
2. Las tres particiones externas se conservan. Se comprueban claves e IDs separados, cobertura completa, ausencia de test y ambas clases. Los cinco pliegues internos se construyen solo con desarrollo, verificando grupos, clases, medianas calculables y cobertura única.
3. Cada familia tiene su propia memoria de tablas de cortes, vacía al cambiar de desarrollo/modelo. En XGBoost se inicia con la tabla del bloque 14, cuyo protocolo se verificará contra Grid antes de reutilizar Random.
4. Se ejecuta Grid y después Random. Para RF y LightGBM se entrenan ambas búsquedas; en XGBoost se completa Grid y se recupera Random ya verificado, distinguiendo su origen y costo nuevo cero.
5. El diccionario de evidencia guarda cada búsqueda completada. Sus parámetros y AP interna deciden la pareja elegida **antes de calcular métricas externas**; en empates se aplica la regla previa de menor presupuesto y orden alfabético.
6. La selección conjunta toma las puntuaciones y decisiones del ganador interno sin otro entrenamiento. No se escoge el modelo por su precisión, recall o AP externa.
7. Se evalúan los seis procedimientos externos tanto a 0,50 como al corte interno. Todas esas respuestas externas se usan para informar resultados, no para volver a decidir parámetros o cortes.
8. Se incorpora una referencia XGBoost inicial con artista con la configuración fija del bloque 12 y las mismas particiones: son tres ajustes adicionales. Permite comparar antes/después sin mezclar entradas sin artista o distintos tamaños de ajuste. Las referencias iniciales RF y LightGBM ya están ejecutadas y se conservan.
9. Los contadores exigen una evaluación externa por ID. Las tablas finales distinguen AP interna, parámetros, corte viable, tiempos, configuraciones representadas y ajustes efectivamente nuevos. Una puntuación histórica verificada no se presenta como otra ejecución nueva.

No se ajusta un modelo final con las 71.728 canciones. Se evalúan tres desarrollos distintos y pueden elegir parámetros, cortes y familias diferentes. El historial de decisiones del proyecto debe acompañar la interpretación de esta comparación.

In [34]:
def resumir_decisiones_conjuntas_u3(y, proba, decisiones):
    tn, fp, fn, tp = confusion_matrix(y, decisiones, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = recall_score(y, decisiones, zero_division=0)
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Precisión": precision, "Recall": recall, "F1": f1_score(y, decisiones, zero_division=0),
        "Accuracy": accuracy_score(y, decisiones), "ROC-AUC": roc_auc_score(y, proba),
        "Average Precision": average_precision_score(y, proba),
        "KPI precisión": "Cumple" if pd.notna(precision) and precision >= META_PRECISION else "No cumple",
        "KPI recall": "Cumple" if recall >= META_RECALL else "No cumple",
        "KPI simultáneos": "Cumple" if pd.notna(precision) and precision >= META_PRECISION and recall >= META_RECALL else "No cumple"}

filas_resumen_externo_conjunto = []
probas_conjuntas_por_pareja_u3 = {}
preds_conjuntas_por_pareja_u3 = {}
matrices_conjuntas_u3 = {}
for modelo in espacios_conjuntos_u3:
    for metodo in ["Grid", "Random"]:
        proba = pd.Series(np.nan, index=y_train_u3.index, dtype="float64")
        decisiones = pd.Series(pd.NA, index=y_train_u3.index, dtype="Int64")
        for externa in range(1, PLIEGUES_EXTERNOS + 1):
            r = evidencia_conjunta_u3[(externa, modelo, metodo)]
            proba.loc[r["proba"].index] = r["proba"]
            decisiones.loc[r["decision"].index] = r["decision"]
        if proba.isna().any() or decisiones.isna().any() or not proba.between(0, 1).all():
            raise ValueError("Predicciones externas incompletas o no válidas.")
        probas_conjuntas_por_pareja_u3[(modelo, metodo)] = proba
        preds_conjuntas_por_pareja_u3[(modelo, metodo)] = decisiones.astype("int64")
        for tipo, pred in [("Fijo 0.50", proba.ge(0.50).astype("int8")), ("Elegido interno", decisiones.astype("int64"))]:
            metricas = resumir_decisiones_conjuntas_u3(y_train_u3, proba, pred)
            filas_resumen_externo_conjunto.append({"Modelo": modelo, "Método": metodo, "Decisión": tipo, **metricas})
            matrices_conjuntas_u3[(modelo, metodo, tipo)] = pd.DataFrame(
                [[metricas["TN"], metricas["FP"]], [metricas["FN"], metricas["TP"]]],
                index=["Real 0", "Real 1"], columns=["Predice 0", "Predice 1"])
metricas_selector_conjunto_u3 = resumir_decisiones_conjuntas_u3(y_train_u3, proba_seleccion_conjunta_u3, pred_seleccion_conjunta_u3.astype("int64"))
filas_resumen_externo_conjunto.append({"Modelo": "Selección interna conjunta", "Método": "AP interna", "Decisión": "Elegido interno", **metricas_selector_conjunto_u3})
resumen_externo_conjunto_u3 = pd.DataFrame(filas_resumen_externo_conjunto)
inicial_xgb_artista_u3 = metricas_binarias_u3(y_train_u3, proba_xgb_inicial_artista_conjunta_u3, 0.50)
comparacion_iniciales_conjunta_u3 = pd.DataFrame([
    {"Modelo": modelo, "Etapa": "Inicial; corte 0.50", **m}
    for modelo, m in [("Random Forest", metricas_rf_inicial_u3), ("LightGBM", metricas_lgb_inicial_u3), ("XGBoost", inicial_xgb_artista_u3)]
])
resumen_costos_conjunto_u3 = resumen_busquedas_conjuntas_u3.groupby(["Modelo", "Método"]).agg(
    Configuraciones_representadas=("Configuraciones", "sum"), CV_representados=("CV representados", "sum"),
    CV_nuevos=("CV nuevos", "sum"), Refits_nuevos=("Refits nuevos", "sum"), OOF_corte_nuevos=("OOF corte nuevos", "sum"),
    Segundos_busqueda_y_refit=("Segundos búsqueda y refit", "sum"), Segundos_corte=("Segundos corte", "sum"))
ajustes_nuevos_conjunto_u3 = int(resumen_busquedas_conjuntas_u3[["CV nuevos", "Refits nuevos", "OOF corte nuevos"]].to_numpy().sum()) + PLIEGUES_EXTERNOS

display(comparacion_iniciales_conjunta_u3.round(6))
display(resumen_externo_conjunto_u3.round(6))
display(metricas_conjuntas_por_ronda_u3.round(4))
display(resumen_costos_conjunto_u3.round(2))
print(f"Ajustes nuevos de este bloque: {ajustes_nuevos_conjunto_u3}, incluidos tres XGBoost iniciales con artista.")
print("Random de XGBoost representa 168 ajustes previos; no se repitieron.")
print("Matrices disponibles por modelo, método y tipo de decisión en matrices_conjuntas_u3.")

,Modelo,Etapa,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI
0,Random Forest,Inicial; corte 0.50,63775,92,7086,775,0.893887,0.098588,0.177589,0.899928,0.895779,0.557724,False
1,LightGBM,Inicial; corte 0.50,62181,1686,4772,3089,0.646911,0.392953,0.488921,0.909965,0.906320,0.597222,False
2,XGBoost,Inicial; corte 0.50,62418,1449,5141,2720,0.652435,0.346012,0.452203,0.908125,0.897566,0.576426,False


,Modelo,Método,Decisión,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,KPI precisión,KPI recall,KPI simultáneos
0,Random Forest,Grid,Fijo 0.50,62939,928,5583,2278,0.710543,0.289785,0.411674,0.909227,0.906953,0.590255,Cumple,No cumple,No cumple
1,Random Forest,Grid,Elegido interno,58825,5042,2741,5120,0.503838,0.651317,0.568163,0.891493,0.906953,0.590255,Cumple,Cumple,Cumple
2,Random Forest,Random,Fijo 0.50,62939,928,5583,2278,0.710543,0.289785,0.411674,0.909227,0.906953,0.590255,Cumple,No cumple,No cumple
3,Random Forest,Random,Elegido interno,58825,5042,2741,5120,0.503838,0.651317,0.568163,0.891493,0.906953,0.590255,Cumple,Cumple,Cumple
4,LightGBM,Grid,Fijo 0.50,62073,1794,4599,3262,0.645174,0.414960,0.505071,0.910872,0.910703,0.601028,Cumple,No cumple,No cumple
5,LightGBM,Grid,Elegido interno,58579,5288,2648,5213,0.496429,0.663147,0.567803,0.889360,0.910703,0.601028,Cumple,Cumple,Cumple
6,LightGBM,Random,Fijo 0.50,62073,1794,4599,3262,0.645174,0.414960,0.505071,0.910872,0.910703,0.601028,Cumple,No cumple,No cumple
7,LightGBM,Random,Elegido interno,58579,5288,2648,5213,0.496429,0.663147,0.567803,0.889360,0.910703,0.601028,Cumple,Cumple,Cumple
8,XGBoost,Grid,Fijo 0.50,62053,1814,4584,3277,0.643685,0.416868,0.506022,0.910802,0.910567,0.605341,Cumple,No cumple,No cumple
9,XGBoost,Grid,Elegido interno,58232,5635,2501,5360,0.487494,0.681847,0.568519,0.886571,0.910567,0.605341,Cumple,Cumple,Cumple


,Externa,Modelo,Método,Decisión,Corte,TN,FP,FN,TP,Precisión,Recall,F1,Accuracy,ROC-AUC,Average Precision,Cumple ambos KPI
0,1,Random Forest,Grid,Fijo 0.50,0.50,21092,336,1861,730,0.6848,0.2817,0.3992,0.9085,0.9055,0.5706,False
1,1,Random Forest,Grid,Elegido interno,0.30,19675,1753,919,1672,0.4882,0.6453,0.5559,0.8888,0.9055,0.5706,True
2,1,Random Forest,Random,Fijo 0.50,0.50,21092,336,1861,730,0.6848,0.2817,0.3992,0.9085,0.9055,0.5706,False
3,1,Random Forest,Random,Elegido interno,0.30,19675,1753,919,1672,0.4882,0.6453,0.5559,0.8888,0.9055,0.5706,True
4,1,LightGBM,Grid,Fijo 0.50,0.50,20785,643,1533,1058,0.6220,0.4083,0.4930,0.9094,0.9096,0.5921,False
5,1,LightGBM,Grid,Elegido interno,0.30,19591,1837,872,1719,0.4834,0.6635,0.5593,0.8872,0.9096,0.5921,True
6,1,LightGBM,Random,Fijo 0.50,0.50,20785,643,1533,1058,0.6220,0.4083,0.4930,0.9094,0.9096,0.5921,False
7,1,LightGBM,Random,Elegido interno,0.30,19591,1837,872,1719,0.4834,0.6635,0.5593,0.8872,0.9096,0.5921,True
8,1,XGBoost,Grid,Fijo 0.50,0.50,20764,664,1520,1071,0.6173,0.4134,0.4951,0.9091,0.9101,0.5964,False
9,1,XGBoost,Grid,Elegido interno,0.30,19643,1785,878,1713,0.4897,0.6611,0.5627,0.8891,0.9101,0.5964,True


Configuraciones_representadas  CV_representados  CV_nuevos  Refits_nuevos  OOF_corte_nuevos  Segundos_busqueda_y_refit  Segundos_corte
Modelo        Método                                                                                                                                        
LightGBM      Grid                               48               240        240              3                15                     224.02           16.89
              Random                             30               150        150              3                 0                     134.19            0.00
Random Forest Grid                               36               180        180              3                15                     643.26          105.11
              Random                             30               150        150              3                 0                     574.82            0.00
XGBoost       Grid                               48               240        240              3                 0                     300.06            0.00
              Random                             30               150          0              0                 0                     230.40           28.68

Ajustes nuevos de este bloque: 1008, incluidos tres XGBoost iniciales con artista.
Random de XGBoost representa 168 ajustes previos; no se repitieron.
Matrices disponibles por modelo, método y tipo de decisión en matrices_conjuntas_u3.


### Explicación de los resultados, costos y KPI

1. La función de resumen usa decisiones ya fijadas, no elige un corte con las respuestas externas. Deja precisión no definida si no hay positivos señalados y evalúa ambos KPI con las metas originales.
2. Se reúnen las puntuaciones y decisiones de cada pareja en el índice completo. Cada ID debe aparecer una vez. El corte 0,50 usa las mismas puntuaciones que las decisiones internas: AUC y AP serán iguales entre esos cortes de una misma pareja.
3. Las métricas globales se calculan sobre las 71.728 OOF externas. También se muestran por ronda porque proceden de tres modelos entrenados distintos; el promedio de métricas no tiene por qué coincidir con la métrica global. Su desviación no se presenta como intervalo de confianza.
4. La fila de selección conjunta evalúa una regla que eligió familia y método internamente. No es una confirmación adicional del mejor valor observado al ordenar la tabla externa.
5. La tabla inicial incluye ahora XGBoost inicial con artista en las mismas tres particiones. Comparar inicialmente a 0,50 y después optimizado a 0,50 ayuda a separar mejora del modelo de la mejora por elegir otro corte.
6. `groupby().agg()` suma costos por herramienta. Las configuraciones sumadas incluyen repetición en tres desarrollos, no combinaciones globales únicas. Se separan representados, nuevos y reutilizados; el total añade tres referencias XGBoost iniciales.
7. Si dos métodos eligieron el mismo modelo y corte, pueden tener resultados externos idénticos. Eso no demuestra equivalencia universal: Random muestreó un subconjunto que pudo contener el ganador de Grid.
8. Precisión informa falsas alarmas entre señaladas; recall informa positivas omitidas; F1 resume su equilibrio. Accuracy puede bajar al recuperar más positivas. AP y AUC describen ordenamiento, no porcentaje de aciertos ni calibración.
9. El incumplimiento se conserva y explica aunque una configuración tenga AUC favorable. Una meta interna cumplida no garantiza que se cumpla externamente. Los resultados externos no se usan para mover los cortes.

La familia con mayor métrica externa puede ser una elección provisional del proyecto, pero elegirla usando esa tabla reutiliza evidencia de evaluación para selección. La fila de selección interna conjunta proporciona el resultado de la regla predefinida. Ninguna de ellas elimina la exploración histórica de estos datos ni sustituye una evaluación externa independiente nueva.

### 17.3. Estado y continuidad

Se completa la comparación Grid/Random para los tres ensambles con artista bajo el mismo protocolo agrupado, conservando sus puntos de partida y el origen de la evidencia. Los parámetros, métricas y costos deben leerse junto a la viabilidad de los cortes y los errores por ronda, sin afirmar un óptimo global.

No se usa test ni se entrena un modelo final con todo entrenamiento. El siguiente paso será analizar errores e importancia de características, justificar la elección provisional y cerrar clasificación; también sigue pendiente la aplicación acotada de Leave-P-Out solicitada por la pauta. Después corresponde optimizar clustering y preparar los entregables.

Referencias: [GridSearchCV](https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.GridSearchCV.html), [RandomizedSearchCV](https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) y [validación anidada](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).

### 17.4. Lectura simple de los resultados ejecutados

Se completaron las búsquedas y se verificó la compatibilidad de Random de XGBoost anterior. Las siguientes cifras reúnen las predicciones externas, con cortes elegidos solo dentro de desarrollo.

**Random Forest / Grid:** cortes internos por ronda 0.30, 0.30, 0.30. Precisión 50.38%, recall 65.13%, F1 0.5682, ROC-AUC 0.9070 y AP 0.5903. KPI simultáneos: **Cumple** globalmente; 3 de tres rondas externas cumplieron ambos. Detectó 5,120 positivas, omitió 2,741 y produjo 5,042 falsas alarmas.
Al corte fijo 0,50, precisión 71.05%, recall 28.98% y F1 0.4117. Esta fila permite observar la configuración optimizada antes del cambio de decisión. La comparación no promete que bajar el corte mejore todas las métricas: normalmente permite recuperar positivas a costa de más falsas alarmas.

**Random Forest / Random:** cortes internos por ronda 0.30, 0.30, 0.30. Precisión 50.38%, recall 65.13%, F1 0.5682, ROC-AUC 0.9070 y AP 0.5903. KPI simultáneos: **Cumple** globalmente; 3 de tres rondas externas cumplieron ambos. Detectó 5,120 positivas, omitió 2,741 y produjo 5,042 falsas alarmas.
Al corte fijo 0,50, precisión 71.05%, recall 28.98% y F1 0.4117. Esta fila permite observar la configuración optimizada antes del cambio de decisión. La comparación no promete que bajar el corte mejore todas las métricas: normalmente permite recuperar positivas a costa de más falsas alarmas.

**Antes y después, Random Forest, referencia Random:** al mismo corte 0,50, F1 pasó de 0.1776 a 0.4117, recall de 9.86% a 28.98% y AP de 0.5577 a 0.5903. Es una comparación con las mismas particiones y entradas. Con corte interno, F1 fue 0.5682; ese último cambio incorpora también la regla de decisión.

En Random Forest, las puntuaciones externas de Grid y Random fueron idénticas. Random revisó diez configuraciones por ronda frente a 12 de Grid. Ese resultado pertenece a la semilla y el espacio ensayados, no garantiza equivalencia en cualquier búsqueda.

**LightGBM / Grid:** cortes internos por ronda 0.30, 0.30, 0.30. Precisión 49.64%, recall 66.31%, F1 0.5678, ROC-AUC 0.9107 y AP 0.6010. KPI simultáneos: **Cumple** globalmente; 3 de tres rondas externas cumplieron ambos. Detectó 5,213 positivas, omitió 2,648 y produjo 5,288 falsas alarmas.
Al corte fijo 0,50, precisión 64.52%, recall 41.50% y F1 0.5051. Esta fila permite observar la configuración optimizada antes del cambio de decisión. La comparación no promete que bajar el corte mejore todas las métricas: normalmente permite recuperar positivas a costa de más falsas alarmas.

**LightGBM / Random:** cortes internos por ronda 0.30, 0.30, 0.30. Precisión 49.64%, recall 66.31%, F1 0.5678, ROC-AUC 0.9107 y AP 0.6010. KPI simultáneos: **Cumple** globalmente; 3 de tres rondas externas cumplieron ambos. Detectó 5,213 positivas, omitió 2,648 y produjo 5,288 falsas alarmas.
Al corte fijo 0,50, precisión 64.52%, recall 41.50% y F1 0.5051. Esta fila permite observar la configuración optimizada antes del cambio de decisión. La comparación no promete que bajar el corte mejore todas las métricas: normalmente permite recuperar positivas a costa de más falsas alarmas.

**Antes y después, LightGBM, referencia Random:** al mismo corte 0,50, F1 pasó de 0.4889 a 0.5051, recall de 39.30% a 41.50% y AP de 0.5972 a 0.6010. Es una comparación con las mismas particiones y entradas. Con corte interno, F1 fue 0.5678; ese último cambio incorpora también la regla de decisión.

En LightGBM, las puntuaciones externas de Grid y Random fueron idénticas. Random revisó diez configuraciones por ronda frente a 16 de Grid. Ese resultado pertenece a la semilla y el espacio ensayados, no garantiza equivalencia en cualquier búsqueda.

**XGBoost / Grid:** cortes internos por ronda 0.30, 0.30, 0.25. Precisión 48.75%, recall 68.18%, F1 0.5685, ROC-AUC 0.9106 y AP 0.6053. KPI simultáneos: **Cumple** globalmente; 3 de tres rondas externas cumplieron ambos. Detectó 5,360 positivas, omitió 2,501 y produjo 5,635 falsas alarmas.
Al corte fijo 0,50, precisión 64.37%, recall 41.69% y F1 0.5060. Esta fila permite observar la configuración optimizada antes del cambio de decisión. La comparación no promete que bajar el corte mejore todas las métricas: normalmente permite recuperar positivas a costa de más falsas alarmas.

**XGBoost / Random:** cortes internos por ronda 0.30, 0.30, 0.25. Precisión 48.75%, recall 68.18%, F1 0.5685, ROC-AUC 0.9106 y AP 0.6053. KPI simultáneos: **Cumple** globalmente; 3 de tres rondas externas cumplieron ambos. Detectó 5,360 positivas, omitió 2,501 y produjo 5,635 falsas alarmas.
Al corte fijo 0,50, precisión 64.37%, recall 41.69% y F1 0.5060. Esta fila permite observar la configuración optimizada antes del cambio de decisión. La comparación no promete que bajar el corte mejore todas las métricas: normalmente permite recuperar positivas a costa de más falsas alarmas.

**Antes y después, XGBoost, referencia Random:** al mismo corte 0,50, F1 pasó de 0.4522 a 0.5060, recall de 34.60% a 41.69% y AP de 0.5764 a 0.6053. Es una comparación con las mismas particiones y entradas. Con corte interno, F1 fue 0.5685; ese último cambio incorpora también la regla de decisión.

En XGBoost, las puntuaciones externas de Grid y Random fueron idénticas. Random revisó diez configuraciones por ronda frente a 16 de Grid. Ese resultado pertenece a la semilla y el espacio ensayados, no garantiza equivalencia en cualquier búsqueda.

**Regla de selección interna conjunta:** precisión 48.71%, recall 68.02%, F1 0.5677, AUC 0.9105 y AP 0.6026. KPI simultáneos: **Cumple**. La tabla de selección muestra qué familia y método se eligieron con cada desarrollo; no se eligieron a partir del mejor valor externo.

**Costo observado:** 1,008 ajustes nuevos, incluidos tres XGBoost iniciales con artista. Las tablas representan 1.110 ajustes de candidatos entre familias y métodos, con 150 CV de XGBoost Random ya realizados antes. Sus 168 ajustes históricos incluyen además tres refits y quince OOF para cortes. La reutilización de cortes de configuraciones idénticas ahorra ajustes adicionales y está identificada; no se suman como si hubieran vuelto a ejecutarse.

**Alcance:** cumplir KPI sigue dejando falsas alarmas y positivas omitidas. Si algún procedimiento no cumple, el estado se conserva; no se modificaron metas según resultados. Las diferencias son descriptivas, sin prueba de superioridad estadística. No hay un modelo final ni un test nuevo. El cierre debe integrar errores, interpretación, costo y limitaciones de artista y versiones.


**Qué significa que los F1 sean tan cercanos:** Random Forest obtuvo 0,5682; LightGBM 0,5678; XGBoost 0,5685. La diferencia máxima es aproximadamente 0,0007 y no se realizó una prueba estadística que respalde superioridad. Random Forest tiene mayor precisión (50,38%) y menos falsas alarmas; XGBoost tiene mayor recall (68,18%) y detecta más positivas. Frente a Random Forest, XGBoost recuperó 240 positivas adicionales a costa de 593 falsas alarmas adicionales. Son alternativas con distintos intercambios de errores, aunque ambas cumplen las metas.

**Parámetros elegidos, con Grid y Random coincidentes:** Random Forest eligió 200 árboles, profundidad 20 y hoja mínima 1 en las tres rondas. XGBoost eligió 200 árboles, profundidad 6, tasa 0,10 y subsample 1,0 en las tres. LightGBM eligió 200 etapas y tasa 0,10; en las rondas 1 y 3, 31 hojas y mínimo 10 observaciones por hoja; en la ronda 2, 15 hojas y mínimo 20. Esto muestra que la mejor configuración interna puede variar con los datos. No hay todavía una única configuración final ajustada con todo entrenamiento.

**Cambio antes de elegir otros cortes:** al corte fijo 0,50 las tres familias mejoraron AP y F1 externas frente a sus configuraciones iniciales, aunque continuaron fallando el recall mínimo. El aumento posterior de recall con cortes internos es un efecto adicional del procedimiento de decisión, no únicamente de los hiperparámetros. Con los cortes internos, los tres ensambles cumplieron ambas metas en cada una de las tres rondas externas.

**Selección conjunta:** la regla prefijada eligió XGBoost/Random en las rondas 1 y 3 y LightGBM/Random en la 2. Obtuvo precisión 48,71%, recall 68,02% y F1 0,5677: cumple ambos KPI y no supera todas las métricas de cada alternativa. La elección provisional debe integrar los errores, la interpretación y el objetivo del proyecto, no tratar una diferencia decimal como una garantía de superioridad.
